In [2]:
import json
from pathlib import Path
import pandas as pd

# Project root (assuming notebook runs from notebooks/)
BASE_DIR = Path.cwd().parent

folder = BASE_DIR / "data" / "raw" / "train_positions"

# Find the latest JSON file
latest_file = max(
    folder.glob("*.json"),
    key=lambda p: p.stat().st_mtime
)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print("File:", latest_file.name)
print("Number of train entities:", len(data["entity"]))

File: train_positions_20261027_090330_522854.json
Number of train entities: 238


In [3]:
df_trains = pd.json_normalize(data["entity"])

print("Shape:", df_trains.shape)
print("\nColumns:")
print(df_trains.columns.tolist())

display(df_trains.head())

Shape: (238, 12)

Columns:
['id', 'vehicle.trip.trip_id', 'vehicle.trip.schedule_relationship', 'vehicle.trip.route_id', 'vehicle.position.latitude', 'vehicle.position.longitude', 'vehicle.timestamp', 'vehicle.congestion_level', 'vehicle.stop_id', 'vehicle.vehicle.id', 'vehicle.vehicle.label', 'vehicle.occupancy_status']


,id,vehicle.trip.trip_id,vehicle.trip.schedule_relationship,vehicle.trip.route_id,vehicle.position.latitude,vehicle.position.longitude,vehicle.timestamp,vehicle.congestion_level,vehicle.stop_id,vehicle.vehicle.id,vehicle.vehicle.label,vehicle.occupancy_status
0,1,N410.1324.165.12.D.10.92000922,SCHEDULED,CCN_2a,-33.109630,151.48701,1793091567,UNKNOWN_CONGESTION_LEVEL,MetroNet.561,7086.6002.1846.9718.4636.2113.8979.9394.7450.5944,19:21 Newcastle Interchange Station to Central...,NaN
1,2,8976.1324.165.8.W.0.92004721,SCHEDULED,RTTA_DEF,-33.709538,150.31633,1793090570,UNKNOWN_CONGESTION_LEVEL,Mountains.KT10/21 Loc,3874,NaN,NaN
2,3,W667.1324.165.44.D.10.92001717,SCHEDULED,BMT_1,-33.582930,150.25812,1793089072,UNKNOWN_CONGESTION_LEVEL,Mountains.MV50 Loc,1729.7376.7581.6303.5153.4080.6744.2589.1077.2402,16:26 Central Station to Katoomba Station,NaN
3,4,R761.1324.165.12.D.6.92003182,SCHEDULED,RTTA_REV,-33.863810,151.06140,1793088654,UNKNOWN_CONGESTION_LEVEL,Lidcombe.ST329 Loc,1623.6714.4505.2115.7559.8411,18:54 Central Station to Central Station,NaN
4,5,W474.1324.165.12.D.10.92002212,SCHEDULED,BMT_2,-33.721560,150.60304,1793091687,UNKNOWN_CONGESTION_LEVEL,Mountains.Warrimoo 1 Loc,7179.4653.8449.9642.4807.5809.6649.2763.7792.4225,18:52 Mount Victoria Station to Central Station,NaN


In [4]:
df_train_clean = df_trains[[
    "vehicle.trip.trip_id",
    "vehicle.trip.route_id",
    "vehicle.position.latitude",
    "vehicle.position.longitude",
    "vehicle.timestamp",
    "vehicle.stop_id",
    "vehicle.vehicle.label"
]].copy()

df_train_clean = df_train_clean.rename(columns={
    "vehicle.trip.trip_id": "trip_id",
    "vehicle.trip.route_id": "route_id",
    "vehicle.position.latitude": "latitude",
    "vehicle.position.longitude": "longitude",
    "vehicle.timestamp": "timestamp",
    "vehicle.stop_id": "stop_id",
    "vehicle.vehicle.label": "train_label"
})

# Convert Unix timestamps to readable UTC dates
df_train_clean["timestamp"] = pd.to_datetime(
    df_train_clean["timestamp"],
    unit="s",
    utc=True,
    errors="coerce"
)

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,N410.1324.165.12.D.10.92000922,CCN_2a,-33.109630,151.48701,NaT,MetroNet.561,19:21 Newcastle Interchange Station to Central...
1,8976.1324.165.8.W.0.92004721,RTTA_DEF,-33.709538,150.31633,NaT,Mountains.KT10/21 Loc,NaN
2,W667.1324.165.44.D.10.92001717,BMT_1,-33.582930,150.25812,NaT,Mountains.MV50 Loc,16:26 Central Station to Katoomba Station
3,R761.1324.165.12.D.6.92003182,RTTA_REV,-33.863810,151.06140,NaT,Lidcombe.ST329 Loc,18:54 Central Station to Central Station
4,W474.1324.165.12.D.10.92002212,BMT_2,-33.721560,150.60304,NaT,Mountains.Warrimoo 1 Loc,18:52 Mount Victoria Station to Central Station


In [5]:
df_train_clean["timestamp"] = pd.to_datetime(
    pd.to_numeric(df_train_clean["timestamp"], errors="coerce"),
    unit="s",
    utc=True,
    errors="coerce"
)

# Convert UTC to Sydney time
df_train_clean["timestamp"] = (
    df_train_clean["timestamp"]
    .dt.tz_convert("Australia/Sydney")
)

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,N410.1324.165.12.D.10.92000922,CCN_2a,-33.109630,151.48701,NaT,MetroNet.561,19:21 Newcastle Interchange Station to Central...
1,8976.1324.165.8.W.0.92004721,RTTA_DEF,-33.709538,150.31633,NaT,Mountains.KT10/21 Loc,NaN
2,W667.1324.165.44.D.10.92001717,BMT_1,-33.582930,150.25812,NaT,Mountains.MV50 Loc,16:26 Central Station to Katoomba Station
3,R761.1324.165.12.D.6.92003182,RTTA_REV,-33.863810,151.06140,NaT,Lidcombe.ST329 Loc,18:54 Central Station to Central Station
4,W474.1324.165.12.D.10.92002212,BMT_2,-33.721560,150.60304,NaT,Mountains.Warrimoo 1 Loc,18:52 Mount Victoria Station to Central Station


In [6]:
import json
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent
folder = BASE_DIR / "data" / "raw" / "train_positions"

latest_file = max(folder.glob("*.json"), key=lambda p: p.stat().st_mtime)

with open(latest_file, "r", encoding="utf-8") as f:
    data = json.load(f)

# Recreate DataFrame from original data
df_trains = pd.json_normalize(data["entity"])

# Check the original timestamp values
print(df_trains["vehicle.timestamp"].head())

0    1793091567
1    1793090570
2    1793089072
3    1793088654
4    1793091687
Name: vehicle.timestamp, dtype: str


In [7]:
# Start fresh from the original DataFrame
df_train_clean["timestamp"] = df_trains["vehicle.timestamp"]

# Convert strings to numbers
df_train_clean["timestamp"] = pd.to_numeric(
    df_train_clean["timestamp"],
    errors="coerce"
)

# Convert Unix seconds to Sydney local time
df_train_clean["timestamp"] = pd.to_datetime(
    df_train_clean["timestamp"],
    unit="s",
    utc=True
).dt.tz_convert("Australia/Sydney")

display(df_train_clean.head())

,trip_id,route_id,latitude,longitude,timestamp,stop_id,train_label
0,N410.1324.165.12.D.10.92000922,CCN_2a,-33.109630,151.48701,2026-10-27 19:59:27+11:00,MetroNet.561,19:21 Newcastle Interchange Station to Central...
1,8976.1324.165.8.W.0.92004721,RTTA_DEF,-33.709538,150.31633,2026-10-27 19:42:50+11:00,Mountains.KT10/21 Loc,NaN
2,W667.1324.165.44.D.10.92001717,BMT_1,-33.582930,150.25812,2026-10-27 19:17:52+11:00,Mountains.MV50 Loc,16:26 Central Station to Katoomba Station
3,R761.1324.165.12.D.6.92003182,RTTA_REV,-33.863810,151.06140,2026-10-27 19:10:54+11:00,Lidcombe.ST329 Loc,18:54 Central Station to Central Station
4,W474.1324.165.12.D.10.92002212,BMT_2,-33.721560,150.60304,2026-10-27 20:01:27+11:00,Mountains.Warrimoo 1 Loc,18:52 Mount Victoria Station to Central Station


In [8]:
# Calculate how old each train position is
now = pd.Timestamp.now(tz="Australia/Sydney")

df_train_clean["minutes_old"] = (
    (now - df_train_clean["timestamp"])
    .dt.total_seconds() / 60
)

# Show the freshest positions first
display(
    df_train_clean[
        ["train_label", "route_id", "timestamp", "minutes_old"]
    ].sort_values("minutes_old").head(10)
)

,train_label,route_id,timestamp,minutes_old
150,19:42 Leppington Station to Schofields Station,CMB_2a,2026-10-27 20:03:15+11:00,1502.581103
209,NaN,NaN,2026-10-27 20:03:13+11:00,1502.614436
164,19:47 Central Station to Sydenham Station,APS_1e,2026-10-27 20:03:13+11:00,1502.614436
174,19:33 Cronulla Station to Bondi Junction Station,ESI_2d,2026-10-27 20:03:12+11:00,1502.631103
184,19:14 Central Station to Revesby Station,APS_1a,2026-10-27 20:03:12+11:00,1502.631103
199,19:41 Central Station to Leppington Station,IWL_1c,2026-10-27 20:03:12+11:00,1502.631103
191,19:50 Central Station to Macarthur Station,APS_1a,2026-10-27 20:03:11+11:00,1502.647769
120,18:41 Central Station to Kiama Station,SCO_1a,2026-10-27 20:03:11+11:00,1502.647769
103,19:15 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,1502.647769
173,19:45 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,1502.647769


In [9]:
# Get the collection time from the filename
snapshot_time = pd.to_datetime(
    latest_file.stem.split("_")[-3] + "_" +
    latest_file.stem.split("_")[-2] + "_" +
    latest_file.stem.split("_")[-1],
    format="%Y%m%d_%H%M%S_%f",
    utc=True
).tz_convert("Australia/Sydney")

# Compare each train's timestamp with collection time
df_train_clean["minutes_old_at_collection"] = (
    (snapshot_time - df_train_clean["timestamp"])
    .dt.total_seconds() / 60
)

print("Snapshot collected:", snapshot_time)

display(
    df_train_clean[
        ["train_label", "route_id", "timestamp",
         "minutes_old_at_collection"]
    ].sort_values("minutes_old_at_collection").head(10)
)

Snapshot collected: 2026-10-27 20:03:30.522854+11:00


,train_label,route_id,timestamp,minutes_old_at_collection
150,19:42 Leppington Station to Schofields Station,CMB_2a,2026-10-27 20:03:15+11:00,0.258714
209,NaN,NaN,2026-10-27 20:03:13+11:00,0.292048
164,19:47 Central Station to Sydenham Station,APS_1e,2026-10-27 20:03:13+11:00,0.292048
174,19:33 Cronulla Station to Bondi Junction Station,ESI_2d,2026-10-27 20:03:12+11:00,0.308714
184,19:14 Central Station to Revesby Station,APS_1a,2026-10-27 20:03:12+11:00,0.308714
199,19:41 Central Station to Leppington Station,IWL_1c,2026-10-27 20:03:12+11:00,0.308714
191,19:50 Central Station to Macarthur Station,APS_1a,2026-10-27 20:03:11+11:00,0.325381
120,18:41 Central Station to Kiama Station,SCO_1a,2026-10-27 20:03:11+11:00,0.325381
103,19:15 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,0.325381
173,19:45 Bondi Junction Station to Cronulla Station,ESI_1d,2026-10-27 20:03:11+11:00,0.325381


In [10]:
# Convert position age into seconds
df_train_clean["seconds_old"] = (
    df_train_clean["minutes_old_at_collection"] * 60
)

print(df_train_clean["seconds_old"].describe())

print("\nPositions within 30 seconds:",
      df_train_clean["seconds_old"].between(0, 30).sum())

print("Positions within 60 seconds:",
      df_train_clean["seconds_old"].between(0, 60).sum())

print("Positions older than 5 minutes:",
      (df_train_clean["seconds_old"] > 300).sum())

count       238.000000
mean       8796.174115
std       62718.200926
min          15.522854
25%          39.522854
50%          81.022854
75%         496.022854
max      873322.522854
Name: seconds_old, dtype: float64

Positions within 30 seconds: 38
Positions within 60 seconds: 102
Positions older than 5 minutes: 73


In [11]:
import folium
import pandas as pd

train_map = folium.Map(
    location=[-33.8688, 151.2093],
    zoom_start=10,
    tiles="OpenStreetMap"
)

for _, train in df_train_clean.iterrows():

    if pd.isna(train["latitude"]) or pd.isna(train["longitude"]):
        continue

    age = train["seconds_old"]

    if pd.isna(age) or age < 0:
        continue
    elif age <= 60:
        colour = "green"
    elif age <= 300:
        colour = "orange"
    else:
        colour = "gray"

    label = train["train_label"]

    if pd.isna(label):
        label = "Unknown train"

    popup = f"""
    <b>Train:</b> {label}<br>
    <b>Route:</b> {train['route_id']}<br>
    <b>Last position:</b> {train['timestamp']}<br>
    <b>Age at collection:</b> {age:.0f} seconds
    """

    folium.CircleMarker(
        location=[train["latitude"], train["longitude"]],
        radius=5,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.8,
        popup=folium.Popup(popup, max_width=350)
    ).add_to(train_map)

train_map.save("sydney_train_map.html")

In [12]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd().parent

gtfs_dir = BASE_DIR / "data" / "raw" / "gtfs_static" / "gtfs_static"

routes = pd.read_csv(gtfs_dir / "routes.txt", dtype=str)
trips = pd.read_csv(gtfs_dir / "trips.txt", dtype=str)

print("Routes:", routes.shape)
print("Trips:", trips.shape)

display(routes.head())

Routes: (10237, 9)
Trips: (211419, 11)


,route_id,agency_id,route_short_name,route_long_name,route_desc,route_type,route_color,route_text_color,exact_times
0,1-SC0-1-sj2-1,701,SC01,Kiama to Bomaderry,Illawarra Buses Network,700,00B5EF,FFFFFF,1
1,1-SH1-0-sj2-2,701,SH100,Moss Vale to Goulburn,Illawarra Buses Network,700,00B5EF,FFFFFF,1
2,11-400-1-sj2-1,2501,4001,Penrith Station to McCarthy College,School buses,712,00B5EF,FFFFFF,1
3,11-400-2-sj2-1,2501,4002,Swallow Dr opp Skylark Cres to Caroline Chisho...,School buses,712,00B5EF,FFFFFF,1
4,11-400-3-sj2-1,2501,4003,Caroline Chisholm College to York PS via Regen...,School buses,712,00B5EF,FFFFFF,1


In [13]:
train_routes = routes[routes["agency_id"] == "x0001"].copy()

print("Number of Sydney Trains routes:", len(train_routes))

display(
    train_routes[
        ["route_short_name", "route_long_name", "route_type"]
    ]
)

Number of Sydney Trains routes: 10


,route_short_name,route_long_name,route_type
2320,T1,T1 North Shore and Western Line,2
2321,T1,T1 North Shore and Western Line,2
2322,T2,T2 Leppington and Inner West Line,2
2323,T3,T3 Liverpool and Inner West Line,2
2324,T4,T4 Eastern Suburbs and Illawarra Line,2
2325,T5,T5 Cumberland Line,2
2326,T6,T6 Lidcombe and Bankstown Line,2
2327,T7,T7 Olympic Park Line,2
2328,T8,T8 Airport and South Line,2
2329,T9,T9 Northern Line,2


In [14]:
train_trips = trips.merge(
    train_routes[["route_id", "route_short_name"]],
    on="route_id",
    how="inner"
)

shape_ids = set(train_trips["shape_id"].dropna())

print("Total Sydney Trains trips:", len(train_trips))
print("Unique railway shapes:", len(shape_ids))

display(
    train_trips[
        ["route_short_name", "trip_id", "shape_id"]
    ].head(10)
)

Total Sydney Trains trips: 48298
Unique railway shapes: 1298


,route_short_name,trip_id,shape_id
0,T1,101A.2010.103.128.T.8.91040144,2-T1-N-sj2-1.13.R
1,T1,101A.2010.103.2.T.8.91040144,2-T1-N-sj2-1.13.R
2,T1,101D.2010.103.128.T.8.91040148,2-T1-N-sj2-1.23.H
3,T1,101D.2010.103.2.T.8.91040148,2-T1-N-sj2-1.23.H
4,T1,101E.2010.103.128.T.8.91040150,2-T1-N-sj2-1.9.R
5,T1,101E.2010.103.2.T.8.91040150,2-T1-N-sj2-1.9.R
6,T1,101H.2010.103.128.T.8.91040156,2-T1-N-sj2-1.23.H
7,T1,101H.2010.103.2.T.8.91040156,2-T1-N-sj2-1.23.H
8,T1,101J.2010.103.128.T.8.91040158,2-T1-N-sj2-1.9.R
9,T1,101J.2010.103.2.T.8.91040158,2-T1-N-sj2-1.9.R


In [15]:
train_shape_chunks = []

for chunk in pd.read_csv(
    gtfs_dir / "shapes.txt",
    dtype={"shape_id": str},
    chunksize=100_000
):
    selected = chunk[
        chunk["shape_id"].isin(shape_ids)
    ]

    if not selected.empty:
        train_shape_chunks.append(selected)

train_shapes = pd.concat(
    train_shape_chunks,
    ignore_index=True
)

train_shapes["shape_pt_sequence"] = pd.to_numeric(
    train_shapes["shape_pt_sequence"]
)

train_shapes = train_shapes.sort_values(
    ["shape_id", "shape_pt_sequence"]
)

print("Total railway coordinates:", len(train_shapes))
print("Shapes extracted:", train_shapes["shape_id"].nunique())

display(train_shapes.head())

Total railway coordinates: 852072
Shapes extracted: 1298


,shape_id,shape_pt_lat,shape_pt_lon,shape_pt_sequence,shape_dist_traveled
0,2-T1-N-sj2-1.1.R,-33.623511,151.153121,1,0.00
1,2-T1-N-sj2-1.1.R,-33.624566,151.151528,2,188.53
2,2-T1-N-sj2-1.1.R,-33.624789,151.151271,3,222.85
3,2-T1-N-sj2-1.1.R,-33.625196,151.150681,4,293.81
4,2-T1-N-sj2-1.1.R,-33.625464,151.150281,5,341.41


In [16]:
output_dir = BASE_DIR / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_shapes.to_parquet(
    output_dir / "sydney_train_shapes.parquet",
    index=False
)

print("Railway shapes saved!")

Railway shapes saved!


In [17]:
import pandas as pd

stops = pd.read_csv(
    gtfs_dir / "stops.txt",
    dtype=str
)

stop_times = pd.read_csv(
    gtfs_dir / "stop_times.txt",
    dtype=str
)

print("Stops:", stops.shape)
print("Stop times:", stop_times.shape)

display(stops.head())
display(stop_times.head())

Stops: (171285, 10)
Stop times: (5705967, 11)


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
0,2533211,2533211,"Kiama Station, Platform 1",-34.67255547,150.85455900,NaN,253330,1,Level -1,1
1,2533212,2533212,"Kiama Station, Platform 2",-34.67260704,150.85463774,NaN,253330,1,Level -1,2
2,253330,NaN,Kiama Station,-34.67266689,150.85445510,1,NaN,0,NaN,NaN
3,253330_CCP1,NaN,CCP Railway Pde N,-34.67188486,150.85496415,NaN,253330,NaN,Level 0,NaN
4,253330_CCP2,NaN,CCP Railway Pde S,-34.67326859,150.85366501,NaN,253330,NaN,Level 0,NaN


,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled,timepoint,stop_note
0,1.AA61.1-SC0-1-sj2-1.1.R,22:50:00,22:50:00,254111,1,NaN,0,0,0.00,1,NaN
1,1.AA61.1-SC0-1-sj2-1.1.R,23:04:00,23:04:00,253572,2,NaN,0,0,16208.96,1,NaN
2,1.AA61.1-SC0-1-sj2-1.1.R,23:20:00,23:20:00,253476,3,NaN,0,0,29527.41,1,NaN
3,1.AA61.1-SC0-1-sj2-1.1.R,23:30:00,23:30:00,253332,4,NaN,0,0,40708.07,1,NaN
4,1.AB22.1-SC0-1-sj2-1.1.R,03:25:00,03:25:00,254111,1,NaN,0,0,0.00,1,NaN


In [18]:
station_names = ["Central Station", "Parramatta Station"]

selected_stations = stops[
    (stops["stop_name"].isin(station_names))
    & (stops["location_type"] == "1")
]

display(
    selected_stations[
        ["stop_id", "stop_name", "location_type"]
    ]
)

,stop_id,stop_name,location_type
800,215020,Parramatta Station,1
22172,200060,Central Station,1


In [19]:
central_id = "200060"
parramatta_id = "215020"

platforms = stops[
    stops["parent_station"].isin([
        central_id,
        parramatta_id
    ])
].copy()

display(
    platforms[
        [
            "stop_id",
            "stop_name",
            "parent_station",
            "platform_code"
        ]
    ]
)

,stop_id,stop_name,parent_station,platform_code
789,2150106,"Parramatta Station, Stand B1",215020,NaN
790,2150107,"Parramatta Station, Stand B2",215020,NaN
791,2150109,"Parramatta Station, Stand B4",215020,NaN
792,2150111,"Parramatta Station, Stand A2",215020,NaN
793,2150114,"Parramatta Station, Stand A1",215020,NaN
...,...,...,...,...
22236,200060_MZ,TicketG Intercity,200060,NaN
22269,2010107,Central Chalmers Street Light Rail,200060,NaN
22270,201016,"Central Station, Chalmers St, Stand G",200060,NaN
22271,201039,"Central Station, Foveaux St, Stand F",200060,NaN


In [20]:
# Get all Sydney Trains trip IDs
train_trip_ids = set(train_trips["trip_id"])

# Find timetable records belonging to Sydney Trains
train_stop_times = stop_times[
    stop_times["trip_id"].isin(train_trip_ids)
].copy()

# Find stops used by these trips
train_stop_ids = set(train_stop_times["stop_id"])

# Filter Central and Parramatta child stops
train_platforms = platforms[
    platforms["stop_id"].isin(train_stop_ids)
].copy()

display(
    train_platforms[
        [
            "stop_id",
            "stop_name",
            "parent_station",
            "platform_code"
        ]
    ]
)

print("Train platforms found:", len(train_platforms))

,stop_id,stop_name,parent_station,platform_code
833,2150411,"Parramatta Station, Platform 1",215020,1
834,2150412,"Parramatta Station, Platform 2",215020,2
835,2150413,"Parramatta Station, Platform 3",215020,3
836,2150414,"Parramatta Station, Platform 4",215020,4
22117,2000321,"Central Station, Platform 1",200060,1
22118,2000322,"Central Station, Platform 2",200060,2
22119,2000323,"Central Station, Platform 3",200060,3
22120,2000324,"Central Station, Platform 4",200060,4
22121,2000325,"Central Station, Platform 5",200060,5
22122,2000326,"Central Station, Platform 6",200060,6


Train platforms found: 28


In [21]:
# Get Central and Parramatta train platform IDs
central_stops = set(
    train_platforms.loc[
        train_platforms["parent_station"] == "200060",
        "stop_id"
    ]
)

parramatta_stops = set(
    train_platforms.loc[
        train_platforms["parent_station"] == "215020",
        "stop_id"
    ]
)

# Find departures from Central
central_departures = train_stop_times[
    train_stop_times["stop_id"].isin(central_stops)
][
    ["trip_id", "departure_time", "stop_sequence", "stop_id"]
].copy()

central_departures = central_departures.rename(columns={
    "departure_time": "central_departure",
    "stop_sequence": "central_sequence",
    "stop_id": "central_stop_id"
})

# Find arrivals at Parramatta
parramatta_arrivals = train_stop_times[
    train_stop_times["stop_id"].isin(parramatta_stops)
][
    ["trip_id", "arrival_time", "stop_sequence", "stop_id"]
].copy()

parramatta_arrivals = parramatta_arrivals.rename(columns={
    "arrival_time": "parramatta_arrival",
    "stop_sequence": "parramatta_sequence",
    "stop_id": "parramatta_stop_id"
})

# Join both stations on the same train trip
journeys = central_departures.merge(
    parramatta_arrivals,
    on="trip_id",
    how="inner"
)

# Make sure the train stops at Central first
journeys["central_sequence"] = pd.to_numeric(
    journeys["central_sequence"]
)

journeys["parramatta_sequence"] = pd.to_numeric(
    journeys["parramatta_sequence"]
)

journeys = journeys[
    journeys["central_sequence"] < journeys["parramatta_sequence"]
].copy()

print("Matching scheduled journeys:", len(journeys))

display(journeys.head())

Matching scheduled journeys: 4067


,trip_id,central_departure,central_sequence,central_stop_id,parramatta_arrival,parramatta_sequence,parramatta_stop_id
0,101B.2010.103.128.T.8.91040143,04:56:01,1,2000338,05:27:00,16,2150414
1,101B.2010.103.2.T.8.91040143,04:56:01,1,2000338,05:27:00,16,2150414
4,101F.2010.103.128.T.8.91040152,08:41:01,1,2000338,09:13:00,16,2150414
5,101F.2010.103.2.T.8.91040152,08:41:01,1,2000338,09:13:00,16,2150414
8,101K.2010.103.128.T.8.91040160,12:26:01,1,2000338,12:58:00,16,2150414


In [22]:
calendar = pd.read_csv(
    gtfs_dir / "calendar.txt",
    dtype=str
)

calendar_dates = pd.read_csv(
    gtfs_dir / "calendar_dates.txt",
    dtype=str
)

print("Calendar:", calendar.shape)
print("Calendar exceptions:", calendar_dates.shape)

display(calendar.head())
display(calendar_dates.head())

Calendar: (2336, 10)
Calendar exceptions: (37650, 3)


,service_id,monday,tuesday,wednesday,thursday,friday,saturday,sunday,start_date,end_date
0,AA61+1,1,1,1,1,1,0,0,20261027,20270125
1,AA61#1+1,1,1,1,1,1,0,0,20261027,20270125
2,AB22+1,0,0,0,0,0,1,1,20261027,20270125
3,AB22#1+1,0,0,0,0,0,1,1,20261027,20270125
4,TA+2,1,1,1,1,1,1,1,20261027,20270125


,service_id,date,exception_type
0,AA61+1,20261225,2
1,AA61+1,20261228,2
2,AA61+1,20270101,2
3,AA61#1+1,20261225,2
4,AA61#1+1,20261228,2


In [23]:
from datetime import datetime

travel_date = "2026-10-28"

date_obj = datetime.strptime(travel_date, "%Y-%m-%d")
date_str = date_obj.strftime("%Y%m%d")
weekday = date_obj.strftime("%A").lower()

# Normal calendar services
regular_services = calendar[
    (calendar["start_date"] <= date_str) &
    (calendar["end_date"] >= date_str) &
    (calendar[weekday] == "1")
]

active_service_ids = set(regular_services["service_id"])

# Exceptions for the selected date
exceptions = calendar_dates[
    calendar_dates["date"] == date_str
]

# Remove cancelled calendar services
removed = set(
    exceptions.loc[
        exceptions["exception_type"] == "2",
        "service_id"
    ]
)

# Add special services
added = set(
    exceptions.loc[
        exceptions["exception_type"] == "1",
        "service_id"
    ]
)

active_service_ids = (active_service_ids - removed) | added

print("Active service patterns:", len(active_service_ids))

Active service patterns: 662


In [24]:
# Add calendar service IDs and train line information
journeys_with_services = journeys.merge(
    train_trips[
        ["trip_id", "service_id", "route_short_name"]
    ],
    on="trip_id",
    how="inner"
)

# Keep only services operating on the travel date
valid_journeys = journeys_with_services[
    journeys_with_services["service_id"].isin(
        active_service_ids
    )
].copy()

print("Valid journeys on", travel_date, ":", len(valid_journeys))

Valid journeys on 2026-10-28 : 259


In [25]:
requested_time = "17:30:00"

upcoming = valid_journeys[
    valid_journeys["central_departure"] >= requested_time
].copy()

upcoming = upcoming.sort_values("central_departure")

# Convert departure times into seconds for interval calculations
def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s

upcoming["departure_seconds"] = (
    upcoming["central_departure"].apply(time_to_seconds)
)

# Calculate gap to the following train
upcoming["next_train_gap_min"] = (
    upcoming["departure_seconds"].shift(-1)
    - upcoming["departure_seconds"]
) / 60

display(
    upcoming[
        [
            "route_short_name",
            "central_departure",
            "parramatta_arrival",
            "next_train_gap_min"
        ]
    ].head(5)
)

,route_short_name,central_departure,parramatta_arrival,next_train_gap_min
3198,T2,17:32:01,18:36:30,2.000000
1258,T1,17:34:01,17:59:00,0.983333
296,T2,17:35:00,18:21:30,2.016667
214,T1,17:37:01,18:02:00,6.000000
1558,T1,17:43:01,18:08:00,3.000000


In [26]:
# Calculate arrival time in seconds
upcoming["arrival_seconds"] = (
    upcoming["parramatta_arrival"].apply(time_to_seconds)
)

# Journey duration in minutes
upcoming["journey_duration_min"] = (
    upcoming["arrival_seconds"]
    - upcoming["departure_seconds"]
) / 60

# Waiting time from requested departure
requested_seconds = time_to_seconds(requested_time)

upcoming["waiting_time_min"] = (
    upcoming["departure_seconds"]
    - requested_seconds
) / 60

# Display the results
display(
    upcoming[
        [
            "route_short_name",
            "central_departure",
            "parramatta_arrival",
            "journey_duration_min",
            "waiting_time_min",
            "next_train_gap_min"
        ]
    ].head(10)
)

,route_short_name,central_departure,parramatta_arrival,journey_duration_min,waiting_time_min,next_train_gap_min
3198,T2,17:32:01,18:36:30,64.483333,2.016667,2.000000
1258,T1,17:34:01,17:59:00,24.983333,4.016667,0.983333
296,T2,17:35:00,18:21:30,46.500000,5.000000,2.016667
214,T1,17:37:01,18:02:00,24.983333,7.016667,6.000000
1558,T1,17:43:01,18:08:00,24.983333,13.016667,3.000000
1686,T1,17:46:01,18:11:00,24.983333,16.016667,1.000000
3967,T2,17:47:01,18:51:30,64.483333,17.016667,2.000000
1327,T1,17:49:01,18:14:00,24.983333,19.016667,0.983333
3199,T2,17:50:00,18:36:30,46.500000,20.000000,2.016667
42,T1,17:52:01,18:17:00,24.983333,22.016667,6.000000


In [28]:
def find_journeys(limit=5):
    results = upcoming[
        [
            "route_short_name",
            "central_departure",
            "parramatta_arrival",
            "journey_duration_min",
            "waiting_time_min",
            "next_train_gap_min"
        ]
    ].head(limit).copy()

    return results

In [45]:
import pandas as pd
from datetime import datetime


# Convert GTFS time to seconds
# Supports times such as 17:30:00 and 25:10:00
def time_to_seconds(t):
    h, m, s = map(int, t.split(":"))
    return h * 3600 + m * 60 + s


def find_journeys(origin, destination, travel_date, departure_time, limit=5):

    # 1. Find parent station IDs
    origin_ids = set(
        stops.loc[
            (stops["stop_name"] == origin) &
            (stops["location_type"] == "1"),
            "stop_id"
        ]
    )

    destination_ids = set(
        stops.loc[
            (stops["stop_name"] == destination) &
            (stops["location_type"] == "1"),
            "stop_id"
        ]
    )

    if not origin_ids or not destination_ids:
        raise ValueError("Origin or destination station not found")

    if origin_ids == destination_ids:
        raise ValueError("Origin and destination must be different")

    # 2. Find platforms used by Sydney Trains
    origin_platforms = set(
        stops.loc[
            stops["parent_station"].isin(origin_ids) &
            stops["stop_id"].isin(train_stop_ids),
            "stop_id"
        ]
    )

    destination_platforms = set(
        stops.loc[
            stops["parent_station"].isin(destination_ids) &
            stops["stop_id"].isin(train_stop_ids),
            "stop_id"
        ]
    )

    # 3. Find departure records from origin
    departures = train_stop_times[
        train_stop_times["stop_id"].isin(origin_platforms)
    ][
        ["trip_id", "departure_time", "stop_sequence", "stop_id"]
    ].rename(
        columns={
            "departure_time": "departure",
            "stop_sequence": "origin_sequence",
            "stop_id": "origin_platform_id"
        }
    )

    # 4. Find arrival records at destination
    arrivals = train_stop_times[
        train_stop_times["stop_id"].isin(destination_platforms)
    ][
        ["trip_id", "arrival_time", "stop_sequence", "stop_id"]
    ].rename(
        columns={
            "arrival_time": "arrival",
            "stop_sequence": "destination_sequence",
            "stop_id": "destination_platform_id"
        }
    )

    # 5. Match trips that visit both stations
    results = departures.merge(
        arrivals,
        on="trip_id",
        how="inner"
    )

    # Ensure the train visits origin before destination
    results["origin_sequence"] = pd.to_numeric(
        results["origin_sequence"]
    )

    results["destination_sequence"] = pd.to_numeric(
        results["destination_sequence"]
    )

    results = results[
        results["origin_sequence"] <
        results["destination_sequence"]
    ].copy()

    # 6. Determine active services for the travel date
    date_obj = datetime.strptime(travel_date, "%Y-%m-%d")
    date_str = date_obj.strftime("%Y%m%d")
    weekday = date_obj.strftime("%A").lower()

    regular = calendar[
        (calendar["start_date"] <= date_str) &
        (calendar["end_date"] >= date_str) &
        (calendar[weekday] == "1")
    ]

    active = set(regular["service_id"])

    # Apply calendar exceptions
    exceptions = calendar_dates[
        calendar_dates["date"] == date_str
    ]

    removed = set(
        exceptions.loc[
            exceptions["exception_type"] == "2",
            "service_id"
        ]
    )

    added = set(
        exceptions.loc[
            exceptions["exception_type"] == "1",
            "service_id"
        ]
    )

    active = (active - removed) | added

    # 7. Connect trips to their service calendars
    results = results.merge(
        train_trips[
            ["trip_id", "service_id", "route_short_name"]
        ],
        on="trip_id",
        how="inner"
    )

    results = results[
        results["service_id"].isin(active)
    ].copy()

    # 8. Convert times into seconds
    if len(departure_time) == 5:
        departure_time += ":00"

    requested_seconds = time_to_seconds(departure_time)

    results["departure_seconds"] = (
        results["departure"].apply(time_to_seconds)
    )

    results["arrival_seconds"] = (
        results["arrival"].apply(time_to_seconds)
    )

    # 9. Filter trains departing at or after requested time
    results = results[
        results["departure_seconds"] >= requested_seconds
    ].copy()

    # 10. Calculate journey duration
    results["journey_duration_min"] = (
        results["arrival_seconds"] -
        results["departure_seconds"]
    ) / 60

    # Remove invalid durations
    results = results[
        results["journey_duration_min"] >= 0
    ].copy()

    # 11. Handle trains visiting the origin more than once
    # Keep the shortest valid journey for each trip
    results = results.sort_values(
        ["trip_id", "journey_duration_min", "departure_seconds"]
    )

    results = results.drop_duplicates(
        subset=["trip_id"],
        keep="first"
    )

    # 12. Sort chronologically
    results = results.sort_values(
        ["departure_seconds", "arrival_seconds"]
    ).copy()

    # 13. Calculate waiting time
    results["waiting_time_min"] = (
        results["departure_seconds"] -
        requested_seconds
    ) / 60

    # 14. Calculate gap until the next departure
    results["next_train_gap_min"] = (
        results["departure_seconds"].shift(-1) -
        results["departure_seconds"]
    ) / 60

    # 15. Return next scheduled journeys
    return results[
        [
            "trip_id",
            "service_id",
            "route_short_name",
            "origin_platform_id",
            "destination_platform_id",
            "departure",
            "arrival",
            "journey_duration_min",
            "waiting_time_min",
            "next_train_gap_min"
        ]
    ].head(limit).reset_index(drop=True)

In [32]:
find_journeys(
    origin="Central Station",
    destination="Parramatta Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,route_short_name,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
3198,T2,17:32:01,18:36:30,64.483333,2.016667,2.000000
1258,T1,17:34:01,17:59:00,24.983333,4.016667,0.983333
296,T2,17:35:00,18:21:30,46.500000,5.000000,2.016667
214,T1,17:37:01,18:02:00,24.983333,7.016667,6.000000
1558,T1,17:43:01,18:08:00,24.983333,13.016667,3.000000


In [33]:
find_journeys(
    origin="Parramatta Station",
    destination="Central Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,route_short_name,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
2224,T1,17:32:00,17:58:00,26.000000,2.000000,11.000000
1570,T1,17:43:00,18:09:00,26.000000,13.000000,0.516667
3302,T2,17:43:31,18:28:30,44.983333,13.516667,3.483333
892,T1,17:47:00,18:13:00,26.000000,17.000000,11.000000
1857,T1,17:58:00,18:24:00,26.000000,28.000000,0.516667


In [34]:
# Check whether multiple trips have identical departure/arrival times
duplicate_trains = valid_journeys[
    valid_journeys.duplicated(
        subset=[
            "central_departure",
            "parramatta_arrival",
            "route_short_name"
        ],
        keep=False
    )
]

print("Potential duplicates:", len(duplicate_trains))

display(
    duplicate_trains[
        [
            "trip_id",
            "service_id",
            "route_short_name",
            "central_departure",
            "parramatta_arrival"
        ]
    ].head(10)
)

Potential duplicates: 0


,trip_id,service_id,route_short_name,central_departure,parramatta_arrival


In [46]:
find_journeys(
    origin="Central Station",
    destination="Strathfield Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
0,40-E.1324.165.124.A.8.92002717,TA+hv000+7,T2,2000339,2135238,17:32:00,17:56:00,24.000000,2.000000,2.016667
1,152P.1324.165.16.A.8.91999393,TA+bk000+7,T1,2000338,2135236,17:34:01,17:46:30,12.483333,4.016667,0.983333
2,11-R.1324.165.16.B.8.92001395,TA+bk000+7,T2,2000339,2135238,17:35:00,18:00:00,25.000000,5.000000,1.016667
3,192S.1324.165.16.B.8.92002023,TA+bk000+7,T9,2000332,2135233,17:36:01,17:49:00,12.983333,6.016667,1.000000
4,113E.1324.165.48.T.8.92000803,TA+bk000+7,T1,2000338,2135236,17:37:01,17:49:30,12.483333,7.016667,2.983333


In [47]:
results = find_journeys(
    origin="Central Station",
    destination="Strathfield Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=100
)

display(
    results[
        results["trip_id"] == "78-P.1324.165.16.A.8.92002733"
    ]
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
12,78-P.1324.165.16.A.8.92002733,TA+bk000+7,T2,2000339,2135238,17:50:00,18:15:00,25.0,20.0,1.016667


In [48]:
find_journeys(
    origin="Town Hall Station",
    destination="Central Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=5
)

,trip_id,service_id,route_short_name,origin_platform_id,destination_platform_id,departure,arrival,journey_duration_min,waiting_time_min,next_train_gap_min
0,152N.1324.165.16.A.8.92003560,TA+bk000+7,T1,2000392,2000338,17:30:00,17:33:00,3.0,0.0,1.0
1,11-R.1324.165.16.B.8.92001395,TA+bk000+7,T2,2000391,2000339,17:31:00,17:34:00,3.0,1.0,1.7
2,43-V.1324.165.120.A.8.91999817,TA+hv000+7,T8,2000396,2000343,17:32:42,17:46:00,13.3,2.7,0.3
3,601F.1324.165.48.T.8.92004281,TA+bk000+7,T4,2000394,2000345,17:33:00,17:35:00,2.0,3.0,0.0
4,113D.1324.165.48.T.8.92000801,TA+bk000+7,T1,2000392,2000338,17:33:00,17:36:00,3.0,3.0,2.7


In [49]:
t8_trip = "43-V.1324.165.120.A.8.91999817"

t8_stops = train_stop_times[
    train_stop_times["trip_id"] == t8_trip
].merge(
    stops[["stop_id", "stop_name"]],
    on="stop_id",
    how="left"
)

t8_stops["stop_sequence"] = pd.to_numeric(
    t8_stops["stop_sequence"]
)

display(
    t8_stops.sort_values("stop_sequence")[
        [
            "stop_sequence",
            "stop_name",
            "arrival_time",
            "departure_time"
        ]
    ]
)

,stop_sequence,stop_name,arrival_time,departure_time
0,1,"Central Station, Platform 17",17:29:01,17:29:01
1,2,"Town Hall Station, Platform 6",17:31:42,17:32:42
2,3,"Wynyard Station, Platform 6",17:34:30,17:35:30
3,4,"Circular Quay Station, Platform 2",17:38:00,17:38:30
4,5,"St James Station, Platform 2",17:41:18,17:41:48
5,6,"Museum Station, Platform 2",17:43:12,17:43:42
6,7,"Central Station, Platform 23",17:46:00,17:47:00
7,8,"Green Square Station, Platform 2",17:50:18,17:50:48
8,9,"Mascot Station, Platform 2",17:54:00,17:54:30
9,10,"Domestic Airport Station, Platform 2",17:57:00,17:57:30


In [50]:
journeys = find_journeys(
    origin="Town Hall Station",
    destination="Central Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=10
)

recommended = journeys.loc[
    journeys["arrival"].apply(time_to_seconds).idxmin()
]

print("Recommended journey:")
print("Line:", recommended["route_short_name"])
print("Departure:", recommended["departure"])
print("Arrival:", recommended["arrival"])
print("Journey duration:", round(recommended["journey_duration_min"], 1), "minutes")
print("Waiting time:", round(recommended["waiting_time_min"], 1), "minutes")

Recommended journey:
Line: T1
Departure: 17:30:00
Arrival: 17:33:00
Journey duration: 3.0 minutes
Waiting time: 0.0 minutes


In [51]:
routes = pd.read_csv(
    gtfs_dir / "routes.txt",
    dtype=str
)

print("Total routes:", len(routes))

display(
    routes.groupby(
        ["agency_id", "route_type"],
        dropna=False
    ).agg(
        route_count=("route_id", "count"),
        route_names=("route_short_name", lambda x: ", ".join(
            sorted(x.dropna().unique())
        ))
    ).reset_index()
)

Total routes: 10237


,agency_id,route_type,route_count,route_names
0,2447,700,11,"160, 161, 162, 163, 164, 165, 166, 167, 168, 1..."
1,2447,712,95,"S850, S851, S852, S853, S854, S855, S856, S857..."
2,2448,700,25,"136, 137, 138, 140, 141, 145, 177, 178, 179, 1..."
3,2448,712,419,"0011, 0012, 0032, 0037, 0050, 0051, 0052, 0061..."
4,2449,700,6,"130, 131, 132, 133, 134, 135"
...,...,...,...,...
784,SLR,900,3,"L2, L3, LX"
785,SMNW,401,1,M1
786,X000,2,4,"BMT, HUN, SCO, SHL"
787,X0000,2,3,"BMT, CCN, SCO"


In [52]:
train_routes = routes[routes["agency_id"] == "x0001"]

In [53]:
selected_routes = routes[
    routes["agency_id"].isin([
        "x0001",  # Sydney Trains
        "SMNW",   # Sydney Metro
        "SLR"     # Sydney Light Rail
    ])
].copy()

mode_mapping = {
    "x0001": "Train",
    "SMNW": "Metro",
    "SLR": "Light Rail"
}

selected_routes["transport_mode"] = (
    selected_routes["agency_id"].map(mode_mapping)
)

display(
    selected_routes[
        ["route_short_name", "transport_mode", "route_type"]
    ].sort_values(["transport_mode", "route_short_name"])
)

,route_short_name,transport_mode,route_type
10154,L2,Light Rail,900
10155,L3,Light Rail,900
10157,LX,Light Rail,900
3272,M1,Metro,401
2320,T1,Train,2
2321,T1,Train,2
2322,T2,Train,2
2323,T3,Train,2
2324,T4,Train,2
2325,T5,Train,2


In [54]:
all_transport_trips = trips.merge(
    selected_routes[
        ["route_id", "route_short_name", "transport_mode"]
    ],
    on="route_id",
    how="inner"
)

print("Total multimodal trips:", len(all_transport_trips))

display(
    all_transport_trips.groupby("transport_mode").agg(
        trip_count=("trip_id", "count"),
        route_count=("route_id", "nunique")
    )
)

Total multimodal trips: 58455


,trip_count,route_count
transport_mode,,
Light Rail,7790,3
Metro,2367,1
Train,48298,10


In [55]:
# Get trip IDs for Train, Metro and Light Rail
all_trip_ids = set(all_transport_trips["trip_id"])

# Keep stop times belonging to those trips
all_transport_stop_times = stop_times[
    stop_times["trip_id"].isin(all_trip_ids)
].copy()

# Add route and transport mode information
all_transport_stop_times = all_transport_stop_times.merge(
    all_transport_trips[
        ["trip_id", "route_short_name", "transport_mode"]
    ],
    on="trip_id",
    how="left"
)

print("Total stop-time records:", len(all_transport_stop_times))

display(
    all_transport_stop_times.groupby("transport_mode").agg(
        stop_time_records=("stop_id", "count"),
        unique_stops=("stop_id", "nunique"),
        unique_trips=("trip_id", "nunique")
    )
)

Total stop-time records: 938621


,stop_time_records,unique_stops,unique_trips
transport_mode,,,
Light Rail,110048,40,7790
Metro,68749,62,2367
Train,759824,466,48298


In [56]:
# Get all stops used by our three transport modes
used_stop_ids = set(all_transport_stop_times["stop_id"])

# Select relevant stops
transport_stops = stops[
    stops["stop_id"].isin(used_stop_ids)
].copy()

# Create a station identifier
transport_stops["station_id"] = (
    transport_stops["parent_station"]
    .fillna(transport_stops["stop_id"])
)

# Attach transport mode information
stop_modes = (
    all_transport_stop_times[
        ["stop_id", "transport_mode"]
    ]
    .drop_duplicates()
)

transport_stops = transport_stops.merge(
    stop_modes,
    on="stop_id",
    how="left"
)

print("Total transport stops:", len(transport_stops))

display(
    transport_stops[
        [
            "stop_id",
            "stop_name",
            "station_id",
            "transport_mode"
        ]
    ].head(20)
)

Total transport stops: 568


,stop_id,stop_name,station_id,transport_mode
0,2113341,"Macquarie Park Station, Platform 1",211340,Metro
1,2113342,"Macquarie Park Station, Platform 2",211340,Metro
2,2113351,"Macquarie University Station, Platform 1",211310,Metro
3,2113352,"Macquarie University Station, Platform 2",211310,Metro
4,2122231,"Eastwood Station, Platform 1",212210,Train
5,2122232,"Eastwood Station, Platform 2",212210,Train
6,2122233,"Eastwood Station, Platform 3",212210,Train
7,2122234,"Eastwood Station, Platform 4",212210,Train
8,2147421,"Seven Hills Station, Platform 1",214710,Train
9,2147422,"Seven Hills Station, Platform 2",214710,Train


In [57]:
# Count the transport modes available at each station
station_modes = (
    transport_stops
    .groupby("station_id")
    .agg(
        transport_modes=(
            "transport_mode",
            lambda x: sorted(x.dropna().unique())
        ),
        platform_count=("stop_id", "nunique")
    )
    .reset_index()
)

# Count how many different modes each station supports
station_modes["mode_count"] = (
    station_modes["transport_modes"].apply(len)
)

# Keep stations with more than one transport mode
multimodal_stations = station_modes[
    station_modes["mode_count"] > 1
].copy()

print("Multimodal stations:", len(multimodal_stations))

display(multimodal_stations.head(20))

Multimodal stations: 9


,station_id,transport_modes,platform_count,mode_count
4,200020,"[Light Rail, Train]",5,2
5,200030,"[Metro, Train]",4,2
9,200060,"[Light Rail, Metro, Train]",29,3
11,200070,"[Light Rail, Train]",8,2
12,200080,"[Light Rail, Train]",6,2
37,204420,"[Metro, Train]",6,2
50,206710,"[Metro, Train]",4,2
76,212110,"[Metro, Train]",5,2
139,220010,"[Metro, Train]",4,2


In [58]:
# Get parent station names
parent_stations = stops[
    stops["location_type"] == "1"
][["stop_id", "stop_name"]].drop_duplicates()

# Add station names to our multimodal stations
multimodal_stations_named = multimodal_stations.merge(
    parent_stations,
    left_on="station_id",
    right_on="stop_id",
    how="left"
)

display(
    multimodal_stations_named[
        [
            "station_id",
            "stop_name",
            "transport_modes",
            "platform_count"
        ]
    ].sort_values("stop_name")
)

,station_id,stop_name,transport_modes,platform_count
8,220010,Bankstown Station,"[Metro, Train]",4
2,200060,Central Station,"[Light Rail, Metro, Train]",29
6,206710,Chatswood Station,"[Metro, Train]",4
0,200020,Circular Quay,"[Light Rail, Train]",5
7,212110,Epping Station,"[Metro, Train]",5
1,200030,Martin Place Station,"[Metro, Train]",4
5,204420,Sydenham Station,"[Metro, Train]",6
3,200070,Town Hall Station,"[Light Rail, Train]",8
4,200080,Wynyard Station,"[Light Rail, Train]",6


In [59]:
import numpy as np

# Get parent station information
station_locations = stops[
    stops["stop_id"].isin(
        transport_stops["station_id"]
    )
][
    ["stop_id", "stop_name", "stop_lat", "stop_lon"]
].copy()

station_locations = station_locations.rename(
    columns={"stop_id": "station_id"}
)

station_locations["stop_lat"] = pd.to_numeric(
    station_locations["stop_lat"], errors="coerce"
)

station_locations["stop_lon"] = pd.to_numeric(
    station_locations["stop_lon"], errors="coerce"
)

station_locations = station_locations.dropna(
    subset=["stop_lat", "stop_lon"]
)

# Attach available transport modes
station_locations = station_locations.merge(
    station_modes[
        ["station_id", "transport_modes"]
    ],
    on="station_id",
    how="left"
)

print("Stations with coordinates:", len(station_locations))

display(station_locations.head(10))

Stations with coordinates: 200


,station_id,stop_name,stop_lat,stop_lon,transport_modes
0,211310,Macquarie University Station,-33.777282,151.118267,[Metro]
1,211340,Macquarie Park Station,-33.785080,151.128374,[Metro]
2,212210,Eastwood Station,-33.790301,151.082297,[Train]
3,214710,Seven Hills Station,-33.774351,150.936123,[Train]
4,214810,Blacktown Station,-33.768544,150.907080,[Train]
5,215020,Parramatta Station,-33.817490,151.005325,[Train]
6,2153477,Norwest Station,-33.734617,150.963522,[Metro]
7,2153478,Bella Vista Station,-33.730566,150.944020,[Metro]
8,2154391,Castle Hill Station,-33.731626,151.007288,[Metro]
9,2154392,Hills Showground Station,-33.727912,150.987025,[Metro]


In [64]:
%pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 3.1 MB/s  0:00:03 eta 0:00:020m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 5.5 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.


In [65]:
from sklearn.neighbors import BallTree

print("scikit-learn installed successfully!")

scikit-learn installed successfully!


In [66]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# Convert station coordinates to radians
coords = np.radians(
    station_locations[["stop_lat", "stop_lon"]].to_numpy()
)

# Build spatial search tree
tree = BallTree(coords, metric="haversine")

# Search within 500 metres
earth_radius_m = 6371000
radius_m = 500

nearby_indices, nearby_distances = tree.query_radius(
    coords,
    r=radius_m / earth_radius_m,
    return_distance=True,
    sort_results=True
)

connections = []

for i, (indices, distances) in enumerate(
    zip(nearby_indices, nearby_distances)
):
    for j, distance in zip(indices, distances):

        # Avoid matching a station with itself
        # and duplicate station pairs
        if j <= i:
            continue

        station_a = station_locations.iloc[i]
        station_b = station_locations.iloc[j]

        # Keep pairs that offer different transport modes
        modes_a = set(station_a["transport_modes"])
        modes_b = set(station_b["transport_modes"])

        if not (modes_a - modes_b or modes_b - modes_a):
            continue

        connections.append({
            "from_station_id": station_a["station_id"],
            "from_station": station_a["stop_name"],
            "from_modes": ", ".join(sorted(modes_a)),
            "to_station_id": station_b["station_id"],
            "to_station": station_b["stop_name"],
            "to_modes": ", ".join(sorted(modes_b)),
            "distance_m": round(distance * earth_radius_m),
        })

walking_connections = pd.DataFrame(connections)

print("Potential walking connections:", len(walking_connections))

display(
    walking_connections.sort_values("distance_m").head(20)
)

Potential walking connections: 14


,from_station_id,from_station,from_modes,to_station_id,to_station,to_modes,distance_m
0,200070,Town Hall Station,"Light Rail, Train",200066,Gadigal Station,Metro,173
1,200070,Town Hall Station,"Light Rail, Train",2000171,QVB,Light Rail,207
6,200080,Wynyard Station,"Light Rail, Train",2000177,Bridge Street Light Rail,Light Rail,236
3,2000171,QVB,Light Rail,200066,Gadigal Station,Metro,245
12,200040,Museum Station,Train,200066,Gadigal Station,Metro,299
5,200060,Central Station,"Light Rail, Metro, Train",2000168,Haymarket Light Rail,Light Rail,305
10,200030,Martin Place Station,"Metro, Train",200050,St James Station,Train,351
2,200070,Town Hall Station,"Light Rail, Train",200040,Museum Station,Train,359
8,206010,North Sydney Station,Train,206044,Victoria Cross Station,Metro,381
13,200050,St James Station,Train,200066,Gadigal Station,Metro,399


In [67]:
import numpy as np

walking_connections["estimated_walk_distance_m"] = (
    walking_connections["distance_m"] * 1.3
)

walking_connections["transfer_time_min"] = np.ceil(
    walking_connections["estimated_walk_distance_m"] / 75 + 2
).astype(int)

display(
    walking_connections[
        [
            "from_station",
            "to_station",
            "distance_m",
            "transfer_time_min"
        ]
    ].sort_values("distance_m")
)

,from_station,to_station,distance_m,transfer_time_min
0,Town Hall Station,Gadigal Station,173,5
1,Town Hall Station,QVB,207,6
6,Wynyard Station,Bridge Street Light Rail,236,7
3,QVB,Gadigal Station,245,7
12,Museum Station,Gadigal Station,299,8
5,Central Station,Haymarket Light Rail,305,8
10,Martin Place Station,St James Station,351,9
2,Town Hall Station,Museum Station,359,9
8,North Sydney Station,Victoria Cross Station,381,9
13,St James Station,Gadigal Station,399,9


In [68]:
print("transfers.txt:", (gtfs_dir / "transfers.txt").exists())
print("pathways.txt:", (gtfs_dir / "pathways.txt").exists())

transfers.txt: False
pathways.txt: True


In [69]:
pathways = pd.read_csv(
    gtfs_dir / "pathways.txt",
    dtype=str
)

print("Total pathways:", len(pathways))
print("\nColumns:")
print(pathways.columns.tolist())

display(pathways.head(10))

Total pathways: 6924

Columns:
['pathway_id', 'from_stop_id', 'to_stop_id', 'pathway_mode', 'is_bidirectional', 'traversal_time']


,pathway_id,from_stop_id,to_stop_id,pathway_mode,is_bidirectional,traversal_time
0,257861_257810_ENT2_walkway_1,257861,257810_ENT2,1,1,30
1,257862_257810_ENT1_walkway_1,257862,257810_ENT1,1,1,30
2,257811_257810_ENT1_walkway_1,257811,257810_ENT1,1,1,30
3,257812_257810_ENT1_walkway_1,257812,257810_ENT1,1,1,84
4,257821_257810_ENT2_walkway_1,257821,257810_ENT2,1,1,108
5,257823_257810_ENT2_walkway_1,257823,257810_ENT2,1,1,108
6,257810_ENT1_257810_LVX1_walkway_1,257810_ENT1,257810_LVX1,1,1,42
7,257810_LVX1_257810_ENT2_walkway_1,257810_LVX1,257810_ENT2,1,1,30
8,257821_257823_walkway_1,257821,257823,1,1,120
9,257991_257920_DP2_walkway_1,257991,257920_DP2,1,1,42


In [70]:
# All platforms used by Train, Metro and Light Rail
platform_ids = set(all_transport_stop_times["stop_id"])

# All stop IDs referenced by pathways
pathway_stop_ids = set(pathways["from_stop_id"]) | set(
    pathways["to_stop_id"]
)

# Platforms represented in the pathway network
matched_platforms = platform_ids & pathway_stop_ids

print("Total transport platform IDs:", len(platform_ids))
print("Platforms found in pathways:", len(matched_platforms))
print(
    "Platform coverage:",
    round(len(matched_platforms) / len(platform_ids) * 100, 1),
    "%"
)

# Check pathway records involving timetable platforms
matched_pathways = pathways[
    pathways["from_stop_id"].isin(platform_ids)
    | pathways["to_stop_id"].isin(platform_ids)
]

print("Pathway records involving platforms:", len(matched_pathways))

display(matched_pathways.head(10))

Total transport platform IDs: 568
Platforms found in pathways: 566
Platform coverage: 99.6 %
Pathway records involving platforms: 1606


,pathway_id,from_stop_id,to_stop_id,pathway_mode,is_bidirectional,traversal_time
79,2770591_277010_DP5_elevator_1,2770591,277010_DP5,5,1,60
80,2770592_277010_DP5_elevator_1,2770592,277010_DP5,5,1,60
81,2770591_277010_CNC_stairs_1,2770591,277010_CNC,2,1,30
82,2770592_277010_CNC_stairs_1,2770592,277010_CNC,2,1,30
83,2770591_277010_DP7_walkway_1,2770591,277010_DP7,1,1,30
84,2770592_277010_DP7_walkway_1,2770592,277010_DP7,1,1,30
85,2770593_277010_DP6_elevator_1,2770593,277010_DP6,5,1,60
86,2770594_277010_DP6_elevator_1,2770594,277010_DP6,5,1,60
87,2770593_277010_CNC_stairs_1,2770593,277010_CNC,2,1,30
88,2770594_277010_CNC_stairs_1,2770594,277010_CNC,2,1,30


In [71]:
%pip install networkx

  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
Using cached networkx-3.7-py3-none-any.whl (2.1 MB)
Note: you may need to restart the kernel to use updated packages.


In [72]:
import networkx as nx
import pandas as pd

G = nx.DiGraph()

for row in pathways.itertuples(index=False):

    # Skip pathways without valid traversal times
    if pd.isna(row.traversal_time):
        continue

    traversal_time = float(row.traversal_time)

    if traversal_time < 0:
        continue

    # Add the permitted direction
    G.add_edge(
        row.from_stop_id,
        row.to_stop_id,
        weight=traversal_time
    )

    # Add reverse direction only when allowed
    if row.is_bidirectional == "1":
        G.add_edge(
            row.to_stop_id,
            row.from_stop_id,
            weight=traversal_time
        )

print("Pathway nodes:", G.number_of_nodes())
print("Pathway edges:", G.number_of_edges())

Pathway nodes: 4988
Pathway edges: 12162


In [73]:
central_platforms = transport_stops[
    transport_stops["station_id"] == "200060"
][
    ["stop_id", "stop_name", "transport_mode"]
].drop_duplicates()

display(central_platforms)

,stop_id,stop_name,transport_mode
206,2000321,"Central Station, Platform 1",Train
207,2000322,"Central Station, Platform 2",Train
208,2000323,"Central Station, Platform 3",Train
209,2000324,"Central Station, Platform 4",Train
210,2000325,"Central Station, Platform 5",Train
211,2000326,"Central Station, Platform 6",Train
212,2000327,"Central Station, Platform 7",Train
213,2000328,"Central Station, Platform 8",Train
214,2000329,"Central Station, Platform 9",Train
215,2000330,"Central Station, Platform 10",Train


In [74]:
transfer_results = []

platform_records = central_platforms.to_dict("records")

for origin in platform_records:
    for destination in platform_records:

        if origin["stop_id"] == destination["stop_id"]:
            continue

        try:
            seconds = nx.shortest_path_length(
                G,
                source=origin["stop_id"],
                target=destination["stop_id"],
                weight="weight"
            )

            transfer_results.append({
                "from_platform": origin["stop_name"],
                "from_mode": origin["transport_mode"],
                "to_platform": destination["stop_name"],
                "to_mode": destination["transport_mode"],
                "transfer_seconds": seconds,
                "transfer_minutes": round(seconds / 60, 1)
            })

        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

central_transfers = pd.DataFrame(transfer_results)

print("Reachable Central platform pairs:", len(central_transfers))

display(central_transfers.head(20))

Reachable Central platform pairs: 812


,from_platform,from_mode,to_platform,to_mode,transfer_seconds,transfer_minutes
0,"Central Station, Platform 1",Train,"Central Station, Platform 2",Train,60.0,1.0
1,"Central Station, Platform 1",Train,"Central Station, Platform 3",Train,60.0,1.0
2,"Central Station, Platform 1",Train,"Central Station, Platform 4",Train,168.0,2.8
3,"Central Station, Platform 1",Train,"Central Station, Platform 5",Train,168.0,2.8
4,"Central Station, Platform 1",Train,"Central Station, Platform 6",Train,150.0,2.5
5,"Central Station, Platform 1",Train,"Central Station, Platform 7",Train,150.0,2.5
6,"Central Station, Platform 1",Train,"Central Station, Platform 8",Train,132.0,2.2
7,"Central Station, Platform 1",Train,"Central Station, Platform 9",Train,132.0,2.2
8,"Central Station, Platform 1",Train,"Central Station, Platform 10",Train,156.0,2.6
9,"Central Station, Platform 1",Train,"Central Station, Platform 11",Train,156.0,2.6


In [75]:
# Keep transfers between different transport modes
cross_mode_transfers = central_transfers[
    central_transfers["from_mode"] !=
    central_transfers["to_mode"]
].copy()

print(
    "Cross-mode platform connections:",
    len(cross_mode_transfers)
)

# Summarise transfer times by mode combination
cross_mode_summary = (
    cross_mode_transfers
    .groupby(["from_mode", "to_mode"])
    .agg(
        connections=("transfer_minutes", "count"),
        min_minutes=("transfer_minutes", "min"),
        median_minutes=("transfer_minutes", "median"),
        max_minutes=("transfer_minutes", "max")
    )
    .reset_index()
)

display(cross_mode_summary)

Cross-mode platform connections: 252


,from_mode,to_mode,connections,min_minutes,median_minutes,max_minutes
0,Light Rail,Metro,6,4.5,4.5,4.5
1,Light Rail,Train,72,2.8,4.4,5.4
2,Metro,Light Rail,6,4.5,4.5,4.5
3,Metro,Train,48,1.5,2.4,4.1
4,Train,Light Rail,72,2.8,4.4,5.4
5,Train,Metro,48,1.5,2.4,4.1


In [77]:
import networkx as nx
import pandas as pd
from itertools import permutations

transfer_records = []

# Group platforms by parent station
for station_id, group in transport_stops.groupby("station_id"):

    platforms = (
        group[
            ["stop_id", "stop_name", "transport_mode"]
        ]
        .drop_duplicates(subset=["stop_id"])
        .to_dict("records")
    )

    # Test every directed platform pair within the station
    for origin, destination in permutations(platforms, 2):

        try:
            seconds = nx.shortest_path_length(
                G,
                source=origin["stop_id"],
                target=destination["stop_id"],
                weight="weight"
            )

            transfer_records.append({
                "station_id": station_id,
                "from_stop_id": origin["stop_id"],
                "to_stop_id": destination["stop_id"],
                "from_mode": origin["transport_mode"],
                "to_mode": destination["transport_mode"],
                "pathway_seconds": seconds,
                "pathway_minutes": round(seconds / 60, 2)
            })

        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

platform_transfers = pd.DataFrame(transfer_records)

print("Total platform transfer connections:", len(platform_transfers))

display(
    platform_transfers.groupby(
        ["from_mode", "to_mode"]
    ).agg(
        connections=("pathway_seconds", "count"),
        median_minutes=("pathway_minutes", "median")
    ).reset_index()
)

Total platform transfer connections: 2108


,from_mode,to_mode,connections,median_minutes
0,Light Rail,Light Rail,44,0.60
1,Light Rail,Metro,6,4.50
2,Light Rail,Train,98,4.40
3,Metro,Light Rail,6,4.50
4,Metro,Metro,62,1.40
5,Metro,Train,74,2.45
6,Train,Light Rail,98,4.40
7,Train,Metro,74,2.45
8,Train,Train,1646,2.30


In [78]:
# Define additional transfer buffers in seconds
transfer_buffers = {
    ("Train", "Train"): 180,
    ("Metro", "Metro"): 180,
    ("Light Rail", "Light Rail"): 120,
    ("Train", "Metro"): 240,
    ("Metro", "Train"): 240,
    ("Train", "Light Rail"): 300,
    ("Light Rail", "Train"): 300,
    ("Metro", "Light Rail"): 300,
    ("Light Rail", "Metro"): 300
}

# Assign buffer based on transport modes
platform_transfers["transfer_buffer_seconds"] = (
    platform_transfers.apply(
        lambda row: transfer_buffers[
            (row["from_mode"], row["to_mode"])
        ],
        axis=1
    )
)

# Calculate total minimum connection time
platform_transfers["minimum_transfer_seconds"] = (
    platform_transfers["pathway_seconds"]
    + platform_transfers["transfer_buffer_seconds"]
)

platform_transfers["minimum_transfer_minutes"] = (
    platform_transfers["minimum_transfer_seconds"] / 60
).round(2)

# Check for duplicate platform pairs
duplicates = platform_transfers.duplicated(
    subset=["from_stop_id", "to_stop_id"]
).sum()

print("Duplicate platform pairs:", duplicates)

display(
    platform_transfers.groupby(
        ["from_mode", "to_mode"]
    )["minimum_transfer_minutes"]
    .agg(["min", "median", "max"])
    .round(1)
)

Duplicate platform pairs: 0


min  median   max
from_mode  to_mode                      
Light Rail Light Rail  2.4     2.6   4.5
           Metro       9.5     9.5   9.5
           Train       7.2     9.4  10.4
Metro      Light Rail  9.5     9.5   9.5
           Metro       3.6     4.4   7.5
           Train       5.0     6.4   8.9
Train      Light Rail  7.2     9.4  10.4
           Metro       5.0     6.4   8.9
           Train       3.4     5.3   9.3

In [79]:
# Build a clean multimodal timetable
timetable = all_transport_stop_times[
    [
        "trip_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "stop_sequence",
        "route_short_name",
        "transport_mode"
    ]
].copy()

# Add service IDs to identify which dates trips operate
timetable = timetable.merge(
    all_transport_trips[
        ["trip_id", "service_id"]
    ],
    on="trip_id",
    how="left",
    validate="many_to_one"
)

# Convert stop sequences to integers
timetable["stop_sequence"] = pd.to_numeric(
    timetable["stop_sequence"]
)

# Convert GTFS times to seconds
timetable["arrival_seconds"] = (
    timetable["arrival_time"].apply(time_to_seconds)
)

timetable["departure_seconds"] = (
    timetable["departure_time"].apply(time_to_seconds)
)

# Add parent station IDs
timetable = timetable.merge(
    transport_stops[
        ["stop_id", "station_id"]
    ].drop_duplicates("stop_id"),
    on="stop_id",
    how="left",
    validate="many_to_one"
)

print("Timetable records:", len(timetable))
print("Missing station IDs:", timetable["station_id"].isna().sum())

display(timetable.head(10))

Timetable records: 938621
Missing station IDs: 0


,trip_id,stop_id,arrival_time,departure_time,stop_sequence,route_short_name,transport_mode,service_id,arrival_seconds,departure_seconds,station_id
0,101A.1324.165.16.T.8.92005031,207262,05:02:00,05:02:00,1,T9,Train,TA+bk000+7,18120,18120,207210
1,101A.1324.165.16.T.8.92005031,207191,05:03:42,05:04:12,2,T9,Train,TA+bk000+7,18222,18252,207110
2,101A.1324.165.16.T.8.92005031,2070141,05:05:48,05:06:18,3,T9,Train,TA+bk000+7,18348,18378,207010
3,101A.1324.165.16.T.8.92005031,2069131,05:08:06,05:08:36,4,T9,Train,TA+bk000+7,18486,18516,206910
4,101A.1324.165.16.T.8.92005031,2067141,05:11:00,05:12:00,5,T9,Train,TA+bk000+7,18660,18720,206710
5,101A.1324.165.16.T.8.92005031,206451,05:14:00,05:14:30,6,T9,Train,TA+bk000+7,18840,18870,206410
6,101A.1324.165.16.T.8.92005031,2065152,05:16:36,05:17:06,7,T9,Train,TA+bk000+7,18996,19026,206520
7,101A.1324.165.16.T.8.92005031,2065161,05:19:00,05:19:30,8,T9,Train,TA+bk000+7,19140,19170,206510
8,101A.1324.165.16.T.8.92005031,2060111,05:21:12,05:21:42,9,T9,Train,TA+bk000+7,19272,19302,206020
9,101A.1324.165.16.T.8.92005031,2060101,05:24:00,05:25:00,10,T9,Train,TA+bk000+7,19440,19500,206010


In [80]:
from datetime import datetime

def get_active_services(travel_date):
    """
    Return the GTFS service IDs operating on a given date.

    travel_date format: YYYY-MM-DD
    """

    date_obj = datetime.strptime(travel_date, "%Y-%m-%d")

    date_str = date_obj.strftime("%Y%m%d")
    weekday = date_obj.strftime("%A").lower()

    # Regular services operating on this weekday
    regular_services = calendar[
        (calendar["start_date"] <= date_str)
        & (calendar["end_date"] >= date_str)
        & (calendar[weekday] == "1")
    ]

    active_services = set(regular_services["service_id"])

    # Special service additions and cancellations
    exceptions = calendar_dates[
        calendar_dates["date"] == date_str
    ]

    removed_services = set(
        exceptions.loc[
            exceptions["exception_type"] == "2",
            "service_id"
        ]
    )

    added_services = set(
        exceptions.loc[
            exceptions["exception_type"] == "1",
            "service_id"
        ]
    )

    return (active_services - removed_services) | added_services

In [81]:
travel_date = "2026-10-28"

active_services = get_active_services(travel_date)

daily_timetable = timetable[
    timetable["service_id"].isin(active_services)
].copy()

print("Active service IDs:", len(active_services))
print("Daily timetable records:", len(daily_timetable))

display(
    daily_timetable.groupby("transport_mode").agg(
        trips=("trip_id", "nunique"),
        stop_times=("stop_id", "count")
    )
)

Active service IDs: 662
Daily timetable records: 59541


,trips,stop_times
transport_mode,,
Light Rail,563,7997
Metro,434,9114
Train,2708,42430


In [82]:
def find_departures(station_name, departure_time, limit=20):
    # Find the station ID
    matching_stations = stops[
        (stops["stop_name"] == station_name)
        & (stops["location_type"] == "1")
    ]

    if matching_stations.empty:
        raise ValueError(f"Station not found: {station_name}")

    station_ids = set(matching_stations["stop_id"])

    # Find departures from any platform at this station
    departures = daily_timetable[
        daily_timetable["station_id"].isin(station_ids)
    ].copy()

    # Convert requested departure time to seconds
    requested_seconds = time_to_seconds(
        departure_time if len(departure_time) == 8
        else departure_time + ":00"
    )

    # Only keep departures after the requested time
    departures = departures[
        departures["departure_seconds"] >= requested_seconds
    ]

    # Sort by departure time
    departures = departures.sort_values("departure_seconds")

    return departures[
        [
            "trip_id",
            "route_short_name",
            "transport_mode",
            "stop_id",
            "departure_time",
            "departure_seconds",
            "stop_sequence"
        ]
    ].head(limit).reset_index(drop=True)

In [83]:
departures = find_departures(
    station_name="Central Station",
    departure_time="17:30",
    limit=20
)

display(departures)

,trip_id,route_short_name,transport_mode,stop_id,departure_time,departure_seconds,stop_sequence
0,78-N.1324.165.16.A.8.92004860,T8,Train,2000341,17:30:00,63000,15
1,136H.1324.165.16.A.8.92002355,T1,Train,2000336,17:30:01,63001,1
2,0160-006-102-015:1000,M1,Metro,2000466,17:30:02,63002,3
3,612M.1324.165.16.T.8.92000044,T4,Train,2000344,17:31:00,63060,14
4,47197-135:1001,LX,Light Rail,2010107,17:31:10,63070,8
5,47197-10561:1001,L2,Light Rail,2000447,17:31:35,63095,8
6,40-E.1324.165.124.A.8.92002717,T2,Train,2000339,17:32:00,63120,7
7,625J.1324.165.16.T.8.92002566,T4,Train,2000345,17:32:00,63120,6
8,5--R.1324.165.16.B.8.92002199,T8,Train,2000343,17:32:00,63120,7
9,78-P.1324.165.16.A.8.92002733,T2,Train,2000341,17:32:01,63121,1


In [85]:
# Select the first departure
selected_departure = departures.iloc[0]

trip_id = selected_departure["trip_id"]
origin_sequence = selected_departure["stop_sequence"]

# Find stops AFTER the selected departure
downstream_stops = daily_timetable[
    (daily_timetable["trip_id"] == trip_id)
    & (daily_timetable["stop_sequence"] > origin_sequence)
].copy()

# Add station names
station_names = stops[
    (stops["location_type"] == "1")
][["stop_id", "stop_name"]].drop_duplicates()

downstream_stops = downstream_stops.merge(
    station_names,
    left_on="station_id",
    right_on="stop_id",
    how="left",
    suffixes=("", "_station")
)

display(
    downstream_stops[
        [
            "route_short_name",
            "stop_name",
            "arrival_time",
            "stop_sequence",
            "transport_mode"
        ]
    ].sort_values("stop_sequence")
)

,route_short_name,stop_name,arrival_time,stop_sequence,transport_mode


In [86]:
selected_departure = departures.iloc[0]

trip_stops = daily_timetable[
    daily_timetable["trip_id"] == selected_departure["trip_id"]
].sort_values("stop_sequence")

print("Selected line:", selected_departure["route_short_name"])
print("Central stop sequence:", selected_departure["stop_sequence"])
print("Total stops on trip:", len(trip_stops))
print("Last stop sequence:", trip_stops["stop_sequence"].max())

display(
    trip_stops[
        ["stop_id", "arrival_time", "departure_time", "stop_sequence"]
    ].tail(10)
)

Selected line: T8
Central stop sequence: 15
Total stops on trip: 15
Last stop sequence: 15


,stop_id,arrival_time,departure_time,stop_sequence
606011,2208141,17:04:00,17:04:30,6
606012,2207281,17:06:30,17:07:00,7
606013,2207271,17:08:48,17:09:18,8
606014,2205101,17:11:24,17:11:54,9
606015,2205111,17:14:06,17:14:36,10
606016,2020101,17:16:30,17:17:00,11
606017,202091,17:19:06,17:19:36,12
606018,2020111,17:21:54,17:22:24,13
606019,2015111,17:25:30,17:26:00,14
606020,2000341,17:30:00,17:30:00,15


In [87]:
# Find the last stop sequence for every trip
last_stop_sequences = (
    daily_timetable
    .groupby("trip_id")["stop_sequence"]
    .max()
    .rename("last_stop_sequence")
)

# Add this information to our departures
departures_checked = departures.merge(
    last_stop_sequences,
    on="trip_id",
    how="left"
)

# Keep services that continue beyond Central
valid_departures = departures_checked[
    departures_checked["stop_sequence"]
    < departures_checked["last_stop_sequence"]
].copy()

print("Original departures:", len(departures))
print("Departures continuing beyond Central:", len(valid_departures))

display(
    valid_departures[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_sequence",
            "last_stop_sequence"
        ]
    ].head(15)
)

Original departures: 20
Departures continuing beyond Central: 16


,route_short_name,transport_mode,departure_time,stop_sequence,last_stop_sequence
1,T1,Train,17:30:01,1,20
2,M1,Metro,17:30:02,3,21
3,T4,Train,17:31:00,14,19
5,L2,Light Rail,17:31:35,8,14
6,T2,Train,17:32:00,7,36
7,T4,Train,17:32:00,6,19
8,T8,Train,17:32:00,7,21
9,T2,Train,17:32:01,1,26
10,T8,Train,17:32:01,1,22
11,M1,Metro,17:32:49,19,21


In [88]:
# Select the first valid departure (T1 at 17:30:01)
selected_departure = valid_departures.iloc[0]

# Get all stops after boarding
downstream_stops = daily_timetable[
    (daily_timetable["trip_id"] == selected_departure["trip_id"])
    & (
        daily_timetable["stop_sequence"]
        > selected_departure["stop_sequence"]
    )
].copy()

# Add readable station names
station_lookup = (
    stops[stops["location_type"] == "1"]
    [["stop_id", "stop_name"]]
    .drop_duplicates("stop_id")
    .rename(columns={"stop_id": "station_id"})
)

downstream_stops = downstream_stops.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Check which arrival platforms have valid transfer connections
transfer_platform_ids = set(
    platform_transfers["from_stop_id"]
)

downstream_stops["has_transfer"] = (
    downstream_stops["stop_id"].isin(transfer_platform_ids)
)

print("Line:", selected_departure["route_short_name"])
print("Departure:", selected_departure["departure_time"])
print("Downstream stops:", len(downstream_stops))
print(
    "Stops with transfer pathways:",
    downstream_stops["has_transfer"].sum()
)

display(
    downstream_stops[
        [
            "stop_name",
            "arrival_time",
            "stop_id",
            "stop_sequence",
            "has_transfer"
        ]
    ].sort_values("stop_sequence")
)

Line: T1
Departure: 17:30:01
Downstream stops: 19
Stops with transfer pathways: 19


,stop_name,arrival_time,stop_id,stop_sequence,has_transfer
0,Town Hall Station,17:32:36,2000393,2,True
1,Wynyard Station,17:35:30,2000404,3,True
2,Milsons Point Station,17:39:30,206142,4,True
3,North Sydney Station,17:41:30,2060104,5,True
4,Waverton Station,17:44:12,2060112,6,True
5,Wollstonecraft Station,17:46:18,2065162,7,True
6,St Leonards Station,17:49:30,2065153,8,True
7,Artarmon Station,17:52:12,206452,9,True
8,Chatswood Station,17:55:00,2067144,10,True
9,Roseville Station,17:57:48,2069132,11,True


In [89]:
# Select Chatswood as our test interchange
arrival = downstream_stops[
    downstream_stops["stop_name"] == "Chatswood Station"
].iloc[0]

arrival_platform = arrival["stop_id"]
arrival_seconds = arrival["arrival_seconds"]

print("Arriving at:", arrival["stop_name"])
print("Arrival time:", arrival["arrival_time"])
print("Arrival platform:", arrival_platform)

# Get valid pathways from the arrival platform
possible_transfers = platform_transfers[
    platform_transfers["from_stop_id"] == arrival_platform
].copy()

# Add the arrival time to each possible transfer
possible_transfers["earliest_boarding_seconds"] = (
    arrival_seconds
    + possible_transfers["minimum_transfer_seconds"]
)

# Find services departing from reachable platforms
connecting_services = daily_timetable.merge(
    possible_transfers[
        [
            "to_stop_id",
            "earliest_boarding_seconds",
            "minimum_transfer_minutes"
        ]
    ],
    left_on="stop_id",
    right_on="to_stop_id",
    how="inner"
)

# Keep departures within 30 minutes of arriving at Chatswood
connecting_services = connecting_services[
    (connecting_services["departure_seconds"]
     >= connecting_services["earliest_boarding_seconds"])
    & (connecting_services["departure_seconds"]
       <= arrival_seconds + 30 * 60)
    & (connecting_services["trip_id"]
       != selected_departure["trip_id"])
].copy()

# Exclude services terminating at the interchange
last_sequences = (
    daily_timetable.groupby("trip_id")["stop_sequence"].max()
)

connecting_services["last_stop_sequence"] = (
    connecting_services["trip_id"].map(last_sequences)
)

connecting_services = connecting_services[
    connecting_services["stop_sequence"]
    < connecting_services["last_stop_sequence"]
]

# Show possible connecting services
display(
    connecting_services[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_id",
            "minimum_transfer_minutes"
        ]
    ]
    .sort_values("departure_time")
    .head(20)
)

Arriving at: Chatswood Station
Arrival time: 17:55:00
Arrival platform: 2067144


,route_short_name,transport_mode,departure_time,stop_id,minimum_transfer_minutes
235,M1,Metro,18:01:04,2067143,5.0
321,M1,Metro,18:01:55,2067142,5.4
13,T1,Train,18:03:00,2067141,4.4
255,M1,Metro,18:05:04,2067143,5.0
500,M1,Metro,18:05:55,2067142,5.4
89,T1,Train,18:06:00,2067141,4.4
437,M1,Metro,18:09:04,2067143,5.0
517,M1,Metro,18:09:55,2067142,5.4
65,T9,Train,18:12:00,2067141,4.4
455,M1,Metro,18:13:04,2067143,5.0


In [90]:
# Select the earliest feasible Metro connection
metro_connection = (
    connecting_services[
        connecting_services["transport_mode"] == "Metro"
    ]
    .sort_values("departure_seconds")
    .iloc[0]
)

# Find all downstream stops on that Metro trip
metro_destinations = daily_timetable[
    (daily_timetable["trip_id"] == metro_connection["trip_id"])
    & (
        daily_timetable["stop_sequence"]
        > metro_connection["stop_sequence"]
    )
].copy()

# Add readable station names
metro_destinations = metro_destinations.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

print("First leg: T1")
print("Arrive at Chatswood:", arrival["arrival_time"])
print()
print("Second leg:", metro_connection["route_short_name"])
print("Depart Chatswood:", metro_connection["departure_time"])
print("Transfer platform:", metro_connection["stop_id"])
print()

display(
    metro_destinations[
        [
            "stop_name",
            "arrival_time",
            "stop_sequence"
        ]
    ].sort_values("stop_sequence")
)

First leg: T1
Arrive at Chatswood: 17:55:00

Second leg: M1
Depart Chatswood: 18:01:04
Transfer platform: 2067143



,stop_name,arrival_time,stop_sequence
0,North Ryde Station,18:05:56,10
1,Macquarie Park Station,18:08:06,11
2,Macquarie University Station,18:10:07,12
3,Epping Station,18:13:51,13
4,Cherrybrook Station,18:19:38,14
5,Castle Hill Station,18:22:22,15
6,Hills Showground Station,18:24:55,16
7,Norwest Station,18:27:30,17
8,Bella Vista Station,18:30:10,18
9,Kellyville Station,18:32:45,19


In [91]:
destination_name = "Macquarie University Station"

destination_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == destination_name,
        "station_id"
    ]
)

journey_options = []

for _, connection in connecting_services.iterrows():

    # Find downstream stops on this connecting trip
    destinations = daily_timetable[
        (daily_timetable["trip_id"] == connection["trip_id"])
        & (
            daily_timetable["stop_sequence"]
            > connection["stop_sequence"]
        )
        & (
            daily_timetable["station_id"].isin(destination_ids)
        )
    ]

    for _, destination in destinations.iterrows():

        journey_options.append({
            "first_line": selected_departure["route_short_name"],
            "first_departure": selected_departure["departure_time"],
            "transfer_station": "Chatswood Station",
            "transfer_arrival": arrival["arrival_time"],
            "second_line": connection["route_short_name"],
            "second_departure": connection["departure_time"],
            "destination_arrival": destination["arrival_time"],
            "transfer_minutes": round(
                (
                    connection["departure_seconds"]
                    - arrival_seconds
                ) / 60,
                2
            ),
            "total_journey_minutes": round(
                (
                    destination["arrival_seconds"]
                    - selected_departure["departure_seconds"]
                ) / 60,
                2
            )
        })

journey_options = pd.DataFrame(journey_options)

if journey_options.empty:
    print("No connecting journeys found.")
else:
    journey_options = journey_options.sort_values(
        "total_journey_minutes"
    )

    display(journey_options)

,first_line,first_departure,transfer_station,transfer_arrival,second_line,second_departure,destination_arrival,transfer_minutes,total_journey_minutes
0,T1,17:30:01,Chatswood Station,17:55:00,M1,18:01:04,18:10:07,6.07,40.1
1,T1,17:30:01,Chatswood Station,17:55:00,M1,18:05:04,18:14:07,10.07,44.1
3,T1,17:30:01,Chatswood Station,17:55:00,M1,18:09:04,18:18:07,14.07,48.1
4,T1,17:30:01,Chatswood Station,17:55:00,M1,18:13:04,18:22:07,18.07,52.1
5,T1,17:30:01,Chatswood Station,17:55:00,M1,18:17:04,18:26:07,22.07,56.1
2,T1,17:30:01,Chatswood Station,17:55:00,M1,18:21:04,18:30:07,26.07,60.1


In [92]:
def get_origin_departures(
    origin_station,
    departure_time,
    window_minutes=30
):
    # Find the parent station ID
    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin_station,
            "station_id"
        ]
    )

    if not origin_ids:
        raise ValueError(f"Station not found: {origin_station}")

    requested_seconds = time_to_seconds(
        departure_time
        if len(departure_time) == 8
        else departure_time + ":00"
    )

    # Find services within the departure window
    origin_departures = daily_timetable[
        daily_timetable["station_id"].isin(origin_ids)
        & daily_timetable["departure_seconds"].between(
            requested_seconds,
            requested_seconds + window_minutes * 60
        )
    ].copy()

    # Identify the final stop sequence of each trip
    last_sequences = (
        daily_timetable.groupby("trip_id")["stop_sequence"].max()
    )

    origin_departures["last_stop_sequence"] = (
        origin_departures["trip_id"].map(last_sequences)
    )

    # Exclude terminating services
    origin_departures = origin_departures[
        origin_departures["stop_sequence"]
        < origin_departures["last_stop_sequence"]
    ]

    return origin_departures.sort_values(
        "departure_seconds"
    ).reset_index(drop=True)

In [93]:
origin_departures = get_origin_departures(
    origin_station="Central Station",
    departure_time="17:30",
    window_minutes=30
)

print("Valid departures:", len(origin_departures))

display(
    origin_departures.groupby("transport_mode").agg(
        services=("trip_id", "nunique"),
        departures=("trip_id", "count")
    )
)

display(
    origin_departures[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_sequence"
        ]
    ].head(15)
)

Valid departures: 91


,services,departures
transport_mode,,
Light Rail,16,16
Metro,15,15
Train,55,60


,route_short_name,transport_mode,departure_time,stop_sequence
0,T1,Train,17:30:01,1
1,M1,Metro,17:30:02,3
2,T4,Train,17:31:00,14
3,L2,Light Rail,17:31:35,8
4,T4,Train,17:32:00,6
5,T2,Train,17:32:00,7
6,T8,Train,17:32:00,7
7,T8,Train,17:32:01,1
8,T2,Train,17:32:01,1
9,M1,Metro,17:32:49,19


In [94]:
# Give every origin boarding event a unique ID
first_legs = origin_departures.copy()
first_legs["boarding_id"] = range(len(first_legs))

# Keep the information needed for each boarding
boarding_events = first_legs[
    [
        "boarding_id",
        "trip_id",
        "stop_id",
        "stop_sequence",
        "departure_seconds",
        "departure_time",
        "route_short_name",
        "transport_mode"
    ]
].rename(columns={
    "stop_id": "origin_platform",
    "stop_sequence": "origin_sequence",
    "departure_seconds": "origin_departure_seconds",
    "departure_time": "origin_departure_time",
    "route_short_name": "first_line",
    "transport_mode": "first_mode"
})

# Match each boarding to stops on the same trip
first_leg_stops = boarding_events.merge(
    daily_timetable[
        [
            "trip_id",
            "stop_id",
            "station_id",
            "stop_sequence",
            "arrival_seconds",
            "arrival_time"
        ]
    ],
    on="trip_id",
    how="inner"
)

# Only keep stops AFTER the boarding event
first_leg_stops = first_leg_stops[
    first_leg_stops["stop_sequence"]
    > first_leg_stops["origin_sequence"]
].copy()

# Attach readable station names
first_leg_stops = first_leg_stops.merge(
    station_lookup,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Identify platforms with available pathway connections
transfer_platform_ids = set(
    platform_transfers["from_stop_id"]
)

first_leg_stops["has_transfer"] = (
    first_leg_stops["stop_id"].isin(transfer_platform_ids)
)

print("Origin boarding events:", len(boarding_events))
print("Total downstream stop events:", len(first_leg_stops))
print(
    "Downstream events with transfer pathways:",
    first_leg_stops["has_transfer"].sum()
)
print(
    "Unique potential interchange stations:",
    first_leg_stops.loc[
        first_leg_stops["has_transfer"],
        "station_id"
    ].nunique()
)

display(
    first_leg_stops[
        [
            "first_line",
            "first_mode",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "has_transfer"
        ]
    ].head(20)
)

Origin boarding events: 91
Total downstream stop events: 1252
Downstream events with transfer pathways: 1240
Unique potential interchange stations: 182


,first_line,first_mode,origin_departure_time,stop_name,arrival_time,has_transfer
0,T1,Train,17:30:01,Town Hall Station,17:32:36,True
1,T1,Train,17:30:01,Wynyard Station,17:35:30,True
2,T1,Train,17:30:01,Milsons Point Station,17:39:30,True
3,T1,Train,17:30:01,North Sydney Station,17:41:30,True
4,T1,Train,17:30:01,Waverton Station,17:44:12,True
5,T1,Train,17:30:01,Wollstonecraft Station,17:46:18,True
6,T1,Train,17:30:01,St Leonards Station,17:49:30,True
7,T1,Train,17:30:01,Artarmon Station,17:52:12,True
8,T1,Train,17:30:01,Chatswood Station,17:55:00,True
9,T1,Train,17:30:01,Roseville Station,17:57:48,True


In [96]:
# 1. Create same-platform transfers

same_platform_transfers = (
    transport_stops[
        ["stop_id", "station_id", "transport_mode"]
    ]
    .drop_duplicates("stop_id")
    .rename(columns={
        "stop_id": "from_stop_id",
        "transport_mode": "from_mode"
    })
    .copy()
)

same_platform_transfers["to_stop_id"] = (
    same_platform_transfers["from_stop_id"]
)

same_platform_transfers["to_mode"] = (
    same_platform_transfers["from_mode"]
)

same_platform_transfers["pathway_seconds"] = 0
same_platform_transfers["pathway_minutes"] = 0.0

# Allow 2 minutes for same-platform boarding
same_platform_transfers["transfer_buffer_seconds"] = 120
same_platform_transfers["minimum_transfer_seconds"] = 120
same_platform_transfers["minimum_transfer_minutes"] = 2.0

# 2. Combine pathway and same-platform transfers

all_platform_transfers = pd.concat(
    [
        platform_transfers,
        same_platform_transfers.reindex(
            columns=platform_transfers.columns
        )
    ],
    ignore_index=True
)

# 3. Match downstream arrivals with transfer connections

transfer_candidates = first_leg_stops.merge(
    all_platform_transfers[
        [
            "from_stop_id",
            "to_stop_id",
            "minimum_transfer_seconds",
            "minimum_transfer_minutes"
        ]
    ],
    left_on="stop_id",
    right_on="from_stop_id",
    how="inner"
)

# 4. Calculate earliest feasible boarding time

transfer_candidates["earliest_boarding_seconds"] = (
    transfer_candidates["arrival_seconds"]
    + transfer_candidates["minimum_transfer_seconds"]
)

print("Total transfer candidates:", len(transfer_candidates))

print(
    "Unique first-leg boarding events:",
    transfer_candidates["boarding_id"].nunique()
)

print(
    "Same-platform connections:",
    (
        all_platform_transfers["from_stop_id"]
        == all_platform_transfers["to_stop_id"]
    ).sum()
)

display(
    transfer_candidates[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "to_stop_id",
            "minimum_transfer_minutes"
        ]
    ].head(20)
)

Total transfer candidates: 4667
Unique first-leg boarding events: 91
Same-platform connections: 568


,first_line,origin_departure_time,stop_name,arrival_time,to_stop_id,minimum_transfer_minutes
0,T1,17:30:01,Town Hall Station,17:32:36,2000391,4.6
1,T1,17:30:01,Town Hall Station,17:32:36,2000392,4.6
2,T1,17:30:01,Town Hall Station,17:32:36,2000394,4.5
3,T1,17:30:01,Town Hall Station,17:32:36,2000395,4.5
4,T1,17:30:01,Town Hall Station,17:32:36,2000396,4.5
5,T1,17:30:01,Town Hall Station,17:32:36,2000458,7.5
6,T1,17:30:01,Town Hall Station,17:32:36,2000459,7.5
7,T1,17:30:01,Town Hall Station,17:32:36,2000393,2.0
8,T1,17:30:01,Wynyard Station,17:35:30,2000403,5.0
9,T1,17:30:01,Wynyard Station,17:35:30,2000405,5.0


In [97]:
# 1. Prepare second-leg boarding events
second_leg_departures = daily_timetable[
    [
        "trip_id",
        "stop_id",
        "stop_sequence",
        "departure_seconds",
        "departure_time",
        "route_short_name",
        "transport_mode"
    ]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "second_platform",
    "stop_sequence": "second_boarding_sequence",
    "departure_seconds": "second_departure_seconds",
    "departure_time": "second_departure_time",
    "route_short_name": "second_line",
    "transport_mode": "second_mode"
})

# 2. Join reachable platforms with actual departures
feasible_connections = transfer_candidates.merge(
    second_leg_departures,
    left_on="to_stop_id",
    right_on="second_platform",
    how="inner"
)

# 3. Apply timetable feasibility rules
feasible_connections = feasible_connections[
    (
        feasible_connections["second_departure_seconds"]
        >= feasible_connections["earliest_boarding_seconds"]
    )
    & (
        feasible_connections["second_departure_seconds"]
        <= feasible_connections["arrival_seconds"] + 30 * 60
    )
    & (
        feasible_connections["trip_id"]
        != feasible_connections["second_trip_id"]
    )
].copy()

# 4. Exclude terminating second-leg services
last_sequences = (
    daily_timetable.groupby("trip_id")["stop_sequence"].max()
)

feasible_connections["second_last_sequence"] = (
    feasible_connections["second_trip_id"].map(last_sequences)
)

feasible_connections = feasible_connections[
    feasible_connections["second_boarding_sequence"]
    < feasible_connections["second_last_sequence"]
].copy()

# 5. Calculate connection waiting time
feasible_connections["connection_wait_minutes"] = (
    (
        feasible_connections["second_departure_seconds"]
        - feasible_connections["arrival_seconds"]
    ) / 60
).round(2)

print("Feasible second-leg boarding events:", len(feasible_connections))

print(
    "First-leg boarding events with connections:",
    feasible_connections["boarding_id"].nunique()
)

display(
    feasible_connections[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "second_line",
            "second_mode",
            "second_departure_time",
            "minimum_transfer_minutes",
            "connection_wait_minutes"
        ]
    ]
    .sort_values(
        ["origin_departure_time", "arrival_time", "second_departure_time"]
    )
    .head(20)
)

Feasible second-leg boarding events: 15011
First-leg boarding events with connections: 91


,first_line,origin_departure_time,stop_name,arrival_time,second_line,second_mode,second_departure_time,minimum_transfer_minutes,connection_wait_minutes
1779,T1,17:30:01,Town Hall Station,17:32:36,T9,Train,17:37:36,2.0,5.00
393,T1,17:30:01,Town Hall Station,17:32:36,T1,Train,17:39:00,4.6,6.40
551,T1,17:30:01,Town Hall Station,17:32:36,T4,Train,17:39:00,4.5,6.40
141,T1,17:30:01,Town Hall Station,17:32:36,T3,Train,17:40:00,4.6,7.40
739,T1,17:30:01,Town Hall Station,17:32:36,T4,Train,17:40:00,4.5,7.40
1202,T1,17:30:01,Town Hall Station,17:32:36,L2,Light Rail,17:40:40,7.5,8.07
954,T1,17:30:01,Town Hall Station,17:32:36,T8,Train,17:40:42,4.5,8.10
1473,T1,17:30:01,Town Hall Station,17:32:36,L3,Light Rail,17:41:10,7.5,8.57
401,T1,17:30:01,Town Hall Station,17:32:36,T1,Train,17:42:00,4.6,9.40
98,T1,17:30:01,Town Hall Station,17:32:36,T2,Train,17:43:00,4.6,10.40


In [98]:
destination_name = "Macquarie University Station"

# 1. Find destination station IDs
destination_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == destination_name,
        "station_id"
    ]
)

if not destination_ids:
    raise ValueError(f"Destination not found: {destination_name}")

# 2. Find all timetable arrivals at the destination
destination_arrivals = daily_timetable[
    daily_timetable["station_id"].isin(destination_ids)
][
    [
        "trip_id",
        "stop_sequence",
        "arrival_seconds",
        "arrival_time"
    ]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_sequence": "destination_sequence",
    "arrival_seconds": "destination_arrival_seconds",
    "arrival_time": "destination_arrival_time"
})

# 3. Match second-leg trips to destination arrivals
complete_journeys = feasible_connections.merge(
    destination_arrivals,
    on="second_trip_id",
    how="inner"
)

# 4. Ensure destination is AFTER second-leg boarding
complete_journeys = complete_journeys[
    complete_journeys["destination_sequence"]
    > complete_journeys["second_boarding_sequence"]
].copy()

# 5. Calculate complete journey duration
complete_journeys["total_journey_minutes"] = (
    (
        complete_journeys["destination_arrival_seconds"]
        - complete_journeys["origin_departure_seconds"]
    ) / 60
).round(2)

# 6. Remove exact duplicate journey combinations
complete_journeys = complete_journeys.drop_duplicates(
    subset=[
        "boarding_id",
        "stop_sequence",
        "second_trip_id",
        "second_boarding_sequence",
        "destination_sequence"
    ]
)

# 7. Sort by earliest destination arrival
complete_journeys = complete_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "origin_departure_seconds"
    ]
)

print("Complete one-transfer journeys:", len(complete_journeys))

display(
    complete_journeys[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "minimum_transfer_minutes",
            "total_journey_minutes"
        ]
    ].head(15)
)

Complete one-transfer journeys: 1129


,first_line,origin_departure_time,stop_name,arrival_time,second_line,second_departure_time,destination_arrival_time,minimum_transfer_minutes,total_journey_minutes
20,M1,17:30:02,Gadigal Station,17:31:31,M1,17:36:01,17:58:07,2.0,28.08
34,M1,17:30:02,Martin Place Station,17:33:19,M1,17:37:59,17:58:07,2.0,28.08
47,M1,17:30:02,Barangaroo Station,17:35:41,M1,17:40:11,17:58:07,2.0,28.08
61,M1,17:30:02,Victoria Cross Station,17:38:39,M1,17:43:09,17:58:07,2.0,28.08
74,M1,17:30:02,Crows Nest Station,17:40:46,M1,17:45:16,17:58:07,2.0,28.08
88,M1,17:30:02,Chatswood Station,17:44:19,M1,17:49:04,17:58:07,2.0,28.08
101,M1,17:30:02,North Ryde Station,17:49:56,M1,17:54:26,17:58:07,2.0,28.08
115,M1,17:30:02,Macquarie Park Station,17:52:06,M1,17:56:36,17:58:07,2.0,28.08
21,M1,17:30:02,Gadigal Station,17:31:31,M1,17:40:01,18:02:07,2.0,32.08
35,M1,17:30:02,Martin Place Station,17:33:19,M1,17:41:59,18:02:07,2.0,32.08


In [99]:
# Find destination arrivals on the original first-leg trips
direct_destinations = daily_timetable[
    daily_timetable["station_id"].isin(destination_ids)
][
    [
        "trip_id",
        "stop_sequence",
        "arrival_seconds",
        "arrival_time"
    ]
].rename(columns={
    "stop_sequence": "destination_sequence",
    "arrival_seconds": "destination_arrival_seconds",
    "arrival_time": "destination_arrival_time"
})

# Match origin boarding events to destination arrivals
direct_journeys = boarding_events.merge(
    direct_destinations,
    on="trip_id",
    how="inner"
)

# Destination must occur AFTER boarding
direct_journeys = direct_journeys[
    direct_journeys["destination_sequence"]
    > direct_journeys["origin_sequence"]
].copy()

# Calculate journey duration
direct_journeys["total_journey_minutes"] = (
    (
        direct_journeys["destination_arrival_seconds"]
        - direct_journeys["origin_departure_seconds"]
    ) / 60
).round(2)

# Sort by earliest arrival
direct_journeys = direct_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "origin_departure_seconds"
    ]
)

print("Direct journeys found:", len(direct_journeys))

display(
    direct_journeys[
        [
            "first_line",
            "first_mode",
            "origin_departure_time",
            "destination_arrival_time",
            "total_journey_minutes"
        ]
    ].head(15)
)

Direct journeys found: 8


,first_line,first_mode,origin_departure_time,destination_arrival_time,total_journey_minutes
0,M1,Metro,17:30:02,17:54:07,24.08
2,M1,Metro,17:34:02,17:58:07,24.08
4,M1,Metro,17:38:02,18:02:07,24.08
6,M1,Metro,17:42:02,18:06:07,24.08
8,M1,Metro,17:46:02,18:10:07,24.08
10,M1,Metro,17:50:02,18:14:07,24.08
12,M1,Metro,17:54:02,18:18:07,24.08
14,M1,Metro,17:58:02,18:22:07,24.08


In [100]:
# 1. Prepare direct journeys
direct_results = direct_journeys[
    [
        "first_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

direct_results["journey_type"] = "Direct"
direct_results["transfers"] = 0
direct_results["second_line"] = None
direct_results["transfer_station"] = None

# 2. Prepare one-transfer journeys
transfer_results = complete_journeys[
    [
        "first_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "stop_name",
        "second_line",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

transfer_results = transfer_results.rename(
    columns={"stop_name": "transfer_station"}
)

transfer_results["journey_type"] = "One transfer"
transfer_results["transfers"] = 1

# 3. Combine results
all_journeys = pd.concat(
    [direct_results, transfer_results],
    ignore_index=True
)

# 4. Calculate total time from the requested departure
requested_seconds = time_to_seconds("17:30:00")

all_journeys["elapsed_from_request_min"] = (
    (
        all_journeys["destination_arrival_seconds"]
        - requested_seconds
    ) / 60
).round(2)

# 5. Rank journeys
all_journeys = all_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "transfers",
        "origin_departure_seconds"
    ],
    ascending=[True, True, True]
).reset_index(drop=True)

# 6. Show the best journey
print("Total journey candidates:", len(all_journeys))

display(
    all_journeys[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "total_journey_minutes",
            "elapsed_from_request_min"
        ]
    ].head(15)
)

Total journey candidates: 1137


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,total_journey_minutes,elapsed_from_request_min
0,Direct,M1,None,17:30:02,None,17:54:07,24.08,24.12
1,Direct,M1,None,17:34:02,None,17:58:07,24.08,28.12
2,One transfer,M1,M1,17:30:02,Gadigal Station,17:58:07,28.08,28.12
3,One transfer,M1,M1,17:30:02,Martin Place Station,17:58:07,28.08,28.12
4,One transfer,M1,M1,17:30:02,Barangaroo Station,17:58:07,28.08,28.12
5,One transfer,M1,M1,17:30:02,Victoria Cross Station,17:58:07,28.08,28.12
6,One transfer,M1,M1,17:30:02,Crows Nest Station,17:58:07,28.08,28.12
7,One transfer,M1,M1,17:30:02,Chatswood Station,17:58:07,28.08,28.12
8,One transfer,M1,M1,17:30:02,North Ryde Station,17:58:07,28.08,28.12
9,One transfer,M1,M1,17:30:02,Macquarie Park Station,17:58:07,28.08,28.12


In [101]:
# Separate direct and transfer journeys
direct_options = all_journeys[
    all_journeys["transfers"] == 0
].copy()

transfer_options = all_journeys[
    all_journeys["transfers"] == 1
].copy()

# Compare each transfer journey against direct alternatives
comparisons = transfer_options[
    [
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ]
].merge(
    direct_options[
        [
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    ],
    how="cross",
    suffixes=("_transfer", "_direct")
)

# A direct journey dominates a transfer journey if:
# 1. It departs no earlier
# 2. It arrives no later
comparisons["dominated"] = (
    (
        comparisons["origin_departure_seconds_direct"]
        >= comparisons["origin_departure_seconds_transfer"]
    )
    &
    (
        comparisons["destination_arrival_seconds_direct"]
        <= comparisons["destination_arrival_seconds_transfer"]
    )
)

# Identify dominated transfer journeys
dominated_pairs = comparisons[
    comparisons["dominated"]
][
    [
        "origin_departure_seconds_transfer",
        "destination_arrival_seconds_transfer"
    ]
].drop_duplicates()

# Mark transfer journeys that have a better direct alternative
transfer_options = transfer_options.merge(
    dominated_pairs.assign(dominated=True),
    left_on=[
        "origin_departure_seconds",
        "destination_arrival_seconds"
    ],
    right_on=[
        "origin_departure_seconds_transfer",
        "destination_arrival_seconds_transfer"
    ],
    how="left"
)

transfer_options = transfer_options[
    transfer_options["dominated"].isna()
].drop(
    columns=[
        "dominated",
        "origin_departure_seconds_transfer",
        "destination_arrival_seconds_transfer"
    ]
)

# Combine direct journeys with remaining transfers
recommended_journeys = pd.concat(
    [direct_options, transfer_options],
    ignore_index=True
).sort_values(
    [
        "destination_arrival_seconds",
        "transfers",
        "origin_departure_seconds"
    ],
    ascending=[True, True, False]
).reset_index(drop=True)

print("Original candidates:", len(all_journeys))
print("After removing dominated transfers:", len(recommended_journeys))

display(
    recommended_journeys[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "elapsed_from_request_min"
        ]
    ].head(15)
)

Original candidates: 1137
After removing dominated transfers: 18


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,elapsed_from_request_min
0,Direct,M1,None,17:30:02,None,17:54:07,24.12
1,Direct,M1,None,17:34:02,None,17:58:07,28.12
2,Direct,M1,None,17:38:02,None,18:02:07,32.12
3,Direct,M1,None,17:42:02,None,18:06:07,36.12
4,Direct,M1,None,17:46:02,None,18:10:07,40.12
5,Direct,M1,None,17:50:02,None,18:14:07,44.12
6,Direct,M1,None,17:54:02,None,18:18:07,48.12
7,Direct,M1,None,17:58:02,None,18:22:07,52.12
8,One transfer,T2,M1,17:59:01,Central Station,18:50:07,80.12
9,One transfer,T8,M1,17:59:01,Central Station,18:50:07,80.12


In [102]:
# Identify origin station IDs
origin_station_name = "Central Station"

origin_station_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == origin_station_name,
        "station_id"
    ]
)

# Remove one-transfer journeys returning to the origin
valid_transfer_journeys = complete_journeys[
    ~complete_journeys["station_id"].isin(origin_station_ids)
].copy()

print(
    "Original one-transfer journeys:",
    len(complete_journeys)
)

print(
    "After removing transfers at origin:",
    len(valid_transfer_journeys)
)

print(
    "Removed:",
    len(complete_journeys) - len(valid_transfer_journeys)
)

Original one-transfer journeys: 1129
After removing transfers at origin: 1049
Removed: 80


In [103]:
# Identify journeys where both legs use the same line
valid_transfer_journeys = valid_transfer_journeys.copy()

valid_transfer_journeys["same_line_transfer"] = (
    valid_transfer_journeys["first_line"]
    == valid_transfer_journeys["second_line"]
)

print("Remaining transfer journeys:", len(valid_transfer_journeys))

print(
    "Same-line transfers:",
    valid_transfer_journeys["same_line_transfer"].sum()
)

print(
    "Different-line transfers:",
    (~valid_transfer_journeys["same_line_transfer"]).sum()
)

display(
    valid_transfer_journeys.groupby(
        ["first_line", "second_line"]
    ).size()
    .reset_index(name="journeys")
    .sort_values("journeys", ascending=False)
    .head(15)
)

Remaining transfer journeys: 1049
Same-line transfers: 912
Different-line transfers: 137


,first_line,second_line,journeys
0,M1,M1,912
2,T4,M1,56
4,T9,M1,36
1,T1,M1,24
3,T8,M1,21


In [105]:
# Get the first trip's arrival at the destination
first_trip_destinations = daily_timetable[
    daily_timetable["station_id"].isin(destination_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds"]
].rename(columns={
    "stop_sequence": "first_trip_destination_sequence",
    "arrival_seconds": "first_trip_destination_arrival"
})

# Remove any previous merge columns if the cell was partially run
base_journeys = valid_transfer_journeys.drop(
    columns=[
        "first_trip_destination_sequence",
        "first_trip_destination_arrival"
    ],
    errors="ignore"
)

# Merge with explicitly named columns
transfer_check = base_journeys.merge(
    first_trip_destinations,
    on="trip_id",
    how="left"
)

# Check if first service reaches destination AFTER original boarding
first_trip_reaches_destination = (
    transfer_check["first_trip_destination_sequence"]
    > transfer_check["origin_sequence"]
)

# Check if staying on the first service is faster
staying_onboard_is_better = (
    first_trip_reaches_destination
    &
    (
        transfer_check["first_trip_destination_arrival"]
        <= transfer_check["destination_arrival_seconds"]
    )
)

# Remove unnecessary transfers
useful_transfer_journeys = transfer_check[
    ~staying_onboard_is_better
].copy()

print("Before filtering:", len(transfer_check))
print("Unnecessary transfers:", staying_onboard_is_better.sum())
print("Remaining transfers:", len(useful_transfer_journeys))

display(
    useful_transfer_journeys[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "total_journey_minutes"
        ]
    ].sort_values(
        "total_journey_minutes"
    ).head(15)
)

Before filtering: 1049
Unnecessary transfers: 814
Remaining transfers: 235


,first_line,origin_departure_time,stop_name,second_line,second_departure_time,destination_arrival_time,total_journey_minutes
71,T4,17:37:00,Martin Place Station,M1,17:49:59,18:10:07,33.12
449,T4,17:57:00,Martin Place Station,M1,18:09:59,18:30:07,33.12
122,T4,17:41:00,Martin Place Station,M1,17:53:59,18:14:07,33.12
33,M1,17:32:49,Waterloo Station,M1,17:39:42,18:06:07,33.30
266,M1,17:48:49,Waterloo Station,M1,17:55:42,18:22:07,33.30
70,M1,17:36:49,Waterloo Station,M1,17:43:42,18:10:07,33.30
187,M1,17:44:49,Waterloo Station,M1,17:51:42,18:18:07,33.30
353,M1,17:52:49,Waterloo Station,M1,17:59:42,18:26:07,33.30
448,M1,17:56:49,Waterloo Station,M1,18:03:42,18:30:07,33.30
121,M1,17:40:49,Waterloo Station,M1,17:47:42,18:14:07,33.30


In [106]:
# Prepare the remaining transfer journeys
clean_transfers = useful_transfer_journeys[
    [
        "first_line",
        "second_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "stop_name",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

clean_transfers = clean_transfers.rename(
    columns={"stop_name": "transfer_station"}
)

clean_transfers["journey_type"] = "One transfer"
clean_transfers["transfers"] = 1

# Prepare direct journeys
clean_direct = direct_journeys[
    [
        "first_line",
        "origin_departure_time",
        "origin_departure_seconds",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "total_journey_minutes"
    ]
].copy()

clean_direct["second_line"] = None
clean_direct["transfer_station"] = None
clean_direct["journey_type"] = "Direct"
clean_direct["transfers"] = 0

# Combine
candidates = pd.concat(
    [clean_direct, clean_transfers],
    ignore_index=True
).reset_index(drop=True)

# Pareto dominance filtering
keep = []

for i, journey in candidates.iterrows():

    alternatives = candidates.drop(index=i)

    dominated = (
        (
            alternatives["origin_departure_seconds"]
            >= journey["origin_departure_seconds"]
        )
        &
        (
            alternatives["destination_arrival_seconds"]
            <= journey["destination_arrival_seconds"]
        )
        &
        (
            alternatives["transfers"]
            <= journey["transfers"]
        )
        &
        (
            (alternatives["origin_departure_seconds"]
             > journey["origin_departure_seconds"])
            |
            (alternatives["destination_arrival_seconds"]
             < journey["destination_arrival_seconds"])
            |
            (alternatives["transfers"] < journey["transfers"])
        )
    ).any()

    if not dominated:
        keep.append(i)

best_journeys = candidates.loc[keep].sort_values(
    ["destination_arrival_seconds", "transfers"]
).reset_index(drop=True)

print("Candidates before filtering:", len(candidates))
print("Non-dominated journeys:", len(best_journeys))

display(
    best_journeys[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "total_journey_minutes"
        ]
    ].head(15)
)

Candidates before filtering: 243
Non-dominated journeys: 8


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,total_journey_minutes
0,Direct,M1,None,17:30:02,None,17:54:07,24.08
1,Direct,M1,None,17:34:02,None,17:58:07,24.08
2,Direct,M1,None,17:38:02,None,18:02:07,24.08
3,Direct,M1,None,17:42:02,None,18:06:07,24.08
4,Direct,M1,None,17:46:02,None,18:10:07,24.08
5,Direct,M1,None,17:50:02,None,18:14:07,24.08
6,Direct,M1,None,17:54:02,None,18:18:07,24.08
7,Direct,M1,None,17:58:02,None,18:22:07,24.08


In [107]:
# Test only journeys starting on a Train
train_first_journeys = useful_transfer_journeys[
    useful_transfer_journeys["first_mode"] == "Train"
].copy()

# Rank by earliest arrival
train_first_journeys = train_first_journeys.sort_values(
    [
        "destination_arrival_seconds",
        "origin_departure_seconds"
    ]
)

print(
    "Train → Metro journey candidates:",
    len(train_first_journeys)
)

display(
    train_first_journeys[
        [
            "first_line",
            "origin_departure_time",
            "stop_name",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "minimum_transfer_minutes",
            "total_journey_minutes"
        ]
    ].head(10)
)

Train → Metro journey candidates: 137


,first_line,origin_departure_time,stop_name,arrival_time,second_line,second_departure_time,destination_arrival_time,minimum_transfer_minutes,total_journey_minutes
32,T4,17:31:00,Martin Place Station,17:35:30,M1,17:45:59,18:06:07,6.6,35.12
51,T1,17:30:01,Chatswood Station,17:55:00,M1,18:01:04,18:10:07,5.0,40.10
60,T4,17:31:00,Martin Place Station,17:35:30,M1,17:49:59,18:10:07,6.6,39.12
71,T4,17:37:00,Martin Place Station,17:41:30,M1,17:49:59,18:10:07,6.6,33.12
90,T1,17:30:01,Chatswood Station,17:55:00,M1,18:05:04,18:14:07,5.0,44.10
99,T4,17:31:00,Martin Place Station,17:35:30,M1,17:53:59,18:14:07,6.6,43.12
102,T9,17:34:01,Chatswood Station,17:59:00,M1,18:05:04,18:14:07,5.0,40.10
112,T4,17:37:00,Martin Place Station,17:41:30,M1,17:53:59,18:14:07,6.6,37.12
122,T4,17:41:00,Martin Place Station,17:45:30,M1,17:53:59,18:14:07,6.6,33.12
141,T9,17:36:01,Epping Station,18:05:00,M1,18:12:13,18:15:24,5.5,39.38


In [108]:
def prepare_daily_timetable(travel_date):
    """
    Return scheduled Train, Metro and Light Rail stop events
    operating on the requested date.
    """

    active_services = get_active_services(travel_date)

    daily = timetable[
        timetable["service_id"].isin(active_services)
    ].copy()

    daily = daily.sort_values(
        ["trip_id", "stop_sequence"]
    ).reset_index(drop=True)

    return daily


# Test with our existing date
test_timetable = prepare_daily_timetable("2026-10-28")

print("Scheduled stop events:", len(test_timetable))
print("Unique trips:", test_timetable["trip_id"].nunique())

display(
    test_timetable.groupby("transport_mode")
    .agg(
        trips=("trip_id", "nunique"),
        stop_events=("stop_id", "count")
    )
)

Scheduled stop events: 59541
Unique trips: 3705


,trips,stop_events
transport_mode,,
Light Rail,563,7997
Metro,434,9114
Train,2708,42430


In [109]:
def get_station_departures(
    daily,
    origin,
    departure_time,
    window_minutes=30
):
    """
    Find scheduled departures from an origin station
    within the requested time window.

    Excludes trips that terminate at the origin.
    """

    # 1. Find the station ID
    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin,
            "station_id"
        ]
    )

    if not origin_ids:
        raise ValueError(f"Station not found: {origin}")

    # 2. Convert departure time to seconds
    time_string = (
        departure_time
        if len(departure_time) == 8
        else departure_time + ":00"
    )

    requested_seconds = time_to_seconds(time_string)

    # 3. Find departures within the time window
    departures = daily[
        daily["station_id"].isin(origin_ids)
        &
        daily["departure_seconds"].between(
            requested_seconds,
            requested_seconds + window_minutes * 60
        )
    ].copy()

    # 4. Identify the final stop of each trip
    last_sequences = daily.groupby(
        "trip_id"
    )["stop_sequence"].max()

    departures["last_stop_sequence"] = (
        departures["trip_id"].map(last_sequences)
    )

    # 5. Exclude terminating services
    departures = departures[
        departures["stop_sequence"]
        < departures["last_stop_sequence"]
    ].copy()

    # 6. Sort departures
    return departures.sort_values(
        "departure_seconds"
    ).reset_index(drop=True)


# Test our function
test_departures = get_station_departures(
    daily=test_timetable,
    origin="Central Station",
    departure_time="17:30",
    window_minutes=30
)

print("Valid boarding opportunities:", len(test_departures))

display(
    test_departures[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_id"
        ]
    ].head(15)
)

Valid boarding opportunities: 91


,route_short_name,transport_mode,departure_time,stop_id
0,T1,Train,17:30:01,2000336
1,M1,Metro,17:30:02,2000466
2,T4,Train,17:31:00,2000344
3,L2,Light Rail,17:31:35,2000447
4,T4,Train,17:32:00,2000345
5,T8,Train,17:32:00,2000343
6,T2,Train,17:32:00,2000339
7,T2,Train,17:32:01,2000341
8,T8,Train,17:32:01,2000337
9,M1,Metro,17:32:49,2000467


In [110]:
def find_direct_journeys(
    daily,
    origin,
    destination,
    departure_time,
    window_minutes=30
):
    """
    Find direct scheduled journeys between two stations.
    """

    # 1. Get valid origin departures
    departures = get_station_departures(
        daily=daily,
        origin=origin,
        departure_time=departure_time,
        window_minutes=window_minutes
    )

    # 2. Find destination station IDs
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # 3. Find destination arrivals
    arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_seconds",
            "arrival_time"
        ]
    ].rename(columns={
        "stop_sequence": "destination_sequence",
        "arrival_seconds": "destination_arrival_seconds",
        "arrival_time": "destination_arrival_time"
    })

    # 4. Match origin departures to destination arrivals
    journeys = departures.merge(
        arrivals,
        on="trip_id",
        how="inner"
    )

    # 5. Destination must be after origin
    journeys = journeys[
        journeys["destination_sequence"]
        > journeys["stop_sequence"]
    ].copy()

    # 6. Calculate journey duration
    journeys["journey_duration_min"] = (
        (
            journeys["destination_arrival_seconds"]
            - journeys["departure_seconds"]
        ) / 60
    ).round(2)

    # 7. Add useful output fields
    journeys["journey_type"] = "Direct"
    journeys["transfers"] = 0

    # 8. Remove duplicate boarding/arrival combinations
    journeys = journeys.drop_duplicates(
        subset=[
            "trip_id",
            "stop_id",
            "stop_sequence",
            "destination_sequence"
        ]
    )

    # 9. Rank by earliest arrival
    journeys = journeys.sort_values(
        [
            "destination_arrival_seconds",
            "departure_seconds"
        ]
    ).reset_index(drop=True)

    return journeys


# Test: Central → Macquarie University
direct_test = find_direct_journeys(
    daily=test_timetable,
    origin="Central Station",
    destination="Macquarie University Station",
    departure_time="17:30"
)

print("Direct journeys found:", len(direct_test))

display(
    direct_test[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(10)
)

Direct journeys found: 8


,route_short_name,transport_mode,departure_time,destination_arrival_time,journey_duration_min
0,M1,Metro,17:30:02,17:54:07,24.08
1,M1,Metro,17:34:02,17:58:07,24.08
2,M1,Metro,17:38:02,18:02:07,24.08
3,M1,Metro,17:42:02,18:06:07,24.08
4,M1,Metro,17:46:02,18:10:07,24.08
5,M1,Metro,17:50:02,18:14:07,24.08
6,M1,Metro,17:54:02,18:18:07,24.08
7,M1,Metro,17:58:02,18:22:07,24.08


In [111]:
def find_one_transfer_journeys(
    daily,
    origin,
    destination,
    departure_time,
    window_minutes=30,
    connection_window_minutes=30
):
    """Find scheduled journeys requiring one platform transfer."""

    departures = get_station_departures(
        daily, origin, departure_time, window_minutes
    ).copy()

    origin_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == origin, "station_id"
        ]
    )
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination, "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # Identify each first-leg boarding event
    departures["boarding_id"] = range(len(departures))

    first_boardings = departures[
        [
            "boarding_id", "trip_id", "stop_id",
            "stop_sequence", "departure_seconds",
            "departure_time", "route_short_name",
            "transport_mode"
        ]
    ].rename(columns={
        "stop_id": "origin_platform",
        "stop_sequence": "origin_sequence",
        "departure_seconds": "origin_departure_seconds",
        "departure_time": "origin_departure_time",
        "route_short_name": "first_line",
        "transport_mode": "first_mode"
    })

    # Find downstream stops on first-leg services
    first_stops = first_boardings.merge(
        daily[
            [
                "trip_id", "stop_id", "station_id",
                "stop_sequence", "arrival_seconds",
                "arrival_time"
            ]
        ],
        on="trip_id",
        how="inner"
    )

    first_stops = first_stops[
        (first_stops["stop_sequence"] > first_stops["origin_sequence"])
        & (~first_stops["station_id"].isin(origin_ids))
        & (~first_stops["station_id"].isin(destination_ids))
    ].copy()

    # Match first-leg arrival platforms to reachable platforms
    connections = first_stops.merge(
        all_platform_transfers[
            ["from_stop_id", "to_stop_id", "minimum_transfer_seconds"]
        ],
        left_on="stop_id",
        right_on="from_stop_id",
        how="inner"
    )

    connections["earliest_boarding_seconds"] = (
        connections["arrival_seconds"]
        + connections["minimum_transfer_seconds"]
    )

    # Find scheduled departures from connecting platforms
    second_departures = daily[
        [
            "trip_id", "stop_id", "stop_sequence",
            "departure_seconds", "departure_time",
            "route_short_name", "transport_mode"
        ]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "stop_id": "second_platform",
        "stop_sequence": "second_boarding_sequence",
        "departure_seconds": "second_departure_seconds",
        "departure_time": "second_departure_time",
        "route_short_name": "second_line",
        "transport_mode": "second_mode"
    })

    connections = connections.merge(
        second_departures,
        left_on="to_stop_id",
        right_on="second_platform",
        how="inner"
    )

    connections = connections[
        (connections["second_trip_id"] != connections["trip_id"])
        & (
            connections["second_departure_seconds"]
            >= connections["earliest_boarding_seconds"]
        )
        & (
            connections["second_departure_seconds"]
            <= connections["arrival_seconds"]
               + connection_window_minutes * 60
        )
    ].copy()

    # Find destination arrivals on second-leg services
    destination_arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        ["trip_id", "stop_sequence", "arrival_seconds", "arrival_time"]
    ].rename(columns={
        "trip_id": "second_trip_id",
        "stop_sequence": "destination_sequence",
        "arrival_seconds": "destination_arrival_seconds",
        "arrival_time": "destination_arrival_time"
    })

    journeys = connections.merge(
        destination_arrivals,
        on="second_trip_id",
        how="inner"
    )

    journeys = journeys[
        journeys["destination_sequence"]
        > journeys["second_boarding_sequence"]
    ].copy()

    journeys["journey_duration_min"] = (
        (
            journeys["destination_arrival_seconds"]
            - journeys["origin_departure_seconds"]
        ) / 60
    ).round(2)

    journeys["journey_type"] = "One transfer"
    journeys["transfers"] = 1

    journeys = journeys.merge(
        station_lookup.rename(
            columns={"stop_name": "transfer_station"}
        ),
        on="station_id",
        how="left"
    )

    return journeys.sort_values(
        ["destination_arrival_seconds", "origin_departure_seconds"]
    ).reset_index(drop=True)


# Test the function
transfer_test = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Central Station",
    destination="Macquarie University Station",
    departure_time="17:30"
)

print("One-transfer candidates:", len(transfer_test))

display(
    transfer_test[
        [
            "first_line",
            "origin_departure_time",
            "transfer_station",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(10)
)

One-transfer candidates: 1049


,first_line,origin_departure_time,transfer_station,second_line,second_departure_time,destination_arrival_time,journey_duration_min
0,M1,17:30:02,Gadigal Station,M1,17:36:01,17:58:07,28.08
1,M1,17:30:02,Martin Place Station,M1,17:37:59,17:58:07,28.08
2,M1,17:30:02,Barangaroo Station,M1,17:40:11,17:58:07,28.08
3,M1,17:30:02,Victoria Cross Station,M1,17:43:09,17:58:07,28.08
4,M1,17:30:02,Crows Nest Station,M1,17:45:16,17:58:07,28.08
5,M1,17:30:02,Chatswood Station,M1,17:49:04,17:58:07,28.08
6,M1,17:30:02,North Ryde Station,M1,17:54:26,17:58:07,28.08
7,M1,17:30:02,Macquarie Park Station,M1,17:56:36,17:58:07,28.08
8,M1,17:30:02,Gadigal Station,M1,17:40:01,18:02:07,32.08
9,M1,17:30:02,Martin Place Station,M1,17:41:59,18:02:07,32.08


In [112]:
def find_multimodal_journeys(
    origin,
    destination,
    travel_date,
    departure_time,
    window_minutes=30,
    max_transfers=1,
    limit=10
):
    """
    Find scheduled multimodal journeys with up to one transfer.

    Supports Train, Metro and Light Rail.
    """

    if max_transfers not in (0, 1):
        raise ValueError("Only 0 or 1 transfer is currently supported.")

    if origin == destination:
        raise ValueError("Origin and destination must be different.")

    daily = prepare_daily_timetable(travel_date)

    requested_time = (
        departure_time if len(departure_time) == 8
        else departure_time + ":00"
    )
    requested_seconds = time_to_seconds(requested_time)

    # 1. Find direct journeys
    direct = find_direct_journeys(
        daily, origin, destination,
        departure_time, window_minutes
    ).copy()

    direct["first_line"] = direct["route_short_name"]
    direct["second_line"] = None
    direct["transfer_station"] = None
    direct["origin_departure_time"] = direct["departure_time"]
    direct["origin_departure_seconds"] = direct["departure_seconds"]

    columns = [
        "journey_type", "first_line", "second_line",
        "origin_departure_time", "origin_departure_seconds",
        "transfer_station", "destination_arrival_time",
        "destination_arrival_seconds", "journey_duration_min",
        "transfers"
    ]

    results = [direct[columns]]

    if max_transfers == 1:

        # 2. Find feasible one-transfer journeys
        transfer = find_one_transfer_journeys(
            daily, origin, destination,
            departure_time, window_minutes
        ).copy()

        # 3. Check if staying on first service is better
        destination_ids = set(
            station_lookup.loc[
                station_lookup["stop_name"] == destination,
                "station_id"
            ]
        )

        first_trip_arrivals = daily[
            daily["station_id"].isin(destination_ids)
        ][
            ["trip_id", "stop_sequence", "arrival_seconds"]
        ].rename(columns={
            "stop_sequence": "first_destination_sequence",
            "arrival_seconds": "first_destination_arrival"
        })

        transfer = transfer.merge(
            first_trip_arrivals,
            on="trip_id",
            how="left"
        )

        unnecessary = (
            (
                transfer["first_destination_sequence"]
                > transfer["origin_sequence"]
            )
            &
            (
                transfer["first_destination_arrival"]
                <= transfer["destination_arrival_seconds"]
            )
        )

        transfer = transfer.loc[~unnecessary].copy()

        results.append(transfer[columns])

    # 4. Combine direct and transfer journeys
    candidates = pd.concat(
        results,
        ignore_index=True
    ).reset_index(drop=True)

    # 5. Remove dominated journeys
    keep = []

    for i, journey in candidates.iterrows():

        alternatives = candidates.drop(index=i)

        dominated = (
            (
                alternatives["origin_departure_seconds"]
                >= journey["origin_departure_seconds"]
            )
            &
            (
                alternatives["destination_arrival_seconds"]
                <= journey["destination_arrival_seconds"]
            )
            &
            (
                alternatives["transfers"]
                <= journey["transfers"]
            )
            &
            (
                (
                    alternatives["origin_departure_seconds"]
                    > journey["origin_departure_seconds"]
                )
                |
                (
                    alternatives["destination_arrival_seconds"]
                    < journey["destination_arrival_seconds"]
                )
                |
                (
                    alternatives["transfers"] < journey["transfers"]
                )
            )
        ).any()

        if not dominated:
            keep.append(i)

    journeys = candidates.loc[keep].copy()

    # 6. Calculate elapsed time from requested departure
    journeys["elapsed_from_request_min"] = (
        (
            journeys["destination_arrival_seconds"]
            - requested_seconds
        ) / 60
    ).round(2)

    # 7. Rank and limit results
    journeys = journeys.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "origin_departure_seconds"
        ],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    return journeys.head(limit)


# Test our complete journey planner
journey_results = find_multimodal_journeys(
    origin="Central Station",
    destination="Macquarie University Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    limit=10
)

display(journey_results)

,journey_type,first_line,second_line,origin_departure_time,origin_departure_seconds,transfer_station,destination_arrival_time,destination_arrival_seconds,journey_duration_min,transfers,elapsed_from_request_min
0,Direct,M1,None,17:30:02,63002,None,17:54:07,64447,24.08,0,24.12
1,Direct,M1,None,17:34:02,63242,None,17:58:07,64687,24.08,0,28.12
2,Direct,M1,None,17:38:02,63482,None,18:02:07,64927,24.08,0,32.12
3,Direct,M1,None,17:42:02,63722,None,18:06:07,65167,24.08,0,36.12
4,Direct,M1,None,17:46:02,63962,None,18:10:07,65407,24.08,0,40.12
5,Direct,M1,None,17:50:02,64202,None,18:14:07,65647,24.08,0,44.12
6,Direct,M1,None,17:54:02,64442,None,18:18:07,65887,24.08,0,48.12
7,Direct,M1,None,17:58:02,64682,None,18:22:07,66127,24.08,0,52.12


In [113]:
# Test a journey requiring a transfer
journey_test_2 = find_multimodal_journeys(
    origin="Macquarie University Station",
    destination="Parramatta Station",
    travel_date="2026-10-28",
    departure_time="17:30",
    window_minutes=30,
    max_transfers=1,
    limit=10
)

print("Journeys found:", len(journey_test_2))

display(
    journey_test_2[
        [
            "journey_type",
            "first_line",
            "second_line",
            "origin_departure_time",
            "transfer_station",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers",
            "elapsed_from_request_min"
        ]
    ]
)

Journeys found: 7


,journey_type,first_line,second_line,origin_departure_time,transfer_station,destination_arrival_time,journey_duration_min,transfers,elapsed_from_request_min
0,One transfer,M1,T1,17:31:54,Central Station,18:29:00,57.1,1,59.0
1,One transfer,M1,T1,17:35:54,Central Station,18:32:00,56.1,1,62.0
2,One transfer,M1,T1,17:39:54,Central Station,18:38:00,58.1,1,68.0
3,One transfer,M1,T1,17:43:54,Central Station,18:41:00,57.1,1,71.0
4,One transfer,M1,T1,17:47:54,Central Station,18:44:00,56.1,1,74.0
5,One transfer,M1,T1,17:55:54,Central Station,18:56:00,60.1,1,86.0
6,One transfer,M1,T1,17:59:54,Central Station,18:59:00,59.1,1,89.0


In [114]:
# Get all feasible one-transfer candidates
epping_test = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Macquarie University Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=30
)

print("Total transfer candidates:", len(epping_test))

# Count candidates by interchange station
display(
    epping_test.groupby("transfer_station")
    .size()
    .reset_index(name="journeys")
    .sort_values("journeys", ascending=False)
    .head(20)
)

# Specifically inspect Epping
epping_connections = epping_test[
    epping_test["transfer_station"] == "Epping Station"
].copy()

print("Journeys via Epping:", len(epping_connections))

display(
    epping_connections[
        [
            "first_line",
            "origin_departure_time",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "minimum_transfer_seconds",
            "journey_duration_min"
        ]
    ].head(15)
)

Total transfer candidates: 57


,transfer_station,journeys
0,Central Station,57


Journeys via Epping: 0


,first_line,origin_departure_time,arrival_time,second_line,second_departure_time,destination_arrival_time,minimum_transfer_seconds,journey_duration_min


In [115]:
epping_direct = find_direct_journeys(
    daily=test_timetable,
    origin="Epping Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=90
)

print("Direct Epping → Parramatta journeys:", len(epping_direct))

display(
    epping_direct[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)

Direct Epping → Parramatta journeys: 0


,route_short_name,transport_mode,departure_time,destination_arrival_time,journey_duration_min


In [116]:
# Find train departures from Epping
epping_departures = get_station_departures(
    daily=test_timetable,
    origin="Epping Station",
    departure_time="17:30",
    window_minutes=90
)

epping_trains = epping_departures[
    epping_departures["transport_mode"] == "Train"
].copy()

# Find downstream stations on those trains
reachable_stops = epping_trains[
    ["trip_id", "stop_sequence"]
].merge(
    test_timetable[
        ["trip_id", "station_id", "stop_sequence", "arrival_time"]
    ],
    on="trip_id",
    how="inner",
    suffixes=("_boarding", "_destination")
)

reachable_stops = reachable_stops[
    reachable_stops["stop_sequence_destination"]
    > reachable_stops["stop_sequence_boarding"]
]

# Attach station names
reachable_stops = reachable_stops.merge(
    station_lookup,
    on="station_id",
    how="left"
)

print("Train departures from Epping:", len(epping_trains))
print(
    "Unique downstream stations:",
    reachable_stops["station_id"].nunique()
)

display(
    reachable_stops[
        ["stop_name", "arrival_time"]
    ].drop_duplicates()
    .sort_values("stop_name")
    .head(40)
)

Train departures from Epping: 15
Unique downstream stations: 17


,stop_name,arrival_time
1,Beecroft Station,17:40:42
24,Beecroft Station,17:55:42
47,Beecroft Station,18:10:42
70,Beecroft Station,18:25:42
87,Beecroft Station,18:40:42
104,Beecroft Station,18:55:42
10,Burwood Station,17:56:00
20,Burwood Station,18:06:18
33,Burwood Station,18:11:00
43,Burwood Station,18:21:18


In [117]:
burwood_direct = find_direct_journeys(
    daily=test_timetable,
    origin="Burwood Station",
    destination="Parramatta Station",
    departure_time="17:30",
    window_minutes=120
)

print(
    "Direct Burwood → Parramatta journeys:",
    len(burwood_direct)
)

display(
    burwood_direct[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)

Direct Burwood → Parramatta journeys: 8


,route_short_name,transport_mode,departure_time,destination_arrival_time,journey_duration_min
0,T2,Train,17:43:18,18:06:30,23.2
1,T2,Train,17:58:18,18:21:30,23.2
2,T2,Train,18:13:18,18:36:30,23.2
3,T2,Train,18:28:18,18:51:30,23.2
4,T2,Train,18:43:18,19:06:30,23.2
5,T2,Train,18:57:12,19:20:30,23.3
6,T2,Train,19:12:12,19:35:30,23.3
7,T2,Train,19:27:12,19:50:30,23.3


In [118]:
# Find Metro → Train journeys via Epping to Burwood
burwood_connections = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Macquarie University Station",
    destination="Burwood Station",
    departure_time="17:30",
    window_minutes=30
)

# Keep only journeys transferring at Epping
via_epping = burwood_connections[
    burwood_connections["transfer_station"] == "Epping Station"
].copy()

# Sort by arrival at Burwood
via_epping = via_epping.sort_values(
    "destination_arrival_seconds"
)

print("Macquarie University → Epping → Burwood:", len(via_epping))

display(
    via_epping[
        [
            "first_line",
            "origin_departure_time",
            "transfer_station",
            "arrival_time",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].head(15)
)

Macquarie University → Epping → Burwood: 23


,first_line,origin_departure_time,transfer_station,arrival_time,second_line,second_departure_time,destination_arrival_time,journey_duration_min
0,M1,17:30:37,Epping Station,17:33:51,T9,17:42:00,18:06:18,35.68
1,M1,17:30:37,Epping Station,17:33:51,T9,17:53:01,18:11:00,40.38
2,M1,17:34:37,Epping Station,17:37:51,T9,17:53:01,18:11:00,36.38
3,M1,17:38:37,Epping Station,17:41:51,T9,17:53:01,18:11:00,32.38
4,M1,17:42:37,Epping Station,17:45:51,T9,17:53:01,18:11:00,28.38
7,M1,17:30:37,Epping Station,17:33:51,T9,17:57:00,18:21:18,50.68
8,M1,17:34:37,Epping Station,17:37:51,T9,17:57:00,18:21:18,46.68
9,M1,17:38:37,Epping Station,17:41:51,T9,17:57:00,18:21:18,42.68
10,M1,17:42:37,Epping Station,17:45:51,T9,17:57:00,18:21:18,38.68
11,M1,17:46:37,Epping Station,17:49:51,T9,17:57:00,18:21:18,34.68


In [120]:
print("Columns in via_epping:")
print(via_epping.columns.tolist())

print("\nColumns in burwood_connections:")
print(burwood_connections.columns.tolist())

print("\nTransfer-related columns:")
print([
    col for col in burwood_connections.columns
    if "stop_id" in col
    or "transfer" in col
    or "platform" in col
])

Columns in via_epping:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id', 'to_stop_id', 'minimum_transfer_seconds', 'earliest_boarding_seconds', 'second_trip_id', 'second_platform', 'second_boarding_sequence', 'second_departure_seconds', 'second_departure_time', 'second_line', 'second_mode', 'destination_sequence', 'destination_arrival_seconds', 'destination_arrival_time', 'journey_duration_min', 'journey_type', 'transfers', 'transfer_station']

Columns in burwood_connections:
['boarding_id', 'trip_id', 'origin_platform', 'origin_sequence', 'origin_departure_seconds', 'origin_departure_time', 'first_line', 'first_mode', 'stop_id', 'station_id', 'stop_sequence', 'arrival_seconds', 'arrival_time', 'from_stop_id_x', 'to_stop_id_x', 'minimum_transfer_seconds_x', 'earliest_boarding_seconds', 'second_trip_i

In [121]:
# Rename columns to make both transfers explicit
burwood_connections = burwood_connections.rename(columns={
    "from_stop_id_x": "epping_from_platform",
    "to_stop_id_x": "epping_to_platform",
    "minimum_transfer_seconds_x": "epping_transfer_seconds",
    "from_stop_id_y": "burwood_from_platform",
    "to_stop_id_y": "burwood_to_platform",
    "minimum_transfer_seconds_y": "burwood_transfer_seconds"
})

# Match T2 departures at the correct Burwood platform
burwood_connections = burwood_connections.merge(
    burwood_t2_departures,
    left_on="burwood_to_platform",
    right_on="third_platform",
    how="inner"
)

# Apply minimum transfer time at Burwood
burwood_connections = burwood_connections[
    burwood_connections["third_departure_seconds"]
    >= (
        burwood_connections["burwood_arrival_seconds"]
        + burwood_connections["burwood_transfer_seconds"]
    )
].copy()

# Match the third train to its arrival at Parramatta
complete_two_transfer = burwood_connections.merge(
    parramatta_arrivals,
    on="third_trip_id",
    how="inner"
)

# Ensure Parramatta comes after Burwood on the third train
complete_two_transfer = complete_two_transfer[
    complete_two_transfer["final_sequence"]
    > complete_two_transfer["third_boarding_sequence"]
].copy()

# Calculate total journey duration
complete_two_transfer["total_minutes"] = (
    complete_two_transfer["final_arrival_seconds"]
    - complete_two_transfer["origin_departure_seconds"]
) / 60

print("Feasible two-transfer journeys:", len(complete_two_transfer))

display(
    complete_two_transfer[
        [
            "origin_departure_time",
            "second_departure_time",
            "destination_arrival_time",
            "third_departure_time",
            "final_arrival_time",
            "burwood_transfer_seconds",
            "total_minutes"
        ]
    ].sort_values(
        "final_arrival_time"
    ).head(15)
)

KeyError: 'third_trip_id'

In [122]:
# Identify Parramatta Station
parramatta_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Parramatta Station",
        "station_id"
    ]
)

# Get all scheduled arrivals at Parramatta
parramatta_arrivals = test_timetable[
    test_timetable["station_id"].isin(parramatta_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds", "arrival_time"]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_sequence": "final_sequence",
    "arrival_seconds": "final_arrival_seconds",
    "arrival_time": "final_arrival_time"
})

print("Parramatta arrival records:", len(parramatta_arrivals))
print("Columns:", parramatta_arrivals.columns.tolist())

display(parramatta_arrivals.head())

Parramatta arrival records: 484
Columns: ['third_trip_id', 'final_sequence', 'final_arrival_seconds', 'final_arrival_time']


,third_trip_id,final_sequence,final_arrival_seconds,final_arrival_time
9114,1--B.1324.165.116.B.8.92000948,1,20611,05:43:31
9195,1--E.1324.165.116.B.8.92003215,26,33630,09:20:30
9196,1--F.1324.165.116.B.8.92003214,1,34111,09:28:31
9277,1--J.1324.165.116.B.8.92000805,26,47130,13:05:30
9278,1--K.1324.165.116.B.8.92001138,1,47611,13:13:31


In [123]:
# Match Burwood T2 departures to Parramatta arrivals
complete_two_transfer = burwood_connections.merge(
    parramatta_arrivals,
    on="third_trip_id",
    how="inner"
)

# Ensure Parramatta is AFTER Burwood on the same train
complete_two_transfer = complete_two_transfer[
    complete_two_transfer["final_sequence"]
    > complete_two_transfer["third_boarding_sequence"]
].copy()

# Calculate total journey duration
complete_two_transfer["total_minutes"] = (
    complete_two_transfer["final_arrival_seconds"]
    - complete_two_transfer["origin_departure_seconds"]
) / 60

# Show the results
print("Feasible two-transfer journeys:", len(complete_two_transfer))

display(
    complete_two_transfer[
        [
            "origin_departure_time",
            "second_departure_time",
            "destination_arrival_time",
            "third_departure_time",
            "final_arrival_time",
            "burwood_transfer_seconds",
            "total_minutes"
        ]
    ]
    .sort_values("final_arrival_time")
    .head(15)
)

Feasible two-transfer journeys: 283


,origin_departure_time,second_departure_time,destination_arrival_time,third_departure_time,final_arrival_time,burwood_transfer_seconds,total_minutes
21,17:30:37,17:42:00,18:06:18,18:13:18,18:36:30,360.0,65.883333
26,17:30:37,17:42:00,18:06:18,18:28:18,18:51:30,360.0,80.883333
53,17:30:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,80.883333
80,17:34:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,76.883333
107,17:38:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,72.883333
134,17:42:37,17:53:01,18:11:00,18:28:18,18:51:30,390.0,68.883333
160,17:30:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,80.883333
186,17:34:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,76.883333
212,17:38:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,72.883333
238,17:42:37,17:57:00,18:21:18,18:28:18,18:51:30,360.0,68.883333


In [125]:
# Rebuild Central options
central_options = journey_test_2[
    [
        "first_line",
        "second_line",
        "transfer_station",
        "origin_departure_time",
        "destination_arrival_time",
        "destination_arrival_seconds",
        "journey_duration_min",
        "transfers"
    ]
].copy()

central_options["route"] = (
    central_options["first_line"].astype(str)
    + " → "
    + central_options["second_line"].astype(str)
)

central_options["transfer_path"] = central_options["transfer_station"]


# Rebuild two-transfer options without renaming existing columns
two_transfer_options = complete_two_transfer[
    [
        "origin_departure_time",
        "final_arrival_time",
        "final_arrival_seconds",
        "total_minutes"
    ]
].copy()

two_transfer_options["route"] = "M1 → T9 → T2"
two_transfer_options["transfer_path"] = "Epping → Burwood"
two_transfer_options["transfers"] = 2

two_transfer_options["destination_arrival_time"] = (
    two_transfer_options["final_arrival_time"]
)

two_transfer_options["destination_arrival_seconds"] = (
    two_transfer_options["final_arrival_seconds"]
)

two_transfer_options["journey_duration_min"] = (
    two_transfer_options["total_minutes"]
)


# Select exactly the same columns from both DataFrames
comparison_columns = [
    "route",
    "transfer_path",
    "origin_departure_time",
    "destination_arrival_time",
    "destination_arrival_seconds",
    "journey_duration_min",
    "transfers"
]

comparison = pd.concat(
    [
        central_options[comparison_columns],
        two_transfer_options[comparison_columns]
    ],
    ignore_index=True
)

comparison = comparison.sort_values(
    ["destination_arrival_seconds", "transfers"]
)

print("Total journey combinations:", len(comparison))

display(
    comparison[
        [
            "route",
            "transfer_path",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers"
        ]
    ].head(15)
)

Total journey combinations: 290


,route,transfer_path,origin_departure_time,destination_arrival_time,journey_duration_min,transfers
0,M1 → T1,Central Station,17:31:54,18:29:00,57.100000,1
1,M1 → T1,Central Station,17:35:54,18:32:00,56.100000,1
14,M1 → T9 → T2,Epping → Burwood,17:30:37,18:36:30,65.883333,2
2,M1 → T1,Central Station,17:39:54,18:38:00,58.100000,1
3,M1 → T1,Central Station,17:43:54,18:41:00,57.100000,1
4,M1 → T1,Central Station,17:47:54,18:44:00,56.100000,1
19,M1 → T9 → T2,Epping → Burwood,17:30:37,18:51:30,80.883333,2
32,M1 → T9 → T2,Epping → Burwood,17:30:37,18:51:30,80.883333,2
45,M1 → T9 → T2,Epping → Burwood,17:34:37,18:51:30,76.883333,2
58,M1 → T9 → T2,Epping → Burwood,17:38:37,18:51:30,72.883333,2


In [126]:
# Start with the combined comparison table
ranked = comparison.copy()

# Convert departure times into seconds
ranked["origin_departure_seconds"] = (
    ranked["origin_departure_time"].apply(time_to_seconds)
)

# Remove exact duplicates in the displayed journey information
ranked = ranked.drop_duplicates(
    subset=[
        "route",
        "transfer_path",
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "transfers"
    ]
).reset_index(drop=True)

# Identify Pareto-dominated journeys
departures = ranked["origin_departure_seconds"].to_numpy()
arrivals = ranked["destination_arrival_seconds"].to_numpy()
transfers = ranked["transfers"].to_numpy()

keep = np.ones(len(ranked), dtype=bool)

for i in range(len(ranked)):
    better_or_equal = (
        (departures >= departures[i])
        & (arrivals <= arrivals[i])
        & (transfers <= transfers[i])
    )

    strictly_better = (
        (departures > departures[i])
        | (arrivals < arrivals[i])
        | (transfers < transfers[i])
    )

    better_or_equal[i] = False

    if np.any(better_or_equal & strictly_better):
        keep[i] = False

ranked = ranked.loc[keep].sort_values(
    ["destination_arrival_seconds", "transfers"]
)

print("Original combinations:", len(comparison))
print("Non-dominated journeys:", len(ranked))

display(
    ranked[
        [
            "route",
            "transfer_path",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers"
        ]
    ].head(15)
)

Original combinations: 290
Non-dominated journeys: 7


,route,transfer_path,origin_departure_time,destination_arrival_time,journey_duration_min,transfers
0,M1 → T1,Central Station,17:31:54,18:29:00,57.1,1
1,M1 → T1,Central Station,17:35:54,18:32:00,56.1,1
3,M1 → T1,Central Station,17:39:54,18:38:00,58.1,1
4,M1 → T1,Central Station,17:43:54,18:41:00,57.1,1
5,M1 → T1,Central Station,17:47:54,18:44:00,56.1,1
11,M1 → T1,Central Station,17:55:54,18:56:00,60.1,1
12,M1 → T1,Central Station,17:59:54,18:59:00,59.1,1


In [127]:
def get_valid_connections(
    daily,
    arrival_platform,
    arrival_seconds,
    connection_window_minutes=30
):
    """
    Find valid onward boarding events from an arrival platform.

    Applies platform transfer times and a maximum waiting window.
    """

    # Find reachable boarding platforms
    transfers = all_platform_transfers[
        all_platform_transfers["from_stop_id"] == arrival_platform
    ][
        ["to_stop_id", "minimum_transfer_seconds"]
    ].copy()

    if transfers.empty:
        return pd.DataFrame()

    # Get scheduled departures at reachable platforms
    departures = daily.merge(
        transfers,
        left_on="stop_id",
        right_on="to_stop_id",
        how="inner"
    )

    # Earliest departure after completing the transfer
    departures["earliest_departure"] = (
        arrival_seconds
        + departures["minimum_transfer_seconds"]
    )

    # Keep only feasible departures
    departures = departures[
        (
            departures["departure_seconds"]
            >= departures["earliest_departure"]
        )
        & (
            departures["departure_seconds"]
            <= arrival_seconds + connection_window_minutes * 60
        )
    ].copy()

    return departures.sort_values("departure_seconds")

In [119]:
# Find T2 boarding events at Burwood
burwood_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Burwood Station",
        "station_id"
    ]
)

burwood_t2_departures = test_timetable[
    (test_timetable["station_id"].isin(burwood_ids))
    & (test_timetable["route_short_name"] == "T2")
][
    [
        "trip_id", "stop_id", "stop_sequence",
        "departure_seconds", "departure_time"
    ]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_id": "third_platform",
    "stop_sequence": "third_boarding_sequence",
    "departure_seconds": "third_departure_seconds",
    "departure_time": "third_departure_time"
})

# Get the T9 arrival platform at Burwood
burwood_t9_arrivals = test_timetable[
    test_timetable["station_id"].isin(burwood_ids)
][
    ["trip_id", "stop_id", "stop_sequence", "arrival_seconds"]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "burwood_arrival_platform",
    "stop_sequence": "burwood_arrival_sequence",
    "arrival_seconds": "burwood_arrival_seconds"
})

# Match each second-leg trip to its actual arrival platform
second_leg_arrivals = via_epping.merge(
    burwood_t9_arrivals,
    on="second_trip_id",
    how="inner"
)

second_leg_arrivals = second_leg_arrivals[
    second_leg_arrivals["burwood_arrival_sequence"]
    > second_leg_arrivals["second_boarding_sequence"]
].copy()

# Find valid platform-to-platform transfers
burwood_connections = second_leg_arrivals.merge(
    all_platform_transfers[
        ["from_stop_id", "to_stop_id", "minimum_transfer_seconds"]
    ],
    left_on="burwood_arrival_platform",
    right_on="from_stop_id",
    how="inner"
)

# Match T2 departures on reachable platforms
burwood_connections = burwood_connections.merge(
    burwood_t2_departures,
    left_on="to_stop_id",
    right_on="third_platform",
    how="inner"
)

# Check transfer timing
burwood_connections = burwood_connections[
    burwood_connections["third_departure_seconds"]
    >= (
        burwood_connections["burwood_arrival_seconds"]
        + burwood_connections["minimum_transfer_seconds"]
    )
].copy()

# Find T2 arrivals at Parramatta
parramatta_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Parramatta Station",
        "station_id"
    ]
)

parramatta_arrivals = test_timetable[
    test_timetable["station_id"].isin(parramatta_ids)
][
    ["trip_id", "stop_sequence", "arrival_seconds", "arrival_time"]
].rename(columns={
    "trip_id": "third_trip_id",
    "stop_sequence": "final_sequence",
    "arrival_seconds": "final_arrival_seconds",
    "arrival_time": "final_arrival_time"
})

complete_two_transfer = burwood_connections.merge(
    parramatta_arrivals,
    on="third_trip_id",
    how="inner"
)

complete_two_transfer = complete_two_transfer[
    complete_two_transfer["final_sequence"]
    > complete_two_transfer["third_boarding_sequence"]
].copy()

complete_two_transfer["total_minutes"] = (
    (
        complete_two_transfer["final_arrival_seconds"]
        - complete_two_transfer["origin_departure_seconds"]
    ) / 60
).round(2)

print("Feasible two-transfer journeys:", len(complete_two_transfer))

display(
    complete_two_transfer[
        [
            "origin_departure_time",
            "second_departure_time",
            "destination_arrival_time",
            "third_departure_time",
            "final_arrival_time",
            "minimum_transfer_seconds",
            "total_minutes"
        ]
    ].sort_values("final_arrival_time").head(15)
)

KeyError: 'to_stop_id'

In [128]:
# Select our earliest validated two-transfer journey
test_journey = complete_two_transfer.sort_values(
    "final_arrival_seconds"
).iloc[0]

# Get the arrival platform and time at Burwood
arrival_platform = test_journey["burwood_arrival_platform"]
arrival_seconds = test_journey["burwood_arrival_seconds"]

# Find valid onward connections
connections = get_valid_connections(
    daily=test_timetable,
    arrival_platform=arrival_platform,
    arrival_seconds=arrival_seconds,
    connection_window_minutes=30
)

print("Valid onward boarding events:", len(connections))

display(
    connections[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "stop_id",
            "minimum_transfer_seconds"
        ]
    ].head(20)
)

Valid onward boarding events: 15


,route_short_name,transport_mode,departure_time,stop_id,minimum_transfer_seconds
429,T2,Train,18:13:18,213496,360.0
140,T9,Train,18:17:00,213492,300.0
13,T2,Train,18:17:06,213496,360.0
436,T2,Train,18:18:36,213495,330.0
365,T3,Train,18:20:30,213496,360.0
201,T9,Train,18:21:48,213493,120.0
383,T2,Train,18:22:30,213495,330.0
362,T2,Train,18:24:18,213496,360.0
22,T9,Train,18:26:30,213491,330.0
542,T2,Train,18:28:18,213496,360.0


In [130]:
# STEP 55 — Find which onward connections reach Parramatta

# 1. Identify Parramatta Station
parramatta_ids = set(
    station_lookup.loc[
        station_lookup["stop_name"] == "Parramatta Station",
        "station_id"
    ]
)

# 2. Get scheduled arrivals at Parramatta
parramatta_stops = test_timetable[
    test_timetable["station_id"].isin(parramatta_ids)
][
    [
        "trip_id",
        "stop_sequence",
        "arrival_time",
        "arrival_seconds"
    ]
].copy()

# Rename columns BEFORE merging to avoid naming collisions
parramatta_stops = parramatta_stops.rename(columns={
    "stop_sequence": "destination_sequence",
    "arrival_time": "destination_arrival_time",
    "arrival_seconds": "destination_arrival_seconds"
})

# 3. Prepare valid onward boarding events
boarding_connections = connections[
    [
        "trip_id",
        "route_short_name",
        "transport_mode",
        "stop_id",
        "stop_sequence",
        "departure_time",
        "departure_seconds",
        "minimum_transfer_seconds"
    ]
].copy()

boarding_connections = boarding_connections.rename(columns={
    "stop_id": "boarding_platform",
    "stop_sequence": "boarding_sequence"
})

# 4. Match boarding events to Parramatta arrivals
onward_journeys = boarding_connections.merge(
    parramatta_stops,
    on="trip_id",
    how="inner",
    validate="many_to_many"
)

# 5. Ensure Parramatta comes AFTER the boarding station
onward_journeys = onward_journeys[
    onward_journeys["destination_sequence"]
    > onward_journeys["boarding_sequence"]
].copy()

# 6. Calculate onward travel duration
onward_journeys["onward_duration_min"] = (
    onward_journeys["destination_arrival_seconds"]
    - onward_journeys["departure_seconds"]
) / 60

# 7. Sort by earliest arrival
onward_journeys = onward_journeys.sort_values(
    "destination_arrival_seconds"
)

# 8. Display results
print("Valid onward boarding events:", len(connections))
print("Connections reaching Parramatta:", len(onward_journeys))

display(
    onward_journeys[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "onward_duration_min",
            "minimum_transfer_seconds"
        ]
    ].head(20)
)

Valid onward boarding events: 15
Connections reaching Parramatta: 2


,route_short_name,transport_mode,departure_time,destination_arrival_time,onward_duration_min,minimum_transfer_seconds
0,T2,Train,18:13:18,18:36:30,23.2,360.0
2,T2,Train,18:28:18,18:51:30,23.2,360.0


In [131]:
def find_onward_journeys(
    daily,
    arrival_platform,
    arrival_seconds,
    destination,
    connection_window_minutes=30,
    exclude_trip_ids=None
):
    """
    Find journeys from an arrival platform to a destination
    using one additional scheduled service.
    """

    # 1. Find feasible onward boarding events
    connections = get_valid_connections(
        daily=daily,
        arrival_platform=arrival_platform,
        arrival_seconds=arrival_seconds,
        connection_window_minutes=connection_window_minutes
    )

    if connections.empty:
        return pd.DataFrame()

    # 2. Exclude trips already used in the journey
    if exclude_trip_ids is not None:
        connections = connections[
            ~connections["trip_id"].isin(exclude_trip_ids)
        ].copy()

    if connections.empty:
        return pd.DataFrame()

    # 3. Find destination station IDs
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    if not destination_ids:
        raise ValueError(f"Destination not found: {destination}")

    # 4. Get destination arrival events
    destination_stops = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_time",
            "arrival_seconds"
        ]
    ].rename(columns={
        "stop_sequence": "destination_sequence",
        "arrival_time": "destination_arrival_time",
        "arrival_seconds": "destination_arrival_seconds"
    })

    # 5. Prepare boarding events with explicit names
    boarding = connections[
        [
            "trip_id",
            "route_short_name",
            "transport_mode",
            "stop_id",
            "stop_sequence",
            "departure_time",
            "departure_seconds",
            "minimum_transfer_seconds"
        ]
    ].rename(columns={
        "stop_id": "boarding_platform",
        "stop_sequence": "boarding_sequence"
    })

    # 6. Match boarding trips to destination arrivals
    journeys = boarding.merge(
        destination_stops,
        on="trip_id",
        how="inner"
    )

    # 7. Ensure the destination occurs after boarding
    journeys = journeys[
        journeys["destination_sequence"]
        > journeys["boarding_sequence"]
    ].copy()

    # 8. Calculate travel duration
    journeys["onward_duration_min"] = (
        journeys["destination_arrival_seconds"]
        - journeys["departure_seconds"]
    ) / 60

    return journeys.sort_values(
        "destination_arrival_seconds"
    ).reset_index(drop=True)

In [132]:
# STEP 57 — Test reusable onward journey function

test_journey = complete_two_transfer.sort_values(
    "final_arrival_seconds"
).iloc[0]

# Find onward journeys from Burwood to Parramatta
test_onward = find_onward_journeys(
    daily=test_timetable,
    arrival_platform=test_journey["burwood_arrival_platform"],
    arrival_seconds=test_journey["burwood_arrival_seconds"],
    destination="Parramatta Station",
    connection_window_minutes=30,
    exclude_trip_ids=[
        test_journey["trip_id"],
        test_journey["second_trip_id"]
    ]
)

print("Valid onward journeys:", len(test_onward))

display(
    test_onward[
        [
            "route_short_name",
            "transport_mode",
            "departure_time",
            "destination_arrival_time",
            "onward_duration_min",
            "minimum_transfer_seconds"
        ]
    ]
)

Valid onward journeys: 2


,route_short_name,transport_mode,departure_time,destination_arrival_time,onward_duration_min,minimum_transfer_seconds
0,T2,Train,18:13:18,18:36:30,23.2,360.0
1,T2,Train,18:28:18,18:51:30,23.2,360.0


In [134]:
# STEP 58 — Discover possible second interchange stations

# 1. Select the earliest validated two-transfer journey
test_journey = complete_two_transfer.sort_values(
    "final_arrival_seconds"
).iloc[0]

# 2. Identify the second train (T9)
second_trip_id = test_journey["second_trip_id"]
boarding_sequence = test_journey["second_boarding_sequence"]

# 3. Find all downstream stops on the second train
downstream_stops = test_timetable[
    (test_timetable["trip_id"] == second_trip_id)
    & (test_timetable["stop_sequence"] > boarding_sequence)
].copy()

# 4. Prepare a clean station lookup
# Rename BEFORE merging to avoid column naming conflicts
station_names = station_lookup[
    ["station_id", "stop_name"]
].drop_duplicates(
    subset=["station_id"]
).rename(columns={
    "stop_name": "interchange_station"
})

# 5. Attach official station names
downstream_stops = downstream_stops.merge(
    station_names,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# 6. Sort stops by arrival time
downstream_stops = downstream_stops.sort_values(
    "arrival_seconds"
).reset_index(drop=True)

# 7. Display results
print("Second train:", second_trip_id)
print("Possible downstream interchange stops:", len(downstream_stops))

display(
    downstream_stops[
        [
            "interchange_station",
            "arrival_time",
            "stop_id",
            "arrival_seconds"
        ]
    ]
)

Second train: 125R.1324.165.16.A.8.92001037
Possible downstream interchange stops: 11


,interchange_station,arrival_time,stop_id,arrival_seconds
0,Eastwood Station,17:45:00,2122231,63900
1,Denistone Station,17:47:42,2114171,64062
2,West Ryde Station,17:49:30,2114191,64170
3,Meadowbank Station,17:51:42,2114181,64302
4,Rhodes Station,17:53:48,213891,64428
5,Concord West Station,17:56:30,213881,64590
6,North Strathfield Station,17:58:30,2137171,64710
7,Strathfield Station,18:03:00,2135235,64980
8,Burwood Station,18:06:18,213493,65178
9,Redfern Station,18:15:42,2015133,65742


In [135]:
# STEP 59 — Find onward journeys from every downstream station

all_onward_options = []

# Trips already used in the journey
used_trips = [
    test_journey["trip_id"],
    test_journey["second_trip_id"]
]

for _, stop in downstream_stops.iterrows():

    # Find valid onward journeys to Parramatta
    onward = find_onward_journeys(
        daily=test_timetable,
        arrival_platform=stop["stop_id"],
        arrival_seconds=stop["arrival_seconds"],
        destination="Parramatta Station",
        connection_window_minutes=30,
        exclude_trip_ids=used_trips
    )

    if onward.empty:
        continue

    # Record the interchange station
    onward["second_interchange"] = stop["interchange_station"]

    # Record arrival time at the interchange
    onward["interchange_arrival_time"] = stop["arrival_time"]

    # Calculate total journey duration from Macquarie University
    onward["total_journey_min"] = (
        onward["destination_arrival_seconds"]
        - test_journey["origin_departure_seconds"]
    ) / 60

    all_onward_options.append(onward)

# Combine results
if all_onward_options:

    discovered_routes = pd.concat(
        all_onward_options,
        ignore_index=True
    )

    discovered_routes = discovered_routes.sort_values(
        "destination_arrival_seconds"
    ).reset_index(drop=True)

    print(
        "Total feasible onward connections:",
        len(discovered_routes)
    )

    print(
        "Interchange stations with valid connections:",
        discovered_routes["second_interchange"].nunique()
    )

    display(
        discovered_routes[
            [
                "second_interchange",
                "interchange_arrival_time",
                "route_short_name",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min",
                "minimum_transfer_seconds"
            ]
        ].head(20)
    )

else:
    discovered_routes = pd.DataFrame()
    print("No feasible onward connections found.")

Total feasible onward connections: 21
Interchange stations with valid connections: 4


,second_interchange,interchange_arrival_time,route_short_name,departure_time,destination_arrival_time,total_journey_min,minimum_transfer_seconds
0,Strathfield Station,18:03:00,T1,18:11:24,18:23:00,52.383333,300.0
1,Strathfield Station,18:03:00,T1,18:17:24,18:29:00,58.383333,300.0
2,Strathfield Station,18:03:00,T1,18:20:24,18:32:00,61.383333,300.0
3,Strathfield Station,18:03:00,T2,18:16:00,18:36:30,65.883333,384.0
4,Burwood Station,18:06:18,T2,18:13:18,18:36:30,65.883333,360.0
5,Strathfield Station,18:03:00,T1,18:26:24,18:38:00,67.383333,300.0
6,Strathfield Station,18:03:00,T1,18:29:24,18:41:00,70.383333,300.0
7,Redfern Station,18:15:42,T1,18:21:36,18:44:00,73.383333,312.0
8,Strathfield Station,18:03:00,T1,18:32:24,18:44:00,73.383333,300.0
9,Burwood Station,18:06:18,T2,18:28:18,18:51:30,80.883333,360.0


In [136]:
# STEP 60 — Validate the fastest discovered two-transfer journey

fastest = discovered_routes.sort_values(
    "destination_arrival_seconds"
).iloc[0]

# Confirm that the third train is different from the first two
distinct_trips = (
    fastest["trip_id"] not in [
        test_journey["trip_id"],
        test_journey["second_trip_id"]
    ]
)

# Check first transfer: Epping
epping_transfer_valid = (
    test_journey["second_departure_seconds"]
    >= (
        test_journey["arrival_seconds"]
        + test_journey["epping_transfer_seconds"]
    )
)

# Check second transfer: Strathfield
strathfield_transfer_valid = (
    fastest["departure_seconds"]
    >= (
        fastest["interchange_arrival_time"]
        and time_to_seconds(fastest["interchange_arrival_time"])
        + fastest["minimum_transfer_seconds"]
    )
)

# Confirm arrival is after final departure
destination_valid = (
    fastest["destination_arrival_seconds"]
    > fastest["departure_seconds"]
)

print("Three distinct trips:", distinct_trips)
print("Epping transfer valid:", epping_transfer_valid)
print("Strathfield transfer valid:", strathfield_transfer_valid)
print("Destination timing valid:", destination_valid)

print("\nFastest discovered journey:")
print("Second interchange:", fastest["second_interchange"])
print("Final train:", fastest["route_short_name"])
print("Final departure:", fastest["departure_time"])
print("Parramatta arrival:", fastest["destination_arrival_time"])
print("Total duration (minutes):", round(fastest["total_journey_min"], 2))

Three distinct trips: True
Epping transfer valid: True
Strathfield transfer valid: True
Destination timing valid: True

Fastest discovered journey:
Second interchange: Strathfield Station
Final train: T1
Final departure: 18:11:24
Parramatta arrival: 18:23:00
Total duration (minutes): 52.38


In [137]:
# STEP 61 — Reusable downstream stop discovery

def get_downstream_stops(daily, trip_id, boarding_sequence):
    """
    Return all stops visited after boarding a given trip.

    These stops can be potential destinations or interchanges.
    """

    downstream = daily.loc[
        (daily["trip_id"] == trip_id)
        & (daily["stop_sequence"] > boarding_sequence),
        [
            "trip_id",
            "stop_id",
            "station_id",
            "stop_sequence",
            "arrival_time",
            "arrival_seconds"
        ]
    ].copy()

    # Explicitly name the station lookup column
    names = station_lookup[
        ["station_id", "stop_name"]
    ].drop_duplicates("station_id").rename(
        columns={"stop_name": "station_name"}
    )

    downstream = downstream.merge(
        names,
        on="station_id",
        how="left",
        validate="many_to_one"
    )

    return downstream.sort_values(
        ["stop_sequence", "arrival_seconds"]
    ).reset_index(drop=True)


# Test on our validated T9 journey
test_downstream = get_downstream_stops(
    daily=test_timetable,
    trip_id=test_journey["second_trip_id"],
    boarding_sequence=test_journey["second_boarding_sequence"]
)

print("Downstream stops:", len(test_downstream))

display(
    test_downstream[
        ["station_name", "arrival_time", "stop_id"]
    ]
)

Downstream stops: 11


,station_name,arrival_time,stop_id
0,Eastwood Station,17:45:00,2122231
1,Denistone Station,17:47:42,2114171
2,West Ryde Station,17:49:30,2114191
3,Meadowbank Station,17:51:42,2114181
4,Rhodes Station,17:53:48,213891
5,Concord West Station,17:56:30,213881
6,North Strathfield Station,17:58:30,2137171
7,Strathfield Station,18:03:00,2135235
8,Burwood Station,18:06:18,213493
9,Redfern Station,18:15:42,2015133


In [138]:
# STEP 62A — Find first-transfer journeys to Strathfield

first_transfer_candidates = find_one_transfer_journeys(
    daily=test_timetable,
    origin="Macquarie University Station",
    destination="Strathfield Station",
    departure_time="17:30",
    window_minutes=30,
    connection_window_minutes=30
)

print("First-transfer candidates:", len(first_transfer_candidates))

display(
    first_transfer_candidates[
        [
            "first_line",
            "origin_departure_time",
            "transfer_station",
            "second_line",
            "second_departure_time",
            "destination_arrival_time",
            "journey_duration_min"
        ]
    ].sort_values(
        "destination_arrival_time"
    ).head(15)
)

First-transfer candidates: 143


,first_line,origin_departure_time,transfer_station,second_line,second_departure_time,destination_arrival_time,journey_duration_min
0,M1,17:30:37,Epping Station,T9,17:42:00,18:03:00,32.38
1,M1,17:30:37,Epping Station,T9,17:53:01,18:07:48,37.18
2,M1,17:34:37,Epping Station,T9,17:53:01,18:07:48,33.18
3,M1,17:38:37,Epping Station,T9,17:53:01,18:07:48,29.18
4,M1,17:42:37,Epping Station,T9,17:53:01,18:07:48,25.18
5,M1,17:31:54,Central Station,T1,18:04:01,18:16:30,44.60
6,M1,17:30:37,Epping Station,T9,17:57:00,18:18:00,47.38
7,M1,17:34:37,Epping Station,T9,17:57:00,18:18:00,43.38
8,M1,17:38:37,Epping Station,T9,17:57:00,18:18:00,39.38
9,M1,17:42:37,Epping Station,T9,17:57:00,18:18:00,35.38


In [139]:
# STEP 62B — Extend one-transfer journeys to a third service

# Keep unique second-leg arrival events
candidates = (
    first_transfer_candidates
    .sort_values("destination_arrival_seconds")
    .drop_duplicates(
        subset=[
            "second_trip_id",
            "destination_arrival_seconds"
        ]
    )
    .head(20)
    .copy()
)

all_three_leg_journeys = []

for _, journey in candidates.iterrows():

    # Identify the actual arrival platform at Strathfield
    arrival_events = test_timetable[
        (test_timetable["trip_id"] == journey["second_trip_id"])
        & (test_timetable["station_id"].isin(
            station_lookup.loc[
                station_lookup["stop_name"] == "Strathfield Station",
                "station_id"
            ]
        ))
        & (
            test_timetable["stop_sequence"]
            == journey["destination_sequence"]
        )
    ]

    for _, arrival in arrival_events.iterrows():

        # Find a valid third service to Parramatta
        onward = find_onward_journeys(
            daily=test_timetable,
            arrival_platform=arrival["stop_id"],
            arrival_seconds=arrival["arrival_seconds"],
            destination="Parramatta Station",
            connection_window_minutes=30,
            exclude_trip_ids=[
                journey["trip_id"],
                journey["second_trip_id"]
            ]
        )

        if onward.empty:
            continue

        onward = onward.copy()

        onward["first_line"] = journey["first_line"]
        onward["second_line"] = journey["second_line"]
        onward["first_transfer"] = journey["transfer_station"]
        onward["second_transfer"] = "Strathfield Station"
        onward["origin_departure_time"] = journey["origin_departure_time"]
        onward["origin_departure_seconds"] = journey["origin_departure_seconds"]

        onward["total_journey_min"] = (
            onward["destination_arrival_seconds"]
            - onward["origin_departure_seconds"]
        ) / 60

        all_three_leg_journeys.append(onward)

if all_three_leg_journeys:

    three_leg_results = pd.concat(
        all_three_leg_journeys,
        ignore_index=True
    )

    three_leg_results["route"] = (
        three_leg_results["first_line"].astype(str)
        + " → "
        + three_leg_results["second_line"].astype(str)
        + " → "
        + three_leg_results["route_short_name"].astype(str)
    )

    three_leg_results = three_leg_results.sort_values(
        "destination_arrival_seconds"
    )

    print("Three-leg journey combinations:", len(three_leg_results))

    display(
        three_leg_results[
            [
                "route",
                "first_transfer",
                "second_transfer",
                "origin_departure_time",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min"
            ]
        ].head(15)
    )

else:
    three_leg_results = pd.DataFrame()
    print("No valid three-leg journeys found.")

Three-leg journey combinations: 115


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,total_journey_min
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:11:24,18:23:00,52.383333
8,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
9,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
14,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:20:24,18:32:00,60.100000
3,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:16:00,18:36:30,65.883333
4,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:26:24,18:38:00,67.383333
43,M1 → T9 → T1,Central Station,Strathfield Station,17:35:54,18:26:24,18:38:00,62.100000
10,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:26:24,18:38:00,67.383333


In [140]:
# STEP 63 — Automatically discover second interchange stations

# Start with 5 unique second-leg trips
sample_candidates = (
    first_transfer_candidates
    .sort_values("destination_arrival_seconds")
    .drop_duplicates(subset=["second_trip_id"])
    .head(5)
    .copy()
)

all_auto_routes = []

for _, journey in sample_candidates.iterrows():

    # Find every downstream station on the second train
    downstream = get_downstream_stops(
        daily=test_timetable,
        trip_id=journey["second_trip_id"],
        boarding_sequence=journey["second_boarding_sequence"]
    )

    for _, stop in downstream.iterrows():

        # Avoid transferring at the origin or first interchange
        if stop["station_name"] in [
            "Macquarie University Station",
            journey["transfer_station"]
        ]:
            continue

        # Find onward services to Parramatta
        onward = find_onward_journeys(
            daily=test_timetable,
            arrival_platform=stop["stop_id"],
            arrival_seconds=stop["arrival_seconds"],
            destination="Parramatta Station",
            connection_window_minutes=30,
            exclude_trip_ids=[
                journey["trip_id"],
                journey["second_trip_id"]
            ]
        )

        if onward.empty:
            continue

        onward = onward.copy()

        # Store the full journey information
        onward["first_line"] = journey["first_line"]
        onward["second_line"] = journey["second_line"]
        onward["first_transfer"] = journey["transfer_station"]
        onward["second_transfer"] = stop["station_name"]
        onward["origin_departure_time"] = journey["origin_departure_time"]
        onward["origin_departure_seconds"] = journey["origin_departure_seconds"]

        onward["total_journey_min"] = (
            onward["destination_arrival_seconds"]
            - onward["origin_departure_seconds"]
        ) / 60

        all_auto_routes.append(onward)

if all_auto_routes:

    auto_routes = pd.concat(
        all_auto_routes,
        ignore_index=True
    )

    auto_routes["route"] = (
        auto_routes["first_line"].astype(str)
        + " → "
        + auto_routes["second_line"].astype(str)
        + " → "
        + auto_routes["route_short_name"].astype(str)
    )

    auto_routes = auto_routes.sort_values(
        "destination_arrival_seconds"
    )

    print("Automatically discovered journeys:", len(auto_routes))
    print(
        "Second interchange stations:",
        auto_routes["second_transfer"].nunique()
    )

    display(
        auto_routes[
            [
                "route",
                "first_transfer",
                "second_transfer",
                "origin_departure_time",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min"
            ]
        ].head(20)
    )

else:
    auto_routes = pd.DataFrame()
    print("No two-transfer journeys found.")

Automatically discovered journeys: 98
Second interchange stations: 14


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,total_journey_min
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:11:24,18:23:00,52.383333
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
21,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
43,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:20:24,18:32:00,60.100000
36,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:09:36,18:32:00,60.100000
22,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:13:18,18:36:30,65.883333
3,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:16:00,18:36:30,65.883333
37,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:15:36,18:38:00,66.100000


In [141]:
# STEP 64 — Deduplicate journeys using actual trip IDs

print("Original journey combinations:", len(auto_routes))

# Inspect available journey identifiers
print("\nAvailable trip-related columns:")
print([
    col for col in auto_routes.columns
    if "trip" in col.lower()
])

# Check which trip identifiers we have
display(
    auto_routes[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "origin_departure_time",
            "destination_arrival_time",
            "trip_id"
        ]
    ].head(10)
)

Original journey combinations: 98

Available trip-related columns:
['trip_id']


,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,trip_id
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,197F.1324.165.16.B.8.92001727
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,145N.1324.165.16.A.8.92001340
21,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,145N.1324.165.16.A.8.92001340
43,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:32:00,165F.1324.165.124.A.8.92002689
36,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:32:00,165F.1324.165.124.A.8.92002689
22,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,165F.1324.165.124.A.8.92002689
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,165F.1324.165.124.A.8.92002689
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:36:30,78-P.1324.165.16.A.8.92002733
3,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:36:30,78-P.1324.165.16.A.8.92002733
37,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:38:00,155N.1324.165.16.A.8.92002106


In [142]:
# STEP 65 — Rebuild automatically discovered journeys with all trip IDs

sample_candidates = (
    first_transfer_candidates
    .sort_values("destination_arrival_seconds")
    .drop_duplicates(subset=["second_trip_id"])
    .head(5)
    .copy()
)

all_auto_routes = []

for _, journey in sample_candidates.iterrows():

    downstream = get_downstream_stops(
        daily=test_timetable,
        trip_id=journey["second_trip_id"],
        boarding_sequence=journey["second_boarding_sequence"]
    )

    for _, stop in downstream.iterrows():

        if stop["station_name"] in [
            "Macquarie University Station",
            journey["transfer_station"]
        ]:
            continue

        onward = find_onward_journeys(
            daily=test_timetable,
            arrival_platform=stop["stop_id"],
            arrival_seconds=stop["arrival_seconds"],
            destination="Parramatta Station",
            connection_window_minutes=30,
            exclude_trip_ids=[
                journey["trip_id"],
                journey["second_trip_id"]
            ]
        )

        if onward.empty:
            continue

        onward = onward.copy()

        # Preserve all three trip identifiers
        onward["first_trip_id"] = journey["trip_id"]
        onward["second_trip_id"] = journey["second_trip_id"]
        onward["third_trip_id"] = onward["trip_id"]

        onward["first_line"] = journey["first_line"]
        onward["second_line"] = journey["second_line"]

        onward["first_transfer"] = journey["transfer_station"]
        onward["second_transfer"] = stop["station_name"]

        onward["origin_departure_time"] = journey["origin_departure_time"]
        onward["origin_departure_seconds"] = journey["origin_departure_seconds"]

        onward["total_journey_min"] = (
            onward["destination_arrival_seconds"]
            - onward["origin_departure_seconds"]
        ) / 60

        all_auto_routes.append(onward)

if all_auto_routes:

    auto_routes = pd.concat(
        all_auto_routes,
        ignore_index=True
    )

    auto_routes["route"] = (
        auto_routes["first_line"].astype(str)
        + " → "
        + auto_routes["second_line"].astype(str)
        + " → "
        + auto_routes["route_short_name"].astype(str)
    )

    # Deduplicate using the complete itinerary
    unique_routes = (
        auto_routes
        .drop_duplicates(
            subset=[
                "first_trip_id",
                "second_trip_id",
                "third_trip_id",
                "first_transfer",
                "second_transfer",
                "origin_departure_seconds",
                "departure_seconds"
            ]
        )
        .sort_values(
            ["destination_arrival_seconds", "origin_departure_seconds"]
        )
        .reset_index(drop=True)
    )

    print("Original combinations:", len(auto_routes))
    print("Unique itineraries:", len(unique_routes))
    print("Duplicates removed:", len(auto_routes) - len(unique_routes))

    display(
        unique_routes[
            [
                "route",
                "first_transfer",
                "second_transfer",
                "origin_departure_time",
                "departure_time",
                "destination_arrival_time",
                "total_journey_min"
            ]
        ].head(15)
    )

else:
    auto_routes = pd.DataFrame()
    unique_routes = pd.DataFrame()
    print("No two-transfer journeys found.")

Original combinations: 98
Unique itineraries: 98
Duplicates removed: 0


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,total_journey_min
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:11:24,18:23:00,52.383333
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,58.383333
3,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
4,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:20:24,18:32:00,61.383333
5,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:09:36,18:32:00,60.100000
6,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:20:24,18:32:00,60.100000
7,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:16:00,18:36:30,65.883333
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:13:18,18:36:30,65.883333
9,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:26:24,18:38:00,67.383333


In [143]:
# STEP 66 — Investigate apparently duplicate journeys

# Find journeys with identical passenger-facing details
passenger_columns = [
    "route",
    "first_transfer",
    "second_transfer",
    "origin_departure_time",
    "departure_time",
    "destination_arrival_time"
]

duplicate_mask = unique_routes.duplicated(
    subset=passenger_columns,
    keep=False
)

apparent_duplicates = unique_routes.loc[
    duplicate_mask
].copy()

print("Total unique itineraries:", len(unique_routes))
print("Apparently duplicate rows:", len(apparent_duplicates))

print(
    "Distinct passenger-facing journeys:",
    len(unique_routes.drop_duplicates(subset=passenger_columns))
)

# Compare underlying trip IDs
display(
    apparent_duplicates[
        passenger_columns + [
            "first_trip_id",
            "second_trip_id",
            "third_trip_id"
        ]
    ]
    .sort_values(passenger_columns)
    .head(15)
)

Total unique itineraries: 98
Apparently duplicate rows: 39
Distinct passenger-facing journeys: 75


,route,first_transfer,second_transfer,origin_departure_time,departure_time,destination_arrival_time,first_trip_id,second_trip_id,third_trip_id
38,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:30:01,18:56:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,107N.1324.165.16.T.8.92001465
39,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:30:01,18:56:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,107N.1324.165.16.T.8.92001465
46,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:33:01,18:59:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,143N.1324.165.16.A.8.92000923
47,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:33:01,18:59:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,143N.1324.165.16.A.8.92000923
61,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:45:01,19:11:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,157P.1324.165.16.A.8.92002747
62,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:45:01,19:11:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,157P.1324.165.16.A.8.92002747
64,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:45:01,19:11:00,0160-006-154-013:1000,191J.1324.165.16.B.8.92003392,157P.1324.165.16.A.8.92002747
69,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:48:01,19:14:00,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126,195P.1324.165.16.B.8.92001854
71,M1 → T9 → T1,Epping Station,Central Station,17:30:37,18:48:01,19:14:00,0160-006-154-013:1000,191J.1324.165.16.B.8.92003392,195P.1324.165.16.B.8.92001854
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:17:24,18:29:00,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,145N.1324.165.16.A.8.92001340


In [144]:
# STEP 67 — Compare apparently identical journeys

# Select the two journeys from rows 1 and 2
comparison_rows = unique_routes.loc[[1, 2]].copy()

results = []

for _, journey in comparison_rows.iterrows():

    # Find the middle train's arrival at Strathfield
    middle_arrival = test_timetable[
        (test_timetable["trip_id"] == journey["second_trip_id"])
        & (
            test_timetable["station_id"].isin(
                station_lookup.loc[
                    station_lookup["stop_name"] == "Strathfield Station",
                    "station_id"
                ]
            )
        )
    ].sort_values("arrival_seconds")

    if middle_arrival.empty:
        continue

    arrival = middle_arrival.iloc[0]

    results.append({
        "second_trip_id": journey["second_trip_id"],
        "middle_arrival_time": arrival["arrival_time"],
        "third_departure_time": journey["departure_time"],
        "waiting_minutes": round(
            (
                journey["departure_seconds"]
                - arrival["arrival_seconds"]
            ) / 60,
            2
        ),
        "final_arrival": journey["destination_arrival_time"]
    })

transfer_comparison = pd.DataFrame(results)

display(transfer_comparison)

,second_trip_id,middle_arrival_time,third_departure_time,waiting_minutes,final_arrival
0,125R.1324.165.16.A.8.92001037,18:03:00,18:17:24,14.4,18:29:00
1,130N.1324.165.16.A.8.92003126,18:07:48,18:17:24,9.6,18:29:00


In [145]:
# STEP 68 — Calculate second-transfer connection quality

ranked_routes = unique_routes.copy()

# Get actual arrival events for each second-leg trip
middle_arrivals = test_timetable[
    [
        "trip_id",
        "stop_id",
        "station_id",
        "arrival_seconds",
        "arrival_time"
    ]
].rename(columns={
    "trip_id": "second_trip_id",
    "stop_id": "second_arrival_platform",
    "arrival_seconds": "second_arrival_seconds",
    "arrival_time": "second_arrival_time"
})

# Match the second-leg arrival to the third-leg boarding connection
ranked_routes = ranked_routes.merge(
    middle_arrivals,
    left_on=[
        "second_trip_id",
        "boarding_platform"
    ],
    right_on=[
        "second_trip_id",
        "second_arrival_platform"
    ],
    how="left"
) if False else ranked_routes

# Use the existing transfer information from the onward search
# Recover arrival time by matching each trip to its second interchange
station_ids = station_lookup.rename(columns={
    "stop_name": "second_transfer"
})

middle_arrivals = middle_arrivals.merge(
    station_ids[["station_id", "second_transfer"]],
    on="station_id",
    how="left"
)

ranked_routes = ranked_routes.merge(
    middle_arrivals[
        [
            "second_trip_id",
            "second_transfer",
            "second_arrival_platform",
            "second_arrival_seconds",
            "second_arrival_time"
        ]
    ],
    on=["second_trip_id", "second_transfer"],
    how="left"
)

# Calculate connection quality
ranked_routes["waiting_minutes"] = (
    ranked_routes["departure_seconds"]
    - ranked_routes["second_arrival_seconds"]
) / 60

ranked_routes["spare_connection_minutes"] = (
    ranked_routes["departure_seconds"]
    - ranked_routes["second_arrival_seconds"]
    - ranked_routes["minimum_transfer_seconds"]
) / 60

# Only retain valid connections
ranked_routes = ranked_routes[
    ranked_routes["spare_connection_minutes"] >= 0
].copy()

print("Journeys with valid second transfers:", len(ranked_routes))

display(
    ranked_routes[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "second_arrival_time",
            "departure_time",
            "waiting_minutes",
            "spare_connection_minutes",
            "destination_arrival_time"
        ]
    ]
    .sort_values("destination_arrival_time")
    .head(15)
)

Journeys with valid second transfers: 98


,route,first_transfer,second_transfer,second_arrival_time,departure_time,waiting_minutes,spare_connection_minutes,destination_arrival_time
0,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:11:24,8.4,3.4,18:23:00
1,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:17:24,14.4,9.4,18:29:00
2,M1 → T9 → T1,Epping Station,Strathfield Station,18:07:48,18:17:24,9.6,2.2,18:29:00
3,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:20:24,17.4,12.4,18:32:00
4,M1 → T9 → T1,Epping Station,Strathfield Station,18:07:48,18:20:24,12.6,5.2,18:32:00
5,M1 → T1 → T1,Central Station,Redfern Station,18:06:06,18:09:36,3.5,1.5,18:32:00
6,M1 → T1 → T1,Central Station,Strathfield Station,18:16:30,18:20:24,3.9,1.9,18:32:00
7,M1 → T9 → T2,Epping Station,Strathfield Station,18:03:00,18:16:00,13.0,6.6,18:36:30
8,M1 → T9 → T2,Epping Station,Burwood Station,18:06:18,18:13:18,7.0,1.0,18:36:30
9,M1 → T9 → T1,Epping Station,Strathfield Station,18:03:00,18:26:24,23.4,18.4,18:38:00


In [146]:
# STEP 69 — Rank journeys by arrival time and transfer practicality

journey_ranking = ranked_routes.copy()

# Keep only journeys with valid connection times
journey_ranking = journey_ranking[
    journey_ranking["spare_connection_minutes"] >= 0
].copy()

# Calculate journey duration
journey_ranking["journey_duration_min"] = (
    journey_ranking["destination_arrival_seconds"]
    - journey_ranking["origin_departure_seconds"]
) / 60

# Categorise second-transfer connection comfort
def connection_category(spare_minutes):
    if spare_minutes < 2:
        return "Tight"
    elif spare_minutes < 5:
        return "Moderate"
    else:
        return "Comfortable"

journey_ranking["connection_quality"] = (
    journey_ranking["spare_connection_minutes"]
    .apply(connection_category)
)

# Remove passenger-facing duplicates, keeping the option
# with the greatest spare connection time
passenger_columns = [
    "route",
    "first_transfer",
    "second_transfer",
    "origin_departure_time",
    "departure_time",
    "destination_arrival_time"
]

journey_ranking = (
    journey_ranking
    .sort_values("spare_connection_minutes", ascending=False)
    .drop_duplicates(subset=passenger_columns)
)

# Rank by earliest arrival, then duration,
# then connection buffer
journey_ranking = (
    journey_ranking
    .sort_values(
        [
            "destination_arrival_seconds",
            "journey_duration_min",
            "spare_connection_minutes"
        ],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

print("Ranked passenger-facing journeys:", len(journey_ranking))

display(
    journey_ranking[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "spare_connection_minutes",
            "connection_quality"
        ]
    ].head(15)
)

Ranked passenger-facing journeys: 75


,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,spare_connection_minutes,connection_quality
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,3.4,Moderate
1,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,58.383333,9.4,Comfortable
2,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:32:00,60.100000,1.9,Tight
3,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:32:00,60.100000,1.5,Tight
4,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,61.383333,12.4,Comfortable
5,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:36:30,65.883333,6.6,Comfortable
6,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:36:30,65.883333,1.0,Tight
7,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:38:00,66.100000,7.9,Comfortable
8,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:38:00,66.100000,7.5,Comfortable
9,M1 → T9 → T1,Central Station,Strathfield Station,17:31:54,18:38:00,66.100000,1.0,Tight


In [147]:
# STEP 70 — Calculate first-transfer connection buffers

# Build a lookup of the first transfer
first_transfer_lookup = first_transfer_candidates[
    [
        "trip_id",
        "second_trip_id",
        "origin_departure_seconds",
        "arrival_seconds",
        "second_departure_seconds",
        "from_stop_id",
        "to_stop_id",
        "minimum_transfer_seconds"
    ]
].copy()

first_transfer_lookup = first_transfer_lookup.rename(
    columns={
        "trip_id": "first_trip_id",
        "arrival_seconds": "first_transfer_arrival_seconds",
        "second_departure_seconds": "first_transfer_departure_seconds",
        "minimum_transfer_seconds": "first_minimum_transfer_seconds"
    }
)

# Calculate waiting and spare connection time
first_transfer_lookup["first_waiting_minutes"] = (
    first_transfer_lookup["first_transfer_departure_seconds"]
    - first_transfer_lookup["first_transfer_arrival_seconds"]
) / 60

first_transfer_lookup["first_spare_minutes"] = (
    first_transfer_lookup["first_transfer_departure_seconds"]
    - first_transfer_lookup["first_transfer_arrival_seconds"]
    - first_transfer_lookup["first_minimum_transfer_seconds"]
) / 60

# Preserve separate first-transfer alternatives
first_transfer_lookup = (
    first_transfer_lookup
    .sort_values("first_spare_minutes", ascending=False)
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds"
        ]
    )
)

# Merge into ranked journeys
journey_ranking = journey_ranking.merge(
    first_transfer_lookup[
        [
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "first_waiting_minutes",
            "first_spare_minutes"
        ]
    ],
    on=[
        "first_trip_id",
        "second_trip_id",
        "origin_departure_seconds"
    ],
    how="left",
    validate="many_to_one"
)

# Overall journey comfort is determined by the tighter transfer
journey_ranking["minimum_spare_minutes"] = journey_ranking[
    ["first_spare_minutes", "spare_connection_minutes"]
].min(axis=1)

def overall_connection_quality(spare):
    if pd.isna(spare):
        return "Unknown"
    if spare < 2:
        return "Tight"
    elif spare < 5:
        return "Moderate"
    return "Comfortable"

journey_ranking["overall_connection_quality"] = (
    journey_ranking["minimum_spare_minutes"]
    .apply(overall_connection_quality)
)

print(
    "Journeys with first-transfer data:",
    journey_ranking["first_spare_minutes"].notna().sum()
)

print("Total ranked journeys:", len(journey_ranking))

display(
    journey_ranking[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "destination_arrival_time",
            "first_spare_minutes",
            "spare_connection_minutes",
            "minimum_spare_minutes",
            "overall_connection_quality"
        ]
    ].head(15)
)

Journeys with first-transfer data: 75
Total ranked journeys: 75


,route,first_transfer,second_transfer,destination_arrival_time,first_spare_minutes,spare_connection_minutes,minimum_spare_minutes,overall_connection_quality
0,M1 → T9 → T1,Epping Station,Strathfield Station,18:23:00,2.650000,3.4,2.650000,Moderate
1,M1 → T9 → T1,Epping Station,Strathfield Station,18:29:00,2.650000,9.4,2.650000,Moderate
2,M1 → T1 → T1,Central Station,Strathfield Station,18:32:00,1.766667,1.9,1.766667,Tight
3,M1 → T1 → T1,Central Station,Redfern Station,18:32:00,1.766667,1.5,1.500000,Tight
4,M1 → T9 → T1,Epping Station,Strathfield Station,18:32:00,2.650000,12.4,2.650000,Moderate
5,M1 → T9 → T2,Epping Station,Strathfield Station,18:36:30,2.650000,6.6,2.650000,Moderate
6,M1 → T9 → T2,Epping Station,Burwood Station,18:36:30,2.650000,1.0,1.000000,Tight
7,M1 → T1 → T1,Central Station,Strathfield Station,18:38:00,1.766667,7.9,1.766667,Tight
8,M1 → T1 → T1,Central Station,Redfern Station,18:38:00,1.766667,7.5,1.766667,Tight
9,M1 → T9 → T1,Central Station,Strathfield Station,18:38:00,4.366667,1.0,1.000000,Tight


In [148]:
# STEP 71 — Combine one-transfer and two-transfer journeys

# One-transfer journeys
one_transfer = journey_test_2.copy()

one_transfer_results = pd.DataFrame({
    "route": (
        one_transfer["first_line"].astype(str)
        + " → "
        + one_transfer["second_line"].astype(str)
    ),
    "first_transfer": one_transfer["transfer_station"],
    "second_transfer": None,
    "origin_departure_time": one_transfer["origin_departure_time"],
    "destination_arrival_time": one_transfer["destination_arrival_time"],
    "origin_departure_seconds": one_transfer["origin_departure_seconds"],
    "destination_arrival_seconds": one_transfer["destination_arrival_seconds"],
    "journey_duration_min": one_transfer["journey_duration_min"],
    "transfers": 1,
    "connection_quality": "Not yet evaluated"
})

# Two-transfer journeys
two_transfer_results = pd.DataFrame({
    "route": journey_ranking["route"],
    "first_transfer": journey_ranking["first_transfer"],
    "second_transfer": journey_ranking["second_transfer"],
    "origin_departure_time": journey_ranking["origin_departure_time"],
    "destination_arrival_time": journey_ranking["destination_arrival_time"],
    "origin_departure_seconds": journey_ranking["origin_departure_seconds"],
    "destination_arrival_seconds": journey_ranking["destination_arrival_seconds"],
    "journey_duration_min": journey_ranking["journey_duration_min"],
    "transfers": 2,
    "connection_quality": journey_ranking["overall_connection_quality"]
})

# Combine both types
combined_journeys = pd.concat(
    [one_transfer_results, two_transfer_results],
    ignore_index=True
)

# Sort by arrival time, then fewer transfers
combined_journeys = (
    combined_journeys
    .sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    )
    .reset_index(drop=True)
)

print("Total combined journey options:", len(combined_journeys))

display(
    combined_journeys[
        [
            "route",
            "first_transfer",
            "second_transfer",
            "origin_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "transfers",
            "connection_quality"
        ]
    ].head(15)
)

Total combined journey options: 82


,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,transfers,connection_quality
0,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,2,Moderate
1,M1 → T1,Central Station,None,17:31:54,18:29:00,57.100000,1,Not yet evaluated
2,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,58.383333,2,Moderate
3,M1 → T1,Central Station,None,17:35:54,18:32:00,56.100000,1,Not yet evaluated
4,M1 → T1 → T1,Central Station,Strathfield Station,17:31:54,18:32:00,60.100000,2,Tight
5,M1 → T1 → T1,Central Station,Redfern Station,17:31:54,18:32:00,60.100000,2,Tight
6,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:32:00,61.383333,2,Moderate
7,M1 → T9 → T2,Epping Station,Strathfield Station,17:30:37,18:36:30,65.883333,2,Moderate
8,M1 → T9 → T2,Epping Station,Burwood Station,17:30:37,18:36:30,65.883333,2,Tight
9,M1 → T1,Central Station,None,17:39:54,18:38:00,58.100000,1,Not yet evaluated


In [149]:
# STEP 72 — Generate passenger-friendly journey recommendations

def recommend_journeys(journeys):

    if journeys.empty:
        return pd.DataFrame()

    df = journeys.copy()

    recommendations = []

    # 1. Fastest arrival
    fastest = df.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    ).iloc[0].copy()

    fastest["recommendation"] = "Fastest arrival"
    recommendations.append(fastest)

    # 2. Fewest transfers
    simplest = df.sort_values(
        [
            "transfers",
            "destination_arrival_seconds",
            "journey_duration_min"
        ]
    ).iloc[0].copy()

    simplest["recommendation"] = "Fewest transfers"
    recommendations.append(simplest)

    # 3. Most comfortable evaluated connection
    comfort_rank = {
        "Comfortable": 3,
        "Moderate": 2,
        "Tight": 1
    }

    evaluated = df[
        df["connection_quality"].isin(comfort_rank)
    ].copy()

    if not evaluated.empty:

        evaluated["comfort_score"] = (
            evaluated["connection_quality"]
            .map(comfort_rank)
        )

        comfortable = evaluated.sort_values(
            [
                "comfort_score",
                "destination_arrival_seconds",
                "journey_duration_min"
            ],
            ascending=[False, True, True]
        ).iloc[0].copy()

        comfortable["recommendation"] = "Most comfortable evaluated"
        recommendations.append(comfortable)

    result = pd.DataFrame(recommendations)

    columns = [
        "recommendation",
        "route",
        "first_transfer",
        "second_transfer",
        "origin_departure_time",
        "destination_arrival_time",
        "journey_duration_min",
        "transfers",
        "connection_quality"
    ]

    return result[columns].reset_index(drop=True)


# Test our recommendation engine
recommendations = recommend_journeys(combined_journeys)

display(recommendations)

,recommendation,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,transfers,connection_quality
0,Fastest arrival,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,2,Moderate
1,Fewest transfers,M1 → T1,Central Station,NaN,17:31:54,18:29:00,57.100000,1,Not yet evaluated
2,Most comfortable evaluated,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:56:00,85.383333,2,Comfortable


In [151]:
# STEP 74 — Recommend distinct journeys with accurate labels

def recommend_journeys_v3(journeys, max_extra_minutes=15):

    if journeys.empty:
        return pd.DataFrame()

    df = journeys.copy().reset_index(drop=True)
    recommendations = []
    selected_indices = set()

    # 1. Fastest arrival
    fastest_idx = df.sort_values(
        ["destination_arrival_seconds", "transfers", "journey_duration_min"]
    ).index[0]

    fastest = df.loc[fastest_idx].copy()
    fastest["recommendation"] = "Fastest arrival"

    recommendations.append(fastest)
    selected_indices.add(fastest_idx)

    # 2. Fewest transfers (prefer a distinct journey)
    remaining = df.drop(index=list(selected_indices))

    if not remaining.empty:
        simplest_idx = remaining.sort_values(
            ["transfers", "destination_arrival_seconds"]
        ).index[0]

        simplest = df.loc[simplest_idx].copy()
        simplest["recommendation"] = "Fewest transfers"

        recommendations.append(simplest)
        selected_indices.add(simplest_idx)

    # 3. Best evaluated alternative within 15 minutes
    latest_arrival = (
        fastest["destination_arrival_seconds"]
        + max_extra_minutes * 60
    )

    remaining = df.drop(index=list(selected_indices))

    alternatives = remaining[
        (remaining["destination_arrival_seconds"] <= latest_arrival)
        & (remaining["connection_quality"].isin(
            ["Comfortable", "Moderate", "Tight"]
        ))
    ].copy()

    if not alternatives.empty:

        comfort_rank = {
            "Comfortable": 3,
            "Moderate": 2,
            "Tight": 1
        }

        alternatives["comfort_score"] = (
            alternatives["connection_quality"].map(comfort_rank)
        )

        alternative_idx = alternatives.sort_values(
            [
                "comfort_score",
                "destination_arrival_seconds",
                "transfers"
            ],
            ascending=[False, True, True]
        ).index[0]

        alternative = df.loc[alternative_idx].copy()

        if alternative["connection_quality"] == "Comfortable":
            alternative["recommendation"] = "Comfortable alternative"
        else:
            alternative["recommendation"] = "Best evaluated alternative"

        recommendations.append(alternative)

    result = pd.DataFrame(recommendations)

    columns = [
        "recommendation",
        "route",
        "first_transfer",
        "second_transfer",
        "origin_departure_time",
        "destination_arrival_time",
        "journey_duration_min",
        "transfers",
        "connection_quality"
    ]

    return result[columns].reset_index(drop=True)


recommendations_v3 = recommend_journeys_v3(
    combined_journeys,
    max_extra_minutes=15
)

display(recommendations_v3)

,recommendation,route,first_transfer,second_transfer,origin_departure_time,destination_arrival_time,journey_duration_min,transfers,connection_quality
0,Fastest arrival,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:23:00,52.383333,2,Moderate
1,Fewest transfers,M1 → T1,Central Station,NaN,17:31:54,18:29:00,57.100000,1,Not yet evaluated
2,Best evaluated alternative,M1 → T9 → T1,Epping Station,Strathfield Station,17:30:37,18:29:00,58.383333,2,Moderate


In [152]:
# STEP 75A — Discover reachable stations without a destination

def get_reachable_first_leg_stops(
    daily,
    origin,
    departure_time,
    window_minutes=30
):
    # Find available departures from the origin
    departures = get_station_departures(
        daily=daily,
        origin=origin,
        departure_time=departure_time,
        window_minutes=window_minutes
    ).copy()

    if departures.empty:
        return pd.DataFrame()

    # Give boarding-event columns explicit names
    boarding = departures[
        ["trip_id", "stop_sequence", "departure_seconds"]
    ].rename(columns={
        "stop_sequence": "boarding_sequence",
        "departure_seconds": "origin_departure_seconds"
    })

    # Find downstream stops on each boarded trip
    reachable = boarding.merge(
        daily[
            [
                "trip_id",
                "stop_id",
                "station_id",
                "stop_sequence",
                "arrival_time",
                "arrival_seconds"
            ]
        ],
        on="trip_id",
        how="inner"
    )

    # Only include stops after boarding
    reachable = reachable[
        reachable["stop_sequence"]
        > reachable["boarding_sequence"]
    ].copy()

    # Add station names
    names = station_lookup[
        ["station_id", "stop_name"]
    ].drop_duplicates("station_id").rename(
        columns={"stop_name": "station_name"}
    )

    reachable = reachable.merge(
        names,
        on="station_id",
        how="left",
        validate="many_to_one"
    )

    return reachable.sort_values(
        ["origin_departure_seconds", "arrival_seconds"]
    ).reset_index(drop=True)


# Test
reachable_first_leg = get_reachable_first_leg_stops(
    daily=test_timetable,
    origin="Macquarie University Station",
    departure_time="17:30",
    window_minutes=30
)

print("Reachable first-leg stop events:", len(reachable_first_leg))
print(
    "Unique reachable stations:",
    reachable_first_leg["station_name"].nunique()
)

display(
    reachable_first_leg[
        [
            "station_name",
            "arrival_time",
            "trip_id"
        ]
    ].head(20)
)

Reachable first-leg stop events: 160
Unique reachable stations: 20


,station_name,arrival_time,trip_id
0,Epping Station,17:33:51,0160-006-154-013:1000
1,Cherrybrook Station,17:39:38,0160-006-154-013:1000
2,Castle Hill Station,17:42:22,0160-006-154-013:1000
3,Hills Showground Station,17:44:55,0160-006-154-013:1000
4,Norwest Station,17:47:30,0160-006-154-013:1000
5,Bella Vista Station,17:50:10,0160-006-154-013:1000
6,Kellyville Station,17:52:45,0160-006-154-013:1000
7,Rouse Hill Station,17:55:40,0160-006-154-013:1000
8,Tallawong Station,17:58:17,0160-006-154-013:1000
9,Macquarie Park Station,17:33:27,0160-006-106-014:1000


In [153]:
# STEP 75B — Discover first-transfer connections automatically

all_first_connections = []

for _, arrival in reachable_first_leg.iterrows():

    connections = get_valid_connections(
        daily=test_timetable,
        arrival_platform=arrival["stop_id"],
        arrival_seconds=arrival["arrival_seconds"],
        connection_window_minutes=30
    )

    if connections.empty:
        continue

    # Exclude continuing on the same service
    connections = connections[
        connections["trip_id"] != arrival["trip_id"]
    ].copy()

    # Exclude boarding events at terminal stops
    last_sequences = test_timetable.groupby("trip_id")[
        "stop_sequence"
    ].max()

    connections = connections[
        connections["stop_sequence"]
        < connections["trip_id"].map(last_sequences)
    ].copy()

    if connections.empty:
        continue

    # Preserve both journey legs
    connections["first_trip_id"] = arrival["trip_id"]
    connections["first_arrival_station"] = arrival["station_name"]
    connections["first_arrival_platform"] = arrival["stop_id"]
    connections["first_arrival_seconds"] = arrival["arrival_seconds"]
    connections["origin_departure_seconds"] = arrival[
        "origin_departure_seconds"
    ]

    connections["second_trip_id"] = connections["trip_id"]

    connections["first_spare_minutes"] = (
        connections["departure_seconds"]
        - connections["first_arrival_seconds"]
        - connections["minimum_transfer_seconds"]
    ) / 60

    all_first_connections.append(connections)

if all_first_connections:

    first_connections = pd.concat(
        all_first_connections,
        ignore_index=True
    )

    print("Valid first-transfer connections:", len(first_connections))

    print(
        "Unique interchange stations:",
        first_connections["first_arrival_station"].nunique()
    )

    print(
        "Unique second-leg services:",
        first_connections["second_trip_id"].nunique()
    )

    display(
        first_connections[
            [
                "first_arrival_station",
                "first_trip_id",
                "second_trip_id",
                "departure_time",
                "first_spare_minutes"
            ]
        ].head(20)
    )

else:
    first_connections = pd.DataFrame()
    print("No valid first-transfer connections found.")

Valid first-transfer connections: 2694
Unique interchange stations: 20
Unique second-leg services: 191


,first_arrival_station,first_trip_id,second_trip_id,departure_time,first_spare_minutes
0,Epping Station,0160-006-154-013:1000,0160-006-155-013:1000,17:38:36,2.750000
1,Epping Station,0160-006-154-013:1000,0160-006-131-004:1000,17:40:13,1.366667
2,Epping Station,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037,17:42:00,2.650000
3,Epping Station,0160-006-154-013:1000,0160-006-152-015:1000,17:42:36,6.750000
4,Epping Station,0160-006-154-013:1000,0160-006-107-014:1000,17:44:13,5.366667
5,Epping Station,0160-006-154-013:1000,0160-006-112-013:1000,17:46:36,10.750000
6,Epping Station,0160-006-154-013:1000,0160-006-108-014:1000,17:48:13,9.366667
7,Epping Station,0160-006-154-013:1000,0160-006-128-005:1000,17:50:36,14.750000
8,Epping Station,0160-006-154-013:1000,156E.1324.165.16.A.8.92001489,17:51:00,11.650000
9,Epping Station,0160-006-154-013:1000,0160-006-123-012:1000,17:52:13,13.366667


In [156]:
# STEP 75E — Expand all first-transfer connections

# Keep the columns needed for second-leg expansion
connections = first_connections[
    [
        "first_trip_id",
        "second_trip_id",
        "first_arrival_station",
        "origin_departure_seconds",
        "departure_seconds",
        "stop_sequence",
        "first_spare_minutes"
    ]
].copy()

connections = connections.rename(columns={
    "departure_seconds": "second_departure_seconds",
    "stop_sequence": "second_boarding_sequence",
    "first_arrival_station": "first_transfer"
})

# Remove exact duplicate boarding connections
connections = connections.drop_duplicates()

# Prepare downstream timetable events
downstream_stops = test_timetable[
    [
        "trip_id",
        "station_id",
        "stop_id",
        "stop_sequence",
        "arrival_time",
        "arrival_seconds"
    ]
].rename(columns={
    "trip_id": "second_trip_id"
})

# Expand every second-leg trip to its downstream stops
second_leg_reachable_full = connections.merge(
    downstream_stops,
    on="second_trip_id",
    how="inner"
)

# Only keep stops AFTER boarding the second service
second_leg_reachable_full = second_leg_reachable_full[
    second_leg_reachable_full["stop_sequence"]
    > second_leg_reachable_full["second_boarding_sequence"]
].copy()

# Add station names
station_names = (
    station_lookup[
        ["station_id", "stop_name"]
    ]
    .drop_duplicates("station_id")
    .rename(columns={"stop_name": "station_name"})
)

second_leg_reachable_full = second_leg_reachable_full.merge(
    station_names,
    on="station_id",
    how="left",
    validate="many_to_one"
)

# Check destination reachability
parramatta_full = second_leg_reachable_full[
    second_leg_reachable_full["station_name"]
    == "Parramatta Station"
].copy()

print(
    "Total second-leg reachable events:",
    len(second_leg_reachable_full)
)

print(
    "Unique second-leg services:",
    second_leg_reachable_full["second_trip_id"].nunique()
)

print(
    "Unique reachable stations:",
    second_leg_reachable_full["station_name"].nunique()
)

print(
    "One-transfer arrivals at Parramatta:",
    len(parramatta_full)
)

if not parramatta_full.empty:

    parramatta_full["journey_duration_min"] = (
        parramatta_full["arrival_seconds"]
        - parramatta_full["origin_departure_seconds"]
    ) / 60

    display(
        parramatta_full[
            [
                "first_transfer",
                "arrival_time",
                "journey_duration_min",
                "first_spare_minutes"
            ]
        ]
        .sort_values("journey_duration_min")
        .head(10)
    )

Total second-leg reachable events: 28490
Unique second-leg services: 191
Unique reachable stations: 187
One-transfer arrivals at Parramatta: 57


,first_transfer,arrival_time,journey_duration_min,first_spare_minutes
17009,Central Station,18:44:00,56.1,0.766667
6196,Central Station,18:32:00,56.1,0.766667
2497,Central Station,18:29:00,57.1,1.766667
13437,Central Station,18:41:00,57.1,1.766667
9824,Central Station,18:38:00,58.1,2.766667
27476,Central Station,18:59:00,59.1,2.766667
13573,Central Station,18:44:00,60.1,4.766667
24071,Central Station,18:56:00,60.1,3.766667
2641,Central Station,18:32:00,60.1,4.766667
9940,Central Station,18:41:00,61.1,5.766667


In [155]:
# STEP 75D — Identify journeys reaching the destination after one transfer

destination = "Parramatta Station"

# Find second-leg stops matching the destination
one_transfer_arrivals = second_leg_reachable[
    second_leg_reachable["station_name"] == destination
].copy()

if one_transfer_arrivals.empty:
    print(
        f"No one-transfer journeys to {destination} "
        "found in the current sample."
    )
else:
    # Calculate total journey duration
    one_transfer_arrivals["journey_duration_min"] = (
        one_transfer_arrivals["arrival_seconds"]
        - one_transfer_arrivals["origin_departure_seconds"]
    ) / 60

    # Add route labels
    route_lookup = (
        test_timetable[
            ["trip_id", "route_short_name"]
        ]
        .drop_duplicates("trip_id")
        .set_index("trip_id")["route_short_name"]
    )

    one_transfer_arrivals["first_line"] = (
        one_transfer_arrivals["first_trip_id"].map(route_lookup)
    )

    one_transfer_arrivals["second_line"] = (
        one_transfer_arrivals["second_trip_id"].map(route_lookup)
    )

    one_transfer_arrivals["route"] = (
        one_transfer_arrivals["first_line"].astype(str)
        + " → "
        + one_transfer_arrivals["second_line"].astype(str)
    )

    display(
        one_transfer_arrivals[
            [
                "route",
                "first_transfer",
                "arrival_time",
                "journey_duration_min",
                "first_spare_minutes"
            ]
        ].sort_values("journey_duration_min").head(10)
    )

print(
    "Second-leg services explored:",
    second_leg_reachable["second_trip_id"].nunique()
)

print(
    "Unique second-leg stations explored:",
    second_leg_reachable["station_name"].nunique()
)

No one-transfer journeys to Parramatta Station found in the current sample.
Second-leg services explored: 16
Unique second-leg stations explored: 37


In [157]:
# STEP 75F — Evaluate one-transfer journeys

one_transfer_results = parramatta_full.copy()

# Calculate journey duration
one_transfer_results["journey_duration_min"] = (
    one_transfer_results["arrival_seconds"]
    - one_transfer_results["origin_departure_seconds"]
) / 60

# Classify connection quality
def classify_connection(spare_minutes):
    if spare_minutes < 2:
        return "Tight"
    elif spare_minutes < 5:
        return "Moderate"
    else:
        return "Comfortable"


one_transfer_results["connection_quality"] = (
    one_transfer_results["first_spare_minutes"]
    .apply(classify_connection)
)

# Add route labels
trip_routes = (
    test_timetable[
        ["trip_id", "route_short_name"]
    ]
    .drop_duplicates("trip_id")
    .set_index("trip_id")["route_short_name"]
)

one_transfer_results["first_line"] = (
    one_transfer_results["first_trip_id"].map(trip_routes)
)

one_transfer_results["second_line"] = (
    one_transfer_results["second_trip_id"].map(trip_routes)
)

one_transfer_results["route"] = (
    one_transfer_results["first_line"].astype(str)
    + " → "
    + one_transfer_results["second_line"].astype(str)
)

# Recover origin departure time
def seconds_to_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"


one_transfer_results["origin_departure_time"] = (
    one_transfer_results["origin_departure_seconds"]
    .apply(seconds_to_time)
)

# Remove duplicate itinerary records
one_transfer_results = (
    one_transfer_results
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "arrival_seconds"
        ]
    )
    .sort_values(
        ["arrival_seconds", "journey_duration_min"]
    )
    .reset_index(drop=True)
)

print("Valid one-transfer journeys:", len(one_transfer_results))

display(
    one_transfer_results[
        [
            "route",
            "first_transfer",
            "origin_departure_time",
            "arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "connection_quality"
        ]
    ].head(15)
)

Valid one-transfer journeys: 54


,route,first_transfer,origin_departure_time,arrival_time,journey_duration_min,first_spare_minutes,connection_quality
0,M1 → T1,Central Station,17:31:54,18:29:00,57.1,1.766667,Tight
1,M1 → T1,Central Station,17:35:54,18:32:00,56.1,0.766667,Tight
2,M1 → T1,Central Station,17:31:54,18:32:00,60.1,4.766667,Moderate
3,M1 → T1,Central Station,17:39:54,18:38:00,58.1,2.766667,Moderate
4,M1 → T1,Central Station,17:35:54,18:38:00,62.1,6.766667,Comfortable
5,M1 → T1,Central Station,17:31:54,18:38:00,66.1,10.766667,Comfortable
6,M1 → T1,Central Station,17:43:54,18:41:00,57.1,1.766667,Tight
7,M1 → T1,Central Station,17:39:54,18:41:00,61.1,5.766667,Comfortable
8,M1 → T1,Central Station,17:35:54,18:41:00,65.1,9.766667,Comfortable
9,M1 → T1,Central Station,17:31:54,18:41:00,69.1,13.766667,Comfortable


In [158]:
# STEP 76A — Discover potential second-transfer stations

origin_station = "Macquarie University Station"
destination_station = "Parramatta Station"

second_transfer_candidates = second_leg_reachable_full[
    (
        second_leg_reachable_full["station_name"]
        != origin_station
    )
    &
    (
        second_leg_reachable_full["station_name"]
        != destination_station
    )
    &
    (
        second_leg_reachable_full["station_name"]
        != second_leg_reachable_full["first_transfer"]
    )
].copy()

# Avoid processing identical arrival events more than once
second_transfer_candidates = (
    second_transfer_candidates
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "origin_departure_seconds",
            "stop_id",
            "arrival_seconds"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Potential second-transfer arrival events:",
    len(second_transfer_candidates)
)

print(
    "Unique potential second-transfer stations:",
    second_transfer_candidates["station_name"].nunique()
)

# Check that Strathfield was discovered automatically
print(
    "Strathfield discovered:",
    (
        second_transfer_candidates["station_name"]
        == "Strathfield Station"
    ).any()
)

display(
    second_transfer_candidates[
        [
            "first_transfer",
            "station_name",
            "arrival_time",
            "first_spare_minutes",
            "second_trip_id"
        ]
    ]
    .sort_values("arrival_time")
    .head(20)
)

Potential second-transfer arrival events: 12062
Unique potential second-transfer stations: 185
Strathfield discovered: True


,first_transfer,station_name,arrival_time,first_spare_minutes,second_trip_id
391,Macquarie Park Station,North Ryde Station,17:39:35,2.500000,0160-006-120-012:1000
410,Macquarie Park Station,North Ryde Station,17:43:35,6.500000,0160-006-121-012:1000
1967,Macquarie Park Station,North Ryde Station,17:43:35,2.500000,0160-006-121-012:1000
0,Epping Station,Cherrybrook Station,17:43:38,2.750000,0160-006-155-013:1000
515,North Ryde Station,Macquarie Park Station,17:44:06,2.850000,0160-006-128-005:1000
19,Epping Station,Eastwood Station,17:45:00,2.650000,125R.1324.165.16.A.8.92001037
392,Macquarie Park Station,Chatswood Station,17:45:10,2.500000,0160-006-120-012:1000
8,Epping Station,Macquarie Park Station,17:45:27,1.366667,0160-006-131-004:1000
401,Macquarie Park Station,Epping Station,17:45:51,3.150000,0160-006-112-013:1000
1,Epping Station,Castle Hill Station,17:46:22,2.750000,0160-006-155-013:1000


In [159]:
# STEP 76B — Inspect potential second-transfer stations

print(
    "Total second-transfer candidates:",
    len(second_transfer_candidates)
)

print(
    "Unique second-transfer stations:",
    second_transfer_candidates["station_name"].nunique()
)

strathfield_candidates = second_transfer_candidates[
    second_transfer_candidates["station_name"] == "Strathfield Station"
].copy()

print(
    "Strathfield transfer candidates:",
    len(strathfield_candidates)
)

# Inspect the best Strathfield candidates
if not strathfield_candidates.empty:

    strathfield_candidates["journey_elapsed_min"] = (
        strathfield_candidates["arrival_seconds"]
        - strathfield_candidates["origin_departure_seconds"]
    ) / 60

    display(
        strathfield_candidates[
            [
                "first_transfer",
                "station_name",
                "arrival_time",
                "journey_elapsed_min",
                "first_spare_minutes",
                "first_trip_id",
                "second_trip_id"
            ]
        ]
        .sort_values("arrival_time")
        .head(15)
    )

else:
    print("No Strathfield candidates found.")

Total second-transfer candidates: 12062
Unique second-transfer stations: 185
Strathfield transfer candidates: 136


,first_transfer,station_name,arrival_time,journey_elapsed_min,first_spare_minutes,first_trip_id,second_trip_id
26,Epping Station,Strathfield Station,18:03:00,32.383333,2.650000,0160-006-154-013:1000,125R.1324.165.16.A.8.92001037
96,Epping Station,Strathfield Station,18:07:48,37.183333,13.666667,0160-006-154-013:1000,130N.1324.165.16.A.8.92003126
1647,Epping Station,Strathfield Station,18:07:48,33.183333,9.666667,0160-006-155-013:1000,130N.1324.165.16.A.8.92003126
3231,Epping Station,Strathfield Station,18:07:48,29.183333,5.666667,0160-006-152-015:1000,130N.1324.165.16.A.8.92003126
4758,Epping Station,Strathfield Station,18:07:48,25.183333,1.666667,0160-006-112-013:1000,130N.1324.165.16.A.8.92003126
888,Central Station,Strathfield Station,18:16:30,44.600000,1.766667,0160-006-106-014:1000,145N.1324.165.16.A.8.92001340
125,Epping Station,Strathfield Station,18:18:00,47.383333,17.650000,0160-006-154-013:1000,191J.1324.165.16.B.8.92003392
1676,Epping Station,Strathfield Station,18:18:00,43.383333,13.650000,0160-006-155-013:1000,191J.1324.165.16.B.8.92003392
3260,Epping Station,Strathfield Station,18:18:00,39.383333,9.650000,0160-006-152-015:1000,191J.1324.165.16.B.8.92003392
4787,Epping Station,Strathfield Station,18:18:00,35.383333,5.650000,0160-006-112-013:1000,191J.1324.165.16.B.8.92003392


In [160]:
# STEP 76C — Validate the third-leg connection

# Select the earliest Strathfield candidate
candidate = (
    strathfield_candidates
    .sort_values("arrival_seconds")
    .iloc[0]
)

print("First transfer:", candidate["first_transfer"])
print("Second transfer:", candidate["station_name"])
print("Arrival at second transfer:", candidate["arrival_time"])

# Find valid services after arriving at Strathfield
third_connections = get_valid_connections(
    daily=test_timetable,
    arrival_platform=candidate["stop_id"],
    arrival_seconds=candidate["arrival_seconds"],
    connection_window_minutes=30
)

# Exclude services already used
third_connections = third_connections[
    ~third_connections["trip_id"].isin([
        candidate["first_trip_id"],
        candidate["second_trip_id"]
    ])
].copy()

print("Valid third-leg boarding options:", len(third_connections))

# Find services that continue to Parramatta
third_leg_journeys = []

for _, connection in third_connections.iterrows():

    downstream = get_downstream_stops(
        daily=test_timetable,
        trip_id=connection["trip_id"],
        boarding_sequence=connection["stop_sequence"]
    )

    destination_stops = downstream[
        downstream["station_name"] == "Parramatta Station"
    ]

    for _, destination_stop in destination_stops.iterrows():

        third_leg_journeys.append({
            "third_trip_id": connection["trip_id"],
            "third_departure_time": connection["departure_time"],
            "destination_arrival_time": destination_stop["arrival_time"],
            "journey_duration_min": (
                destination_stop["arrival_seconds"]
                - candidate["origin_departure_seconds"]
            ) / 60,
            "second_spare_minutes": (
                connection["departure_seconds"]
                - candidate["arrival_seconds"]
                - connection["minimum_transfer_seconds"]
            ) / 60
        })

third_leg_results = pd.DataFrame(third_leg_journeys)

print("Third-leg journeys reaching Parramatta:", len(third_leg_results))

if not third_leg_results.empty:
    display(
        third_leg_results
        .sort_values("destination_arrival_time")
        .head(10)
    )

First transfer: Epping Station
Second transfer: Strathfield Station
Arrival at second transfer: 18:03:00
Valid third-leg boarding options: 25
Third-leg journeys reaching Parramatta: 8


,third_trip_id,third_departure_time,destination_arrival_time,journey_duration_min,second_spare_minutes
0,197F.1324.165.16.B.8.92001727,18:11:24,18:23:00,52.383333,3.4
2,145N.1324.165.16.A.8.92001340,18:17:24,18:29:00,58.383333,9.4
3,165F.1324.165.124.A.8.92002689,18:20:24,18:32:00,61.383333,12.4
1,78-P.1324.165.16.A.8.92002733,18:16:00,18:36:30,65.883333,6.6
4,155N.1324.165.16.A.8.92002106,18:26:24,18:38:00,67.383333,18.4
5,111U.1324.165.16.T.8.92004513,18:29:24,18:41:00,70.383333,21.4
7,154N.1324.165.16.A.8.92001114,18:32:24,18:44:00,73.383333,24.4
6,92-P.1324.165.60.A.8.92004162,18:31:00,18:51:30,80.883333,21.6


In [161]:
# STEP 76D — Vectorised third-leg journey discovery

def find_third_leg_journeys(
    daily,
    candidates,
    destination,
    max_connection_minutes=30
):
    if candidates.empty:
        return pd.DataFrame()

    # Prepare potential third-leg boarding events
    boarding = daily[
        [
            "trip_id",
            "stop_id",
            "stop_sequence",
            "departure_time",
            "departure_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_id": "third_boarding_platform",
        "stop_sequence": "third_boarding_sequence",
        "departure_time": "third_departure_time",
        "departure_seconds": "third_departure_seconds"
    })

    # Platform transfer rules
    transfers = all_platform_transfers[
        [
            "from_stop_id",
            "to_stop_id",
            "minimum_transfer_seconds"
        ]
    ].drop_duplicates()

    # Match second-leg arrivals to transfer pathways
    connections = candidates.merge(
        transfers,
        left_on="stop_id",
        right_on="from_stop_id",
        how="inner"
    )

    # Match pathways to third-leg departures
    connections = connections.merge(
        boarding,
        left_on="to_stop_id",
        right_on="third_boarding_platform",
        how="inner"
    )

    # Keep only valid connection times
    earliest_boarding = (
        connections["arrival_seconds"]
        + connections["minimum_transfer_seconds"]
    )

    latest_boarding = (
        connections["arrival_seconds"]
        + max_connection_minutes * 60
    )

    connections = connections[
        (connections["third_departure_seconds"] >= earliest_boarding)
        & (connections["third_departure_seconds"] <= latest_boarding)
        & (connections["third_trip_id"] != connections["first_trip_id"])
        & (connections["third_trip_id"] != connections["second_trip_id"])
    ].copy()

    # Prepare destination arrivals
    destination_ids = set(
        station_lookup.loc[
            station_lookup["stop_name"] == destination,
            "station_id"
        ]
    )

    arrivals = daily[
        daily["station_id"].isin(destination_ids)
    ][
        [
            "trip_id",
            "stop_sequence",
            "arrival_time",
            "arrival_seconds"
        ]
    ].rename(columns={
        "trip_id": "third_trip_id",
        "stop_sequence": "destination_sequence",
        "arrival_time": "destination_arrival_time",
        "arrival_seconds": "destination_arrival_seconds"
    })

    # Find third-leg services reaching destination
    results = connections.merge(
        arrivals,
        on="third_trip_id",
        how="inner"
    )

    # Destination must occur after boarding
    results = results[
        results["destination_sequence"]
        > results["third_boarding_sequence"]
    ].copy()

    results["journey_duration_min"] = (
        results["destination_arrival_seconds"]
        - results["origin_departure_seconds"]
    ) / 60

    results["second_spare_minutes"] = (
        results["third_departure_seconds"]
        - results["arrival_seconds"]
        - results["minimum_transfer_seconds"]
    ) / 60

    return results.sort_values(
        ["destination_arrival_seconds", "journey_duration_min"]
    ).reset_index(drop=True)


# Test only Strathfield candidates first
third_leg_test = find_third_leg_journeys(
    daily=test_timetable,
    candidates=strathfield_candidates,
    destination="Parramatta Station"
)

print("Third-leg journeys found:", len(third_leg_test))

display(
    third_leg_test[
        [
            "first_transfer",
            "station_name",
            "third_departure_time",
            "destination_arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "second_spare_minutes"
        ]
    ].head(15)
)

Third-leg journeys found: 731


,first_transfer,station_name,third_departure_time,destination_arrival_time,journey_duration_min,first_spare_minutes,second_spare_minutes
0,Epping Station,Strathfield Station,18:11:24,18:23:00,52.383333,2.650000,3.4
1,Epping Station,Strathfield Station,18:17:24,18:29:00,46.383333,1.666667,2.2
2,Epping Station,Strathfield Station,18:17:24,18:29:00,50.383333,5.666667,2.2
3,Epping Station,Strathfield Station,18:17:24,18:29:00,54.383333,9.666667,2.2
4,Epping Station,Strathfield Station,18:17:24,18:29:00,58.383333,2.650000,9.4
5,Epping Station,Strathfield Station,18:17:24,18:29:00,58.383333,13.666667,2.2
6,Epping Station,Strathfield Station,18:20:24,18:32:00,49.383333,1.666667,5.2
7,Epping Station,Strathfield Station,18:20:24,18:32:00,53.383333,5.666667,5.2
8,Epping Station,Strathfield Station,18:20:24,18:32:00,57.383333,9.666667,5.2
9,Central Station,Strathfield Station,18:20:24,18:32:00,60.100000,1.766667,1.9


In [162]:
# STEP 76E — Search all second-transfer stations

import time

start_time = time.perf_counter()

# Run the vectorised search across every candidate
all_two_transfer_journeys = find_third_leg_journeys(
    daily=test_timetable,
    candidates=second_transfer_candidates,
    destination="Parramatta Station",
    max_connection_minutes=30
)

# Keep journeys lasting no more than 120 minutes
all_two_transfer_journeys = all_two_transfer_journeys[
    (all_two_transfer_journeys["journey_duration_min"] > 0)
    & (all_two_transfer_journeys["journey_duration_min"] <= 120)
].copy()

# Remove duplicate itinerary records
all_two_transfer_journeys = (
    all_two_transfer_journeys
    .drop_duplicates(
        subset=[
            "first_trip_id",
            "second_trip_id",
            "third_trip_id",
            "origin_departure_seconds",
            "destination_arrival_seconds"
        ]
    )
    .sort_values(
        ["destination_arrival_seconds", "journey_duration_min"]
    )
    .reset_index(drop=True)
)

elapsed = time.perf_counter() - start_time

print(f"Search completed in {elapsed:.2f} seconds")
print("Two-transfer journeys found:", len(all_two_transfer_journeys))

print(
    "Unique first-transfer stations:",
    all_two_transfer_journeys["first_transfer"].nunique()
)

print(
    "Unique second-transfer stations:",
    all_two_transfer_journeys["station_name"].nunique()
)

display(
    all_two_transfer_journeys[
        [
            "first_transfer",
            "station_name",
            "destination_arrival_time",
            "journey_duration_min",
            "first_spare_minutes",
            "second_spare_minutes"
        ]
    ].head(15)
)

Search completed in 1.17 seconds
Two-transfer journeys found: 2935
Unique first-transfer stations: 11
Unique second-transfer stations: 17


,first_transfer,station_name,destination_arrival_time,journey_duration_min,first_spare_minutes,second_spare_minutes
0,Epping Station,Strathfield Station,18:23:00,52.383333,2.650000,3.400000
1,Epping Station,Strathfield Station,18:29:00,46.383333,1.666667,2.200000
2,Epping Station,Strathfield Station,18:29:00,50.383333,5.666667,2.200000
3,Epping Station,Strathfield Station,18:29:00,54.383333,9.666667,2.200000
4,Epping Station,Strathfield Station,18:29:00,58.383333,2.650000,9.400000
5,Epping Station,Strathfield Station,18:29:00,58.383333,13.666667,2.200000
6,Epping Station,Strathfield Station,18:32:00,49.383333,1.666667,5.200000
7,Epping Station,Strathfield Station,18:32:00,53.383333,5.666667,5.200000
8,Epping Station,Strathfield Station,18:32:00,57.383333,9.666667,5.200000
9,Macquarie Park Station,Central Station,18:32:00,60.100000,2.500000,0.766667


In [163]:
# STEP 77A — Pareto filter across one- and two-transfer journeys

# Standardise one-transfer journeys
one = one_transfer_results[
    [
        "origin_departure_seconds",
        "arrival_seconds",
        "journey_duration_min",
        "first_transfer",
        "connection_quality"
    ]
].copy()

one = one.rename(columns={
    "arrival_seconds": "destination_arrival_seconds"
})

one["second_transfer"] = None
one["transfers"] = 1

# Standardise two-transfer journeys
two = all_two_transfer_journeys[
    [
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "journey_duration_min",
        "first_transfer",
        "station_name"
    ]
].copy()

two = two.rename(columns={
    "station_name": "second_transfer"
})

two["transfers"] = 2
two["connection_quality"] = "Not yet classified"

# Combine both
all_journeys = pd.concat(
    [one, two],
    ignore_index=True
)

# Remove exact duplicates
all_journeys = all_journeys.drop_duplicates()

# Pareto dominance filtering
def pareto_filter(journeys):

    df = journeys.reset_index(drop=True).copy()

    keep = []

    for i, journey in df.iterrows():

        dominated = (
            (df["origin_departure_seconds"]
             >= journey["origin_departure_seconds"])
            &
            (df["destination_arrival_seconds"]
             <= journey["destination_arrival_seconds"])
            &
            (df["transfers"] <= journey["transfers"])
            &
            (
                (df["origin_departure_seconds"]
                 > journey["origin_departure_seconds"])
                |
                (df["destination_arrival_seconds"]
                 < journey["destination_arrival_seconds"])
                |
                (df["transfers"] < journey["transfers"])
            )
        ).any()

        keep.append(not dominated)

    return df[keep].sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).reset_index(drop=True)


pareto_journeys = pareto_filter(all_journeys)

print("Journeys before filtering:", len(all_journeys))
print("Journeys after filtering:", len(pareto_journeys))

display(
    pareto_journeys[
        [
            "first_transfer",
            "second_transfer",
            "journey_duration_min",
            "transfers",
            "connection_quality"
        ]
    ].head(20)
)

Journeys before filtering: 1028
Journeys after filtering: 11


,first_transfer,second_transfer,journey_duration_min,transfers,connection_quality
0,Epping Station,Strathfield Station,52.383333,2,Not yet classified
1,Central Station,None,57.100000,1,Tight
2,Epping Station,Strathfield Station,46.383333,2,Not yet classified
3,Central Station,None,56.100000,1,Tight
4,Central Station,None,58.100000,1,Moderate
5,Epping Station,Strathfield Station,51.383333,2,Not yet classified
6,Central Station,None,57.100000,1,Tight
7,Central Station,None,56.100000,1,Tight
8,Epping Station,Strathfield Station,45.383333,2,Not yet classified
9,Central Station,None,60.100000,1,Moderate


In [164]:
# STEP 77B — Evaluate connection quality for two-transfer journeys

two_transfer_quality = all_two_transfer_journeys.copy()

# The weakest connection determines overall quality
two_transfer_quality["minimum_spare_minutes"] = (
    two_transfer_quality[
        ["first_spare_minutes", "second_spare_minutes"]
    ].min(axis=1)
)

two_transfer_quality["connection_quality"] = (
    two_transfer_quality["minimum_spare_minutes"]
    .apply(classify_connection)
)

# Match quality back to the selected two-transfer journeys
quality_lookup = (
    two_transfer_quality[
        [
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer",
            "station_name",
            "journey_duration_min",
            "connection_quality",
            "minimum_spare_minutes"
        ]
    ]
    .rename(columns={
        "station_name": "second_transfer"
    })
    .drop_duplicates(
        subset=[
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer",
            "second_transfer",
            "journey_duration_min"
        ]
    )
)

# Preserve the one-transfer quality labels
pareto_journeys_v2 = pareto_journeys.merge(
    quality_lookup,
    on=[
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "first_transfer",
        "second_transfer",
        "journey_duration_min"
    ],
    how="left",
    suffixes=("", "_evaluated")
)

pareto_journeys_v2["connection_quality"] = (
    pareto_journeys_v2["connection_quality_evaluated"]
    .fillna(pareto_journeys_v2["connection_quality"])
)

pareto_journeys_v2 = pareto_journeys_v2.drop(
    columns=["connection_quality_evaluated"]
)

# Format departure and arrival times
pareto_journeys_v2["departure_time"] = (
    pareto_journeys_v2["origin_departure_seconds"]
    .apply(seconds_to_time)
)

pareto_journeys_v2["arrival_time"] = (
    pareto_journeys_v2["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

display(
    pareto_journeys_v2[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "journey_duration_min",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ]
)

,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,52.383333,2,2.650000,Moderate
1,17:31:54,18:29:00,Central Station,None,57.100000,1,NaN,Tight
2,17:42:37,18:29:00,Epping Station,Strathfield Station,46.383333,2,1.666667,Tight
3,17:35:54,18:32:00,Central Station,None,56.100000,1,NaN,Tight
4,17:39:54,18:38:00,Central Station,None,58.100000,1,NaN,Moderate
5,17:46:37,18:38:00,Epping Station,Strathfield Station,51.383333,2,1.650000,Tight
6,17:43:54,18:41:00,Central Station,None,57.100000,1,NaN,Tight
7,17:47:54,18:44:00,Central Station,None,56.100000,1,NaN,Tight
8,17:58:37,18:44:00,Epping Station,Strathfield Station,45.383333,2,0.666667,Tight
9,17:55:54,18:56:00,Central Station,None,60.100000,1,NaN,Moderate


In [165]:
# STEP 77C — Comfort-aware Pareto filtering

# Start from all journeys, before the previous Pareto filter
comfort_journeys = all_journeys.copy()

# Calculate overall connection buffer for two-transfer journeys
two_quality = all_two_transfer_journeys.copy()

two_quality["minimum_spare_minutes"] = two_quality[
    ["first_spare_minutes", "second_spare_minutes"]
].min(axis=1)

# Build a lookup, preserving the best buffer for matching itineraries
two_lookup = (
    two_quality.groupby(
        [
            "origin_departure_seconds",
            "destination_arrival_seconds",
            "first_transfer",
            "station_name",
            "journey_duration_min"
        ],
        as_index=False
    )["minimum_spare_minutes"]
    .max()
    .rename(columns={"station_name": "second_transfer"})
)

# Add two-transfer connection buffers
comfort_journeys = comfort_journeys.merge(
    two_lookup,
    on=[
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "first_transfer",
        "second_transfer",
        "journey_duration_min"
    ],
    how="left"
)

# Add one-transfer connection buffers
one_lookup = (
    one_transfer_results.groupby(
        [
            "origin_departure_seconds",
            "arrival_seconds",
            "first_transfer",
            "journey_duration_min"
        ],
        as_index=False
    )["first_spare_minutes"]
    .max()
    .rename(columns={
        "arrival_seconds": "destination_arrival_seconds",
        "first_spare_minutes": "one_transfer_spare"
    })
)

comfort_journeys = comfort_journeys.merge(
    one_lookup,
    on=[
        "origin_departure_seconds",
        "destination_arrival_seconds",
        "first_transfer",
        "journey_duration_min"
    ],
    how="left"
)

comfort_journeys["minimum_spare_minutes"] = (
    comfort_journeys["minimum_spare_minutes"]
    .fillna(comfort_journeys["one_transfer_spare"])
)

comfort_journeys["connection_quality"] = (
    comfort_journeys["minimum_spare_minutes"]
    .apply(
        lambda x: classify_connection(x)
        if pd.notna(x) else "Unknown"
    )
)

# Convert comfort labels into numerical scores
comfort_scores = {
    "Unknown": 0,
    "Tight": 1,
    "Moderate": 2,
    "Comfortable": 3
}

comfort_journeys["comfort_score"] = (
    comfort_journeys["connection_quality"].map(comfort_scores)
)

# Vectorised dominance check using NumPy
import numpy as np

def comfort_pareto_filter(df):

    data = df.reset_index(drop=True).copy()

    departure = data["origin_departure_seconds"].to_numpy()
    arrival = data["destination_arrival_seconds"].to_numpy()
    transfers = data["transfers"].to_numpy()
    comfort = data["comfort_score"].to_numpy()

    keep = np.ones(len(data), dtype=bool)

    for i in range(len(data)):

        dominates = (
            (departure >= departure[i])
            & (arrival <= arrival[i])
            & (transfers <= transfers[i])
            & (comfort >= comfort[i])
            & (
                (departure > departure[i])
                | (arrival < arrival[i])
                | (transfers < transfers[i])
                | (comfort > comfort[i])
            )
        )

        if dominates.any():
            keep[i] = False

    return data.loc[keep].sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).reset_index(drop=True)


comfort_pareto_journeys = comfort_pareto_filter(comfort_journeys)

comfort_pareto_journeys["departure_time"] = (
    comfort_pareto_journeys["origin_departure_seconds"]
    .apply(seconds_to_time)
)

comfort_pareto_journeys["arrival_time"] = (
    comfort_pareto_journeys["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

print("Journeys before filtering:", len(comfort_journeys))
print("Journeys after comfort-aware filtering:", len(comfort_pareto_journeys))

display(
    comfort_pareto_journeys[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ].head(25)
)

Journeys before filtering: 1028
Journeys after comfort-aware filtering: 25


,departure_time,arrival_time,first_transfer,second_transfer,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,2,2.650000,Moderate
1,17:31:54,18:29:00,Central Station,None,1,1.766667,Tight
2,17:42:37,18:29:00,Epping Station,Strathfield Station,2,1.666667,Tight
3,17:38:37,18:29:00,Epping Station,Strathfield Station,2,2.200000,Moderate
4,17:35:54,18:32:00,Central Station,None,1,0.766667,Tight
5,17:31:54,18:32:00,Central Station,None,1,4.766667,Moderate
6,17:38:37,18:32:00,Epping Station,Strathfield Station,2,5.200000,Comfortable
7,17:39:54,18:38:00,Central Station,None,1,2.766667,Moderate
8,17:35:54,18:38:00,Central Station,None,1,6.766667,Comfortable
9,17:46:37,18:38:00,Epping Station,Strathfield Station,2,1.650000,Tight


In [166]:
# STEP 78 — Passenger-friendly journey recommendations

def recommend_best_journeys(journeys, max_extra_minutes=15):

    if journeys.empty:
        return pd.DataFrame()

    df = journeys.copy().reset_index(drop=True)

    selected = []
    used = set()

    # 1. Fastest arrival
    fastest_idx = df.sort_values(
        ["destination_arrival_seconds", "transfers"]
    ).index[0]

    fastest = df.loc[fastest_idx].copy()
    fastest["recommendation"] = "Fastest arrival"

    selected.append(fastest)
    used.add(fastest_idx)

    # 2. Fewest transfers
    remaining = df.drop(index=list(used))

    if not remaining.empty:

        simple_idx = remaining.sort_values(
            ["transfers", "destination_arrival_seconds"]
        ).index[0]

        simple = df.loc[simple_idx].copy()
        simple["recommendation"] = "Fewest transfers"

        selected.append(simple)
        used.add(simple_idx)

    # 3. Comfortable alternative
    latest_arrival = (
        fastest["destination_arrival_seconds"]
        + max_extra_minutes * 60
    )

    remaining = df.drop(index=list(used))

    comfortable = remaining[
        (remaining["destination_arrival_seconds"] <= latest_arrival)
        & (remaining["connection_quality"] == "Comfortable")
    ].copy()

    if not comfortable.empty:

        # Prefer fewer transfers, then earlier arrival
        comfort_idx = comfortable.sort_values(
            ["transfers", "destination_arrival_seconds"]
        ).index[0]

        comfort = df.loc[comfort_idx].copy()
        comfort["recommendation"] = "Comfortable alternative"

        selected.append(comfort)

    result = pd.DataFrame(selected)

    columns = [
        "recommendation",
        "departure_time",
        "arrival_time",
        "first_transfer",
        "second_transfer",
        "journey_duration_min",
        "transfers",
        "minimum_spare_minutes",
        "connection_quality"
    ]

    return result[columns].reset_index(drop=True)


final_recommendations = recommend_best_journeys(
    comfort_pareto_journeys,
    max_extra_minutes=15
)

display(final_recommendations)

,recommendation,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,Fastest arrival,17:30:37,18:23:00,Epping Station,Strathfield Station,52.383333,2,2.650000,Moderate
1,Fewest transfers,17:31:54,18:29:00,Central Station,NaN,57.100000,1,1.766667,Tight
2,Comfortable alternative,17:35:54,18:38:00,Central Station,NaN,62.100000,1,6.766667,Comfortable


In [167]:
# STEP 79A — Exclude tight connections from recommendations

def filter_reliable_journeys(journeys, min_spare_minutes=2):

    reliable = journeys[
        (journeys["minimum_spare_minutes"] >= min_spare_minutes)
        & (journeys["connection_quality"].isin(
            ["Moderate", "Comfortable"]
        ))
    ].copy()

    return reliable.reset_index(drop=True)


reliable_journeys = filter_reliable_journeys(
    comfort_pareto_journeys,
    min_spare_minutes=2
)

print("Journeys before filtering:", len(comfort_pareto_journeys))
print("Reliable journeys:", len(reliable_journeys))
print(
    "Tight or unevaluated journeys excluded:",
    len(comfort_pareto_journeys) - len(reliable_journeys)
)

reliable_recommendations = recommend_best_journeys(
    reliable_journeys,
    max_extra_minutes=15
)

display(reliable_recommendations)

Journeys before filtering: 25
Reliable journeys: 18
Tight or unevaluated journeys excluded: 7


,recommendation,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,Fastest arrival,17:30:37,18:23:00,Epping Station,Strathfield Station,52.383333,2,2.650000,Moderate
1,Fewest transfers,17:31:54,18:32:00,Central Station,NaN,60.100000,1,4.766667,Moderate
2,Comfortable alternative,17:35:54,18:38:00,Central Station,NaN,62.100000,1,6.766667,Comfortable


In [168]:
# STEP 79B — Reusable reliability filtering

def filter_reliable_journeys_v2(
    journeys,
    min_spare_minutes=2
):
    if journeys.empty:
        return journeys.copy()

    df = journeys.copy()

    # Direct journeys do not require connection buffers
    direct = df["transfers"] == 0

    # Transfer journeys must meet the reliability threshold
    reliable_transfers = (
        (df["transfers"] > 0)
        & (df["minimum_spare_minutes"] >= min_spare_minutes)
    )

    filtered = df[
        direct | reliable_transfers
    ].copy()

    return filtered.sort_values(
        [
            "destination_arrival_seconds",
            "transfers",
            "journey_duration_min"
        ]
    ).reset_index(drop=True)


# Test against existing results
reliable_journeys_v2 = filter_reliable_journeys_v2(
    comfort_pareto_journeys,
    min_spare_minutes=2
)

print("Journeys before:", len(comfort_pareto_journeys))
print("Journeys after:", len(reliable_journeys_v2))

print(
    "Tight connections remaining:",
    (
        (reliable_journeys_v2["transfers"] > 0)
        & (reliable_journeys_v2["minimum_spare_minutes"] < 2)
    ).sum()
)

display(
    reliable_journeys_v2[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ].head(15)
)

Journeys before: 25
Journeys after: 18
Tight connections remaining: 0


,departure_time,arrival_time,first_transfer,second_transfer,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,2,2.650000,Moderate
1,17:38:37,18:29:00,Epping Station,Strathfield Station,2,2.200000,Moderate
2,17:31:54,18:32:00,Central Station,None,1,4.766667,Moderate
3,17:38:37,18:32:00,Epping Station,Strathfield Station,2,5.200000,Comfortable
4,17:39:54,18:38:00,Central Station,None,1,2.766667,Moderate
5,17:35:54,18:38:00,Central Station,None,1,6.766667,Comfortable
6,17:42:37,18:38:00,Epping Station,Strathfield Station,2,3.400000,Moderate
7,17:39:54,18:41:00,Central Station,None,1,5.766667,Comfortable
8,17:42:37,18:41:00,Epping Station,Strathfield Station,2,5.650000,Comfortable
9,17:43:54,18:44:00,Central Station,None,1,4.766667,Moderate


In [169]:
# STEP 80A — Reliability-first journey ranking

# Start from all candidate journeys
all_reliable_candidates = filter_reliable_journeys_v2(
    comfort_journeys,
    min_spare_minutes=2
)

# Apply Pareto filtering only after excluding tight connections
reliable_pareto = comfort_pareto_filter(
    all_reliable_candidates
)

# Format departure and arrival times
reliable_pareto["departure_time"] = (
    reliable_pareto["origin_departure_seconds"]
    .apply(seconds_to_time)
)

reliable_pareto["arrival_time"] = (
    reliable_pareto["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

print("Original candidates:", len(comfort_journeys))
print("Reliable candidates:", len(all_reliable_candidates))
print("After Pareto filtering:", len(reliable_pareto))

print(
    "Tight connections remaining:",
    (
        (reliable_pareto["transfers"] > 0)
        & (reliable_pareto["minimum_spare_minutes"] < 2)
    ).sum()
)

display(
    reliable_pareto[
        [
            "departure_time",
            "arrival_time",
            "first_transfer",
            "second_transfer",
            "transfers",
            "minimum_spare_minutes",
            "connection_quality"
        ]
    ].head(20)
)

Original candidates: 1028
Reliable candidates: 902
After Pareto filtering: 18
Tight connections remaining: 0


,departure_time,arrival_time,first_transfer,second_transfer,transfers,minimum_spare_minutes,connection_quality
0,17:30:37,18:23:00,Epping Station,Strathfield Station,2,2.650000,Moderate
1,17:38:37,18:29:00,Epping Station,Strathfield Station,2,2.200000,Moderate
2,17:31:54,18:32:00,Central Station,None,1,4.766667,Moderate
3,17:38:37,18:32:00,Epping Station,Strathfield Station,2,5.200000,Comfortable
4,17:39:54,18:38:00,Central Station,None,1,2.766667,Moderate
5,17:35:54,18:38:00,Central Station,None,1,6.766667,Comfortable
6,17:42:37,18:38:00,Epping Station,Strathfield Station,2,3.400000,Moderate
7,17:39:54,18:41:00,Central Station,None,1,5.766667,Comfortable
8,17:42:37,18:41:00,Epping Station,Strathfield Station,2,5.650000,Comfortable
9,17:43:54,18:44:00,Central Station,None,1,4.766667,Moderate


In [170]:
# STEP 80B — Require at least 5 minutes of spare transfer time

MIN_SPARE_MINUTES = 5

reliable_candidates_5min = filter_reliable_journeys_v2(
    comfort_journeys,
    min_spare_minutes=MIN_SPARE_MINUTES
)

reliable_pareto_5min = comfort_pareto_filter(
    reliable_candidates_5min
)

reliable_pareto_5min["departure_time"] = (
    reliable_pareto_5min["origin_departure_seconds"]
    .apply(seconds_to_time)
)

reliable_pareto_5min["arrival_time"] = (
    reliable_pareto_5min["destination_arrival_seconds"]
    .apply(seconds_to_time)
)

print("Original candidates:", len(comfort_journeys))
print("Reliable candidates:", len(reliable_candidates_5min))
print("After Pareto filtering:", len(reliable_pareto_5min))

final_recommendations_5min = recommend_best_journeys(
    reliable_pareto_5min,
    max_extra_minutes=15
)

display(final_recommendations_5min)

Original candidates: 1028
Reliable candidates: 717
After Pareto filtering: 8


,recommendation,departure_time,arrival_time,first_transfer,second_transfer,journey_duration_min,transfers,minimum_spare_minutes,connection_quality
0,Fastest arrival,17:38:37,18:32:00,Epping Station,Strathfield Station,53.383333,2,5.200000,Comfortable
1,Fewest transfers,17:35:54,18:38:00,Central Station,NaN,62.100000,1,6.766667,Comfortable
2,Comfortable alternative,17:39:54,18:41:00,Central Station,NaN,61.100000,1,5.766667,Comfortable
